# ABLATION STUDY PIPELINE: EW-LMD BOMBING HYBRID <br>
ABLATION STUDY PIPELINE: EW-LMD BOMBING HYBRID <br>

## Import

In [ ]:
# pip install scikit_posthocs# 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
from typing import Dict, List, Tuple, Optional, Literal, Any, Callable
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment
from scipy.stats import friedmanchisquare, wilcoxon
from scipy.spatial.distance import pdist, squareform
import scikit_posthocs as sp
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path
from sklearn.datasets import make_blobs
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import dijkstra
from collections import deque

import warnings
warnings.filterwarnings('ignore')

## MODEL

### HYBRID SELECTION CONSERVATIVE LOGIC

In [ ]:
# =============================================================================
# 1. Helper Functions & Hybrid Selection Logic
# =============================================================================
def calculate_shannon_entropy(series: pd.Series) -> float:
    p = series.value_counts(normalize=True)
    return -np.sum(p * np.log2(p + 1e-9))

def rule_based_selection(structure: dict) -> dict:
    dist_cons, eta, n_bins = 0.1, 5, 5
    
    if structure['n_nominal'] >= 8 and structure['max_nominal_card'] > 10: dist_cons = 0.2
    elif structure['max_nominal_card'] > 25: dist_cons = 0.2
        
    if structure['is_numeric_heavy'] and structure['n_numeric'] >= 7: eta = 25
    elif structure['is_wide'] and structure['max_nominal_card'] > 15: eta = 25
    elif structure['is_wide'] and structure['n_ordinal'] >= 5: eta = 15
    elif structure['n_ordinal'] >= 3 and structure['n_nominal'] >= 5: eta = 20
        
    if structure['n_numeric'] >= 8: n_bins = 15
    elif structure['n_numeric'] >= 5: n_bins = 10
    elif structure['n_rows'] < 500 and structure['n_numeric'] > 3: n_bins = 10
        
    return {"dist_cons": dist_cons, "eta": float(eta), "n_bins_numeric": int(n_bins)}

def detect_domain(structure: dict) -> str:
    if structure['n_nominal'] >= 8 and structure['max_nominal_card'] > 10: return "medical_categorical"
    elif structure['n_numeric'] >= 10 and structure['n_numeric'] / structure['n_cols'] > 0.5: return "numeric_heavy"
    elif structure['n_ordinal'] >= 5: return "ordinal_rich"
    elif structure['n_rows'] > 5000: return "large_scale"
    return "general"

def hybrid_selection_conservative(df: pd.DataFrame, attr_types: Dict[str, str], structure: dict) -> dict:
    base = rule_based_selection(structure)
    domain = detect_domain(structure)
    
    ents = [calculate_shannon_entropy(df[col]) for col in df.columns]
    mean_ent, std_ent = np.mean(ents), np.std(ents)
    
    thresholds = {
        "medical_categorical": {"ent_high": 1.8, "std_high": 0.9},
        "numeric_heavy": {"ent_high": 2.0, "std_high": 1.0},
        "ordinal_rich": {"ent_high": 2.2, "std_high": 1.1},
        "large_scale": {"ent_high": 1.5, "std_high": 0.8},
        "general": {"ent_high": 2.5, "std_high": 1.2}
    }
    th = thresholds.get(domain, thresholds["general"])
    
    # INTERVENSI HANYA JIKA baseline di zona ekstrem DAN entropi konsisten
    if base['eta'] <= 5 and mean_ent > th['ent_high']:
        base['eta'] = min(25, 5 + (mean_ent - th['ent_high']) * 8)
        
    num_ents = [calculate_shannon_entropy(df[c]) for c, t in attr_types.items() if t == "Numeric"]
    if num_ents and np.std(num_ents) > th['std_high'] and base['n_bins_numeric'] <= 5:
        base['n_bins_numeric'] = min(15, 5 + int(np.std(num_ents) * 4))
        
    max_nom_card = max([df[c].nunique() for c, t in attr_types.items() if t == "Nominal"] or [0])
    if max_nom_card > 30 and base['dist_cons'] == 0.1:
        base['dist_cons'] = 0.2
        
    return base

### Post-Hoc Merging

In [ ]:
class PostHocMergerOLD:
    """
    Post-hoc cluster merging stabil dengan 4 rekomendasi perbaikan:
    1. Guard Conditions (min_K, max_merge_ratio)
    2. EW-LMD Weighted Distance untuk Centroid
    3. Adaptive Threshold berbasis distribusi pairwise distance
    4. Rollback Mechanism (Validasi Compactness vs Separation)
    """
    
    def __init__(self, model, X: pd.DataFrame, D_ewlmd: np.ndarray):
        self.model = model
        self.X = X
        self.D_ewlmd = D_ewlmd
        self.labels_original = model.labels_.copy()
        self.centers_original = model.centers_.copy()
        self.K_original = len(np.unique(self.labels_original))
        
    def _compute_centroid_distance_ewlmd(self, c1: int, c2: int) -> float:
        """
        Menghitung jarak antar centroid menggunakan metrik EW-LMD berbobot.
        Ini lebih akurat daripada Euclidean untuk data mixed-type.
        """
        center1 = self.centers_original.iloc[c1]
        center2 = self.centers_original.iloc[c2]
        
        total_dist = 0.0
        for col in self.X.columns:
            w = self.model.attr_weights.get(col, 1.0)
            d_attr = self.model._dist_attr(col, center1[col], center2[col])
            total_dist += w * d_attr
        return total_dist

    def _compute_cluster_radius(self, label: int, labels: np.ndarray) -> float:
        """
        Menghitung radius rata-rata cluster (compactness).
        Digunakan untuk validasi merge (Rollback mechanism).
        """
        mask = (labels == label)
        idx_cluster = np.where(mask)[0]
        if len(idx_cluster) < 2: return 0.0
        
        # Hitung rata-rata jarak titik ke centroid (approximation using pairwise)
        # Atau lebih akurat: gunakan max distance dari centroid
        center = self.centers_original.loc[label]
        max_dist = 0.0
        count = 0
        
        # Sampling untuk efisiensi jika cluster besar
        indices = idx_cluster
        if len(indices) > 50:
            indices = np.random.choice(indices, 50, replace=False)
            
        for i in indices:
            row = self.X.iloc[i]
            d = 0.0
            for col in self.X.columns:
                w = self.model.attr_weights.get(col, 1.0)
                d += w * self.model._dist_attr(col, row[col], center[col])
            if d > max_dist: max_dist = d
            count += 1
            
        return max_dist

    def _compute_all_centroid_distances(self, labels: np.ndarray) -> Dict[Tuple[int, int], float]:
        """Hitung semua pairwise distance antar centroid aktif."""
        unique_labels = np.unique(labels)
        dists = {}
        for i, c1 in enumerate(unique_labels):
            for c2 in unique_labels[i+1:]:
                d = self._compute_centroid_distance_ewlmd(c1, c2)
                dists[(c1, c2)] = d
        return dists

    def _validate_merge(self, c1: int, c2: int, dist: float, labels: np.ndarray) -> bool:
        """
        REKOMENDASI 4: Rollback/Validation Mechanism.
        Merge DITOLAK jika jarak antar centroid lebih besar dari 'ukuran' cluster.
        Prinsip: Jangan gabungkan dua gunung jika lembah di antaranya sangat dalam/lebar.
        """
        r1 = self._compute_cluster_radius(c1, labels)
        r2 = self._compute_cluster_radius(c2, labels)
        
        # Jika jarak antar pusat > radius terbesar, berarti mereka terpisah jauh.
        # Jangan merge! (Rollback logic)
        separation_ratio = dist / max(r1, r2, 1e-12)
        
        # Ambang batas keamanan: jika pusat terpisah > 1.2x radius, jangan merge
        if separation_ratio > 1.2: 
            return False 
        return True

    def smart_merge_stable(self, min_K: int = 2, max_merge_ratio: float = 0.5) -> Dict:
        """
        REKOMENDASI 1 & 3: Merge loop dengan Guard Conditions dan Adaptive Threshold.
        """
        labels = self.labels_original.copy()
        merge_history = []
        K_start = len(np.unique(labels))
        merged_count = 0
        
        # Hitung batas maksimum penggabungan (Guard Condition)
        # Contoh: Jika K=6 dan min_K=2, max allowed merge = 4
        max_allowed_merges = int((K_start - min_K) * max_merge_ratio)
        if max_allowed_merges < 1: max_allowed_merges = 1 # Izinkan minimal 1 merge jika memungkinkan
        
        iteration = 0
        while iteration < 20: # Safety break
            unique_labels = np.unique(labels)
            K_current = len(unique_labels)
            
            # 1. CHECK GUARD CONDITIONS
            if K_current <= min_K:
                print(f"   -> [Guard] Reached min_K ({min_K}). Stopping.")
                break
                
            if merged_count >= max_allowed_merges:
                print(f"   -> [Guard] Reached max_merge_ratio limit. Stopping.")
                break

            # 2. COMPUTE ADAPTIVE THRESHOLD
            # REKOMENDASI 3: Threshold dinamis berdasarkan median distance
            all_dists = self._compute_all_centroid_distances(labels)
            if not all_dists: break
            
            dist_values = list(all_dists.values())
            median_dist = np.median(dist_values)
            
            # Adaptive threshold: Hanya pertimbangkan merge jika jarak < 80% median
            # Ini mencegah merge cluster yang jauh (outliers)
            adaptive_threshold = median_dist * 0.8 
            
            # 3. FIND BEST CANDIDATE
            best_pair = None
            best_dist = np.inf
            
            for (c1, c2), d in all_dists.items():
                # Hanya pertimbangkan jika di bawah threshold adaptif
                if d < adaptive_threshold and d < best_dist:
                    best_dist = d
                    best_pair = (c1, c2)
            
            # 4. VALIDATE & APPLY (Rollback Check)
            if best_pair:
                c1, c2 = best_pair
                
                # REKOMENDASI 4: Validasi sebelum merge
                if self._validate_merge(c1, c2, best_dist, labels):
                    # Lakukan Merge
                    labels[labels == c2] = c1
                    merge_history.append((c1, c2))
                    merged_count += 1
                    
                    # Update centroids (pindahkan centroid c2 ke c1 agar jarak terhitung ulang)
                    # Sederhana: rata-rata posisi (untuk numerik) atau mode (kategorikal)
                    # Namun, untuk iterasi berikutnya, kita bisa biarkan label berubah
                    # dan recomputing centers dilakukan di akhir.
                else:
                    # VALIDATION FAILED (Rollback)
                    # Tidak ada pasangan yang valid untuk digabung di bawah threshold ini
                    print(f"   -> [Validation] No valid merges found (Ratio check failed).")
                    break
            else:
                # Tidak ada jarak di bawah threshold adaptif
                print(f"   -> [Threshold] No pairs below adaptive threshold ({adaptive_threshold:.3f}).")
                break
                
            iteration += 1
            
        # Update state model
        self.labels_merged = labels
        self.K_final = len(np.unique(labels))
        self.merge_history = merge_history
        
        # Recompute centers untuk label baru
        print("[Post-Hoc] Recomputing centers for merged labels...")
        self.model.centers_ = self.model._update_centers_refinement(self.X, self.labels_merged)
        self.model.labels_ = self.labels_merged
        self.model.K_auto_ = self.K_final
        
        return {
            'labels_merged': self.labels_merged,
            'K_before': K_start,
            'K_final': self.K_final,
            'merge_history': merge_history,
            'metrics': {
                'merges_applied': merged_count,
                'rejected_by_guard': K_start - self.K_final - merged_count # Estimasi
            }
        }

In [ ]:
# ==============================================================================
# POST-HOC MERGER (OPTIMIZED FOR OVER-SEGMENTATION)
# ==============================================================================
class PostHocMerger:
    def __init__(self, model, X: pd.DataFrame, D_ewlmd: np.ndarray):
        self.model = model
        self.X = X
        self.D_ewlmd = D_ewlmd
        self.labels_original = model.labels_.copy()
        self.centers_original = model.centers_.copy()
        self.K_original = len(np.unique(self.labels_original))
        
    def _compute_centroid_distance_ewlmd(self, c1: int, c2: int) -> float:
        center1 = self.centers_original.iloc[c1]
        center2 = self.centers_original.iloc[c2]
        total_dist = 0.0
        for col in self.X.columns:
            # w = self.model.attr_weights.get(col, 1.0)
            w = 1
            d_attr = self.model._dist_attr(col, center1[col], center2[col])
            total_dist += w * d_attr
        return total_dist

    def _compute_cluster_radius(self, label: int, labels: np.ndarray) -> float:
        mask = (labels == label)
        idx_cluster = np.where(mask)[0]
        if len(idx_cluster) < 2: return 0.0
        
        center = self.centers_original.loc[label]
        max_dist = 0.0
        indices = idx_cluster if len(idx_cluster) <= 50 else np.random.choice(idx_cluster, 50, replace=False)
        
        for i in indices:
            row = self.X.iloc[i]
            w = 1
            # d = sum(self.model.attr_weights.get(col, 1.0) * self.model._dist_attr(col, row[col], center[col]) for col in self.X.columns)
            d = sum(w * self.model._dist_attr(col, row[col], center[col]) for col in self.X.columns)
            if d > max_dist: max_dist = d
        return max_dist

    def _validate_merge(self, c1: int, c2: int, dist: float, labels: np.ndarray) -> bool:
        r1 = self._compute_cluster_radius(c1, labels)
        r2 = self._compute_cluster_radius(c2, labels)
        
        # PERBAIKAN: Relaksasi separation_ratio dari 1.2 menjadi 1.5
        separation_ratio = dist / max(r1, r2, 1e-12)
        if separation_ratio > 1.5: 
            return False 
        return True

    def smart_merge_stable(self, min_K: int = 2, max_merge_ratio: float = 1.0) -> Dict:
        labels = self.labels_original.copy()
        merge_history = []
        K_start = len(np.unique(labels))
        merged_count = 0
        
        max_allowed_merges = int((K_start - min_K) * max_merge_ratio)
        if max_allowed_merges < 1 and K_start > min_K: 
            max_allowed_merges = K_start - min_K 
        
        iteration = 0
        while iteration < 50:
            unique_labels = np.unique(labels)
            K_current = len(unique_labels)
            
            if K_current <= min_K:
                if self.model.verbose: print(f"   -> [Guard] Reached min_K ({min_K}). Stopping.")
                break
            if merged_count >= max_allowed_merges:
                if self.model.verbose: print(f"   -> [Guard] Reached max_merge limit. Stopping.")
                break

            all_dists = self._compute_all_centroid_distances(labels)
            if not all_dists: break
            
            dist_values = list(all_dists.values())
            
            # PERBAIKAN KRITIS: Gunakan Persentil ke-15, bukan Median * 0.8
            adaptive_threshold = np.percentile(dist_values, 15) 
            
            best_pair = None
            best_dist = np.inf
            
            for (c1, c2), d in all_dists.items():
                if d < adaptive_threshold and d < best_dist:
                    best_dist = d
                    best_pair = (c1, c2)
            
            if best_pair:
                c1, c2 = best_pair
                if self._validate_merge(c1, c2, best_dist, labels):
                    labels[labels == c2] = c1
                    merge_history.append((c1, c2))
                    merged_count += 1
                    
                    # self.centers_original.loc[c1] = self.model._update_centers_refinement(
                    #     self.X[labels == c1], labels[labels == c1]
                    # ).iloc[0]
                    # Gabungkan member c1 dan c2, lalu hitung mean/mode secara manual
                    merged_mask = (labels == c1) | (labels == c2)
                    merged_cluster_X = self.X[merged_mask]
                    new_center_dict = {}
                    for col in self.X.columns:
                        if self.model.attr_types[col] in ('Nominal', 'Ordinal'):
                            new_center_dict[col] = _mode(merged_cluster_X[col])
                        else:
                            new_center_dict[col] = merged_cluster_X[col].mean()
                    self.centers_original.loc[c1] = new_center_dict
                else:
                    if self.model.verbose: print(f"   -> [Validation] Closest pair rejected (Ratio > 1.5). Stopping.")
                    break
            else:
                if self.model.verbose: print(f"   -> [Threshold] No pairs below 15th percentile ({adaptive_threshold:.3f}). Stopping.")
                break
                
            iteration += 1
            
        self.labels_merged = labels
        self.K_final = len(np.unique(labels))
        self.merge_history = merge_history
        
        self.model.centers_ = self.model._update_centers_refinement(self.X, self.labels_merged)
        self.model.labels_ = self.labels_merged
        self.model.K_auto_ = self.K_final
        
        return {
            'labels_merged': self.labels_merged,
            'K_before': K_start,
            'K_final': self.K_final,
            'merge_history': merge_history,
            'metrics': {'merges_applied': merged_count}
        }

    def _compute_all_centroid_distances(self, labels: np.ndarray) -> Dict[Tuple[int, int], float]:
        unique_labels = np.unique(labels)
        dists = {}
        for i, c1 in enumerate(unique_labels):
            for c2 in unique_labels[i+1:]:
                d = self._compute_centroid_distance_ewlmd(c1, c2)
                dists[(c1, c2)] = d
        return dists

### V1 Bombing + LMD Algorithm

In [ ]:
# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================
def _mode(series: pd.Series) -> Any:
    """Deterministic mode with lexicographic tie-break."""
    vc = series.value_counts(dropna=False)
    if vc.empty: return np.nan
    m = vc.max()
    cands = sorted([v for v, c in vc.items() if c == m], key=lambda x: str(x))
    return cands[0]

def compute_merge_metrics(labels: np.ndarray, jnng_adj: csr_matrix, mixture_density_P: np.ndarray) -> Dict[str, float]:
    """Menghitung statistik struktural untuk trigger Post-Hoc Merger."""
    unique_labels = np.unique(labels)
    K = len(unique_labels)
    N = len(labels)
    cluster_sizes = np.array([np.sum(labels == c) for c in unique_labels])
    
    max_da = 0.0
    adj_list = jnng_adj.tolil().rows
    for i_idx, c1 in enumerate(unique_labels):
        mask_c1 = (labels == c1)
        idx_c1 = np.where(mask_c1)[0]
        for c2 in unique_labels[i_idx+1:]:
            mask_c2 = (labels == c2)
            idx_c2 = np.where(mask_c2)[0]
            if len(idx_c1) == 0 or len(idx_c2) == 0: continue
            
            nap = sum(1 for u in idx_c1 for v in adj_list[u] if mask_c2[v])
            da = nap / min(len(idx_c1), len(idx_c2)) if min(len(idx_c1), len(idx_c2)) > 0 else 0
            if da > max_da: max_da = da
            
    intra_cvs = []
    for c in unique_labels:
        mask = (labels == c)
        if np.sum(mask) < 2: continue
        density_in_cluster = mixture_density_P[mask]
        mean_d = np.mean(density_in_cluster)
        std_d = np.std(density_in_cluster)
        intra_cvs.append(std_d / (mean_d + 1e-12))
    avg_cv = np.mean(intra_cvs) if intra_cvs else 0.0

    return {'K': K, 'N': N, 'cluster_sizes': cluster_sizes, 'max_da': max_da, 'intra_density_cv': avg_cv}

def should_merge_parameter_free(metrics: Dict[str, float], T2: float = 0.2) -> bool:
    """Trigger condition berbasis statistik mapan, tanpa hyperparameter baru."""
    K, N = metrics['K'], metrics['N']
    sizes, max_da, cv = metrics['cluster_sizes'], metrics['max_da'], metrics['intra_density_cv']
    
    if K > 2 * np.sqrt(N): return True
    if np.min(sizes) > 0 and np.max(sizes) / np.min(sizes) > 10: return True
    if max_da > T2: return True
    if cv > 0.5: return True
    return False

# ==============================================================================
# MAIN HYBRID CLASS (PURE BOMBING ARCHITECTURE)
# ==============================================================================
class LMDBombingHybrid_V1:
    def __init__(self, attr_types: dict, ordinal_orders: dict = None,
                 max_iter_refine: int = 20, random_state: int = 42, verbose: bool = True):
        
        self.attr_types = attr_types.copy()
        self.ordinal_orders = (ordinal_orders or {}).copy()
        self.dis_const = 0.3
        self.eta_cat = 20.0
        self.eta_num = 20.0 
        self.alpha_reg = 0.8
        self.tau_weight = 1e-3
        self.n_bins_numeric = 10
        self.max_pairs_per_cluster = 200
        self.max_iter_refine = max_iter_refine
        self.conv_threshold = 0.98
        self._rng = np.random.default_rng(random_state)
        self.verbose = verbose
        
        # Bombing Parameters
        self.k1, self.k2 = 2, 10
        self.dc, self.tau, self.T1, self.T2 = 0.15, 0.6, 0.9, 0.2
        
        # self.attr_weights = {}
        self.nominal_dist, self.ordinal_adj_dist = {}, {}
        self.num_min, self.num_max = {}, {}
        self.num_bin_psi = {}
        self.nominal_phi, self.ordinal_phi, self.numeric_phi = {}, {}, {}
        
        self.labels_ = None
        self.centers_ = None
        self.K_auto_ = None
        self._reachable_indices = {} 

    def _compute_structure(self, X: pd.DataFrame) -> dict:
        counts = {"Numeric": 0, "Ordinal": 0, "Nominal": 0}
        for t in self.attr_types.values():
            counts["Nominal" if t not in ["Numeric", "Ordinal"] else t] += 1
        max_nom = max([X[c].nunique() for c, t in self.attr_types.items() if t == "Nominal"] or [0])
        return {
            'n_rows': len(X), 'n_cols': len(X.columns),
            'n_numeric': counts['Numeric'], 'n_ordinal': counts['Ordinal'], 'n_nominal': counts['Nominal'],
            'max_nominal_card': max_nom,
            'is_wide': len(X.columns) > 20,
            'is_numeric_heavy': counts['Numeric'] > counts['Nominal'],
            'is_nominal_heavy': counts['Nominal'] > counts['Numeric']
        }
    # --- Distance Helpers ---
    def _init_distances(self, X: pd.DataFrame):
        for col in X.columns:
            t = self.attr_types[col]
            if t == "Nominal":
                vals = list(pd.unique(X[col]))
                self.nominal_dist[col] = {(a,b): self.dis_const for a in vals for b in vals if a!=b}
                self.nominal_phi[col] = {(a,b): 0.0 for a in vals for b in vals if a!=b}
            elif t == "Ordinal":
                order = list(self.ordinal_orders[col])
                self.ordinal_adj_dist[col] = {order[i]: self.dis_const for i in range(len(order)-1)}
                self.ordinal_phi[col] = {order[i]: 0.0 for i in range(len(order)-1)}
            elif t == "Numeric":
                v = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())
                self.num_min[col], self.num_max[col] = float(v.min()), float(v.max())
                self.num_bin_psi[col] = np.zeros(self.n_bins_numeric, dtype=float)
                self.numeric_phi[col] = np.zeros(self.n_bins_numeric, dtype=float)

    def _dist_attr(self, col: str, a: Any, b: Any) -> float:
        t = self.attr_types[col]
        if t == "Nominal":
            if a == b: return 0.0
            mp = self.nominal_dist[col]
            return mp.get((a,b), mp.get((b,a), self.dis_const))
        elif t == "Ordinal":
            if a == b: return 0.0
            order = list(self.ordinal_orders[col])
            try: ia, ib = order.index(a), order.index(b)
            except ValueError: return 0.0
            if ia > ib: ia, ib = ib, ia
            return sum(self.ordinal_adj_dist[col].get(order[i], self.dis_const) for i in range(ia, ib))
        elif t == "Numeric":
            vmin, vmax = self.num_min[col], self.num_max[col]
            if vmax <= vmin: return 0.0
            try: va, vb = float(a), float(b)
            except Exception: va, vb = (vmin+vmax)/2.0, (vmin+vmax)/2.0
            
            z_a = np.clip((va - vmin) / (vmax - vmin), 0.0, 1.0)
            z_b = np.clip((vb - vmin) / (vmax - vmin), 0.0, 1.0)
            
            # PERBAIKAN MURNI BOMBING: Absolute Difference (BUKAN Squared)
            # Ini menjaga konektivitas manifold jNNG agar tidak terputus (sparse)
            delta = abs(z_a - z_b)
            
            psi_vec = self.num_bin_psi[col]
            if psi_vec is None or len(psi_vec) == 0: return delta
            bidx = int(np.clip(delta * self.n_bins_numeric, 0, self.n_bins_numeric - 1))
            scale = max(0.1, 1.0 + psi_vec[bidx])
            return delta * scale
        return 0.0

    def distance(self, x: pd.Series, y: pd.Series) -> float:
        d = 0.0
        for col in y.index:
            d += self._dist_attr(col, x[col], y[col])
        return float(d)

    def _compute_distance_matrix(self, X: pd.DataFrame) -> np.ndarray:
        n = len(X)
        D = np.zeros((n, n))
        cols = list(X.columns)
        for i in range(n):
            for j in range(i+1, n):
                d = sum(self._dist_attr(col, X.iloc[i][col], X.iloc[j][col]) for col in cols)
                D[i, j] = D[j, i] = d
        return D

    # --- Bombing Core Mechanics ---
    def _build_jnng(self, D_scaled: np.ndarray):
        n = len(D_scaled)
        
        k1_idx = np.argpartition(D_scaled, self.k1 + 1, axis=1)[:, 1:self.k1 + 1] # +1 untuk skip diri sendiri
        k1_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k1_idx[i]:
                k1_adj[i, j] = k1_adj[j, i] = True
                
        # --- k2 (MUTUAL k-NN FIX) ---
        # Ambil k2+1, lalu slice [:, 1:] untuk membuang jarak ke diri sendiri (yang bernilai 0)
        k2_idx = np.argpartition(D_scaled, self.k2 + 1, axis=1)[:, 1:self.k2 + 1] 
        k2_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k2_idx[i]:
                # Cek mutualitas yang sebenarnya (j != i sudah dijamin oleh slice [1:])
                if i in k2_idx[j]: 
                    k2_adj[i, j] = k2_adj[j, i] = True
                
        jnng_adj = k1_adj | k2_adj
        jnng_sparse = csr_matrix(jnng_adj.astype(float) * D_scaled)
        
        # Geodesic distance
        geo_dist = shortest_path(jnng_sparse, method='D', directed=False)
        geo_dist[np.isinf(geo_dist)] = geo_dist[np.isfinite(geo_dist)].max() * 1.5
        
        return jnng_sparse, geo_dist, csr_matrix(jnng_adj.astype(float))

    def _gaussian_kernel_density_old(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        PERBAIKAN MURNI BOMBING: Local Scaling (Zelnik-Perona style).
        Mencegah density flattening dengan membuat bandwidth adaptif terhadap kepadatan lokal.
        """
        n = geo_dist.shape[0]
        k_local = min(7, n - 1)
        
        sigmas = np.partition(geo_dist, k_local, axis=1)[:, k_local]
        sigmas[sigmas == 0] = 1e-12
        
        sigma_matrix = np.outer(sigmas, sigmas)
        rho = np.sum(np.exp(-geo_dist**2 / (2 * sigma_matrix + 1e-12)), axis=1)
        return rho

    def _gaussian_kernel_density(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        Mengikuti Eq. 9 dari Paper Fei et al. (2025).
        Menggunakan varians tetap (dc) untuk menjaga validitas Two-Round Sigmoid.
        """
        # self.dc = 0.15 (atau sesuai parameter default paper)
        rho = np.sum(np.exp(-(geo_dist ** 2) / (2 * (self.dc ** 2))), axis=1)
        return rho

    def _two_round_sigmoid_correction(self, rho_gjnn: np.ndarray) -> np.ndarray:
        rho_max = rho_gjnn.max()
        slope, delta = 10/rho_max, rho_max/2
        rho_tilde = 1 / (1 + np.exp(-slope * (rho_gjnn - delta)))
        mask = rho_tilde < self.T1
        if np.any(mask):
            rho_max_below = rho_gjnn[mask].max()
            s2, d2 = 10/rho_max_below, rho_max_below/2
            rho_tilde[mask] = 1 / (1 + np.exp(-s2 * (rho_tilde[mask] - d2)))
        return rho_tilde

    def _local_neighbor_density(self, D_scaled: np.ndarray, r: float, valid_mask: np.ndarray) -> np.ndarray:
        """Menghitung rho_r (Eq. 16) hanya berdasarkan titik-titik yang valid (belum di-cluster)."""
        n = len(D_scaled)
        rho_r = np.zeros(n)
        for i in range(n):
            if not valid_mask[i]: continue
            # Hitung tetangga dalam radius r dari titik i
            neighbors_in_r = np.sum((D_scaled[i] <= r) & valid_mask)
            rho_r[i] = neighbors_in_r / self.k2
        return rho_r

    def _bfs_generalized_cluster(self, adj_mat: csr_matrix, core_idx: int, P: np.ndarray, visited_mask: np.ndarray) -> Tuple[List[int], List[int]]:
        R_t, B_t = [], []
        queue = [core_idx]
        visited_mask[core_idx] = True
        R_t.append(core_idx)
        adj_list = adj_mat.tolil().rows
        first_boundary = set()
        
        while queue:
            u = queue.pop(0)
            for v in adj_list[u]:
                if visited_mask[v]: continue
                if P[v] >= self.tau:
                    visited_mask[v] = True
                    R_t.append(v)
                    queue.append(v)
                else:
                    first_boundary.add(v)
                    
        second_boundary = set()
        for v in first_boundary:
            for w in adj_list[v]:
                if not visited_mask[w] and w not in first_boundary and w not in R_t:
                    second_boundary.add(w)
        B_t = list(first_boundary | second_boundary)
        return R_t, B_t

    def _degree_of_adjacency(self, adj_mat: csr_matrix, R_t: List[int], B_t: List[int], existing_clusters: dict) -> Tuple[float, int]:
        if not existing_clusters: return 0.0, -1
        C_star = len(R_t) + len(B_t)
        if C_star == 0: return 0.0, -1
        adj_list = adj_mat.tolil().rows
        max_nap, best_s = 0, -1
        for s, (_, B_s) in existing_clusters.items():
            nap = sum(1 for u in R_t for v in adj_list[u] if v in B_s)
            da = nap / C_star
            if da > max_nap: max_nap, best_s = da, s
        return max_nap, best_s

    def _run_bombing_phase(self, X: pd.DataFrame, D_scaled: np.ndarray, geo_dist: np.ndarray, rho_tilde: np.ndarray, jnng_adj: csr_matrix):
        n = len(X)
        labels = np.full(n, -1, dtype=int)
        visited = np.zeros(n, dtype=bool)
        existing_clusters = {}
        t = 0
        self._reachable_indices = {}
        
        # Copy rho_tilde agar bisa dimanipulasi per iterasi
        current_rho_tilde = rho_tilde.copy()
        
        while not np.all(visited):
            cand = ~visited
            if not np.any(cand): break
            
            # 1. Tentukan Core Point dari titik yang BELUM dikunjungi
            core_idx = np.argmax(current_rho_tilde * cand)
            if current_rho_tilde[core_idx] < 1e-6: break

            # 2. Hitung Ulang Local Neighbor Density (rho_r) berdasarkan sisa titik
            # Paper: r adalah jarak ke k-th nearest neighbor dari core point
            unvisited_indices = np.where(cand)[0]
            if len(unvisited_indices) <= self.k2:
                r = np.max(D_scaled[core_idx, unvisited_indices])
            else:
                r = np.partition(D_scaled[core_idx, unvisited_indices], self.k2)[self.k2]
                
            rho_r = self._local_neighbor_density(D_scaled, r, cand)

            # 3. Mixture Density (Eq. 17)
            P = current_rho_tilde + rho_r
            P_min, P_max = P.min(), P.max()
            if P_max > P_min: 
                P = (P - P_min) / (P_max - P_min)

            # 4. BFS dengan TAU TETAP (0.6) - Ini adalah klaim utama paper!
            R_t, B_t = self._bfs_generalized_cluster(jnng_adj, core_idx, P, visited.copy())
            
            # Tandai titik yang di-cluster dan boundary sebagai visited (dihapus dari X)
            idxs = R_t + B_t
            for idx in idxs: 
                visited[idx] = True

            # 5. Hitung Degree of Adjacency (DA)
            da, best_s = self._degree_of_adjacency(jnng_adj, R_t, B_t, existing_clusters)

            # 6. Aturan Penggabungan (Generalized Cluster)
            if da >= self.T2 and best_s != -1:
                labels[idxs] = best_s
            else:
                labels[idxs] = t
                existing_clusters[t] = (R_t, B_t)
                self._reachable_indices[t] = R_t
                t += 1

            # 7. UPDATE DINAMIS: Nol-kan densitas titik yang sudah di-cluster
            # Agar iterasi berikutnya hanya mencari core point dari sisa dataset
            current_rho_tilde[visited] = 0.0
            
            # Opsional: Recalculate Sigmoid Round 1 & 2 untuk sisa titik jika diperlukan
            # (Paper melakukan recalculate rho_max dari sisa titik)
            rho_max_rem = current_rho_tilde.max()
            if rho_max_rem > 1e-6:
                slope, delta = 10 / rho_max_rem, rho_max_rem / 2
                # Terapkan ulang sigmoid ke titik yang belum visited
                current_rho_tilde[~visited] = 1 / (1 + np.exp(-slope * (current_rho_tilde[~visited] - delta)))

        # Final Clean-up (Assign unvisited/boundary ke nearest visited point)
        unvisited = np.where(labels == -1)[0]
        if len(unvisited) > 0:
            visited_pts = np.where(labels != -1)[0]
            if len(visited_pts) > 0:
                for u in unvisited:
                    labels[u] = labels[visited_pts[np.argmin(D_scaled[u, visited_pts])]]
            else:
                labels[:n//2] = 0
                labels[n//2:] = 1

        self.K_auto_ = len(np.unique(labels[labels != -1]))
        return self.K_auto_, labels

    def _compute_initial_centroids(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        unique_labels = np.unique(labels)
        reachable_dict = getattr(self, '_reachable_indices', {})
        n_points = len(X)
        
        for lbl in unique_labels:
            r_indices = reachable_dict.get(lbl, [])
            if len(r_indices) > 0:
                core_mask = np.zeros(n_points, dtype=bool)
                core_mask[r_indices] = True
                cl = X[core_mask] # Density-Excluded Initialization
            else:
                cl = X[labels == lbl]
                
            c = {}
            for col in X.columns:
                if self.attr_types[col] in ('Nominal', 'Ordinal'):
                    c[col] = _mode(cl[col])
                else:
                    c[col] = cl[col].mean()
            centers.append(c)
        return pd.DataFrame(centers, columns=X.columns)

    # --- EW-LMD Refinement Helpers ---
    def _update_nominal_ordinal_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] not in ('Nominal', 'Ordinal'): continue
            vals = list(pd.unique(X[col]))
            if len(vals) < 2: continue
            P = {}
            for a in vals:
                for b in vals:
                    if a == b: continue
                    val_sum = sum(((X.iloc[np.where(labels==c)[0]][col]==a).sum()/(len(X[labels==c])+eps)) * 
                                  ((X.iloc[np.where(labels==c)[0]][col]==b).sum()/(len(X[labels==c])+eps)) 
                                  for c in range(k) if len(X[labels==c]) > 0)
                    P[(a,b)] = val_sum
            if not P: continue
            P_vals = list(P.values())
            P_max, P_min = max(P_vals), min(P_vals)
            if abs(P_max-P_min) < eps: continue
            
            phi_raw = {key: (-P_max/self.eta_cat if v==P_max else (1-P_min)/self.eta_cat if v==P_min else 0.0) for key, v in P.items()}
            phi_old = self.nominal_phi[col] if self.attr_types[col]=='Nominal' else self.ordinal_phi[col]
            phi_new = {key: np.clip(self.alpha_reg*phi_old.get(key,0.0) + (1-self.alpha_reg)*phi_raw[key], -1, 1) for key in phi_raw}
            
            if self.attr_types[col]=='Nominal':
                self.nominal_phi[col] = phi_new
                self.nominal_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}
            else:
                self.ordinal_phi[col] = phi_new
                self.ordinal_adj_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}

    def _update_numeric_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        B = self.n_bins_numeric
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] != 'Numeric': continue
            if self.num_min[col] >= self.num_max[col]: continue
            z_full = np.clip((pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy() - self.num_min[col]) / (self.num_max[col] - self.num_min[col]), 0.0, 1.0)
            P_sum = np.zeros(B)
            
            for c in range(k):
                idx = np.where(labels==c)[0]
                m = len(idx)
                if m < 2: continue
                if m > self.max_pairs_per_cluster:
                    idx_sample = self._rng.choice(idx, size=self.max_pairs_per_cluster, replace=False)
                else:
                    idx_sample = idx
                m_s = len(idx_sample)
                if m_s < 2: continue
                
                bin_counts = np.zeros(B, dtype=float)
                for i in range(m_s):
                    for j in range(i + 1, m_s):
                        delta = abs(z_full[idx_sample[i]] - z_full[idx_sample[j]])
                        bidx = int(np.clip(delta * B, 0, B - 1))
                        bin_counts[bidx] += 1.0
                total_pairs = bin_counts.sum()
                if total_pairs <= 0: continue
                probs = bin_counts / (total_pairs + eps)
                P_sum += probs * probs

            if np.all(P_sum <= eps): continue
            P_max, P_min = P_sum.max(), P_sum.min()
            if abs(P_max - P_min) < eps: continue

            phi_raw_vec = np.zeros(B, dtype=float)
            for bidx in range(B):
                val = P_sum[bidx]
                if val == P_max: phi = -P_max / self.eta_num
                elif val == P_min: phi = (1.0 - P_min) / self.eta_num
                else: phi = 0.0
                phi_raw_vec[bidx] = phi

            phi_old_vec = self.numeric_phi[col]
            if phi_old_vec.shape[0] != B: phi_old_vec = np.zeros(B, dtype=float)
            phi_new_vec = np.clip(self.alpha_reg * phi_old_vec + (1.0 - self.alpha_reg) * phi_raw_vec, -1.0, 1.0)
            self.numeric_phi[col] = phi_new_vec
            self.num_bin_psi[col] = phi_new_vec.copy()

    def _assign_refinement(self, X: pd.DataFrame, jnng_adj: csr_matrix) -> np.ndarray:
        """
        REFINEMENT DENGAN KENDALA KONEKTIVITAS jNNG (FIX CELAH #1)
        Titik hanya boleh pindah klaster jika memiliki edge jNNG dengan anggota klaster target.
        """
        n = len(X)
        labels = np.empty(n, dtype=int)
        
        # 1. Pre-compute anggota klaster saat ini untuk lookup O(1)
        cluster_members = {c: set(np.where(self.labels_ == c)[0]) for c in range(self.K_auto_)}
        
        # 2. Konversi adjacency matrix ke list of sets untuk akses tetangga yang cepat
        adj_list = jnng_adj.tolil().rows

        for i in range(n):
            # Hitung jarak EW-LMD ke semua sentroid
            dists = [
                sum(self._dist_attr(col, X.iloc[i][col], self.centers_.iloc[c][col]) 
                    for col in X.columns) 
                for c in range(self.K_auto_)
            ]
            best_c = int(np.argmin(dists)) # Klaster terdekat secara geometris
            
            # Cek konektivitas topologi jNNG
            neighbors_i = set(adj_list[i])
            
            # Cari klaster mana saja yang memiliki anggota terhubung dengan titik i
            connected_clusters = [
                c for c, members in cluster_members.items() 
                if neighbors_i.intersection(members)
            ]
            
            # 3. Keputusan Alokasi
            if connected_clusters:
                # Jika ada koneksi, pilih klaster yang TERHUBUNG yang jaraknya paling minimum
                labels[i] = min(connected_clusters, key=lambda c: dists[c])
            else:
                # Fallback: Titik terisolasi (tidak ada edge ke klaster manapun)
                # Gunakan jarak geometris murni agar tidak error
                labels[i] = best_c

        return labels

    def _update_centers_refinement(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        for c in range(self.K_auto_):
            cl = X[labels==c]
            if cl.empty: 
                centers.append(self.centers_.iloc[c].to_dict())
                continue
            c_dict = {col: _mode(cl[col]) if self.attr_types[col] in ('Nominal','Ordinal') else cl[col].mean() for col in X.columns}
            centers.append(c_dict)
        return pd.DataFrame(centers)

    def fit(self, df: pd.DataFrame):
        X = df.drop(columns=["Label"], errors="ignore").copy()
        for col in X.columns:
            if self.attr_types[col] == "Numeric":
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())

        # 1. HYBRID PARAMETER TUNING
        structure = self._compute_structure(X)
        params = hybrid_selection_conservative(X, self.attr_types, structure)
        self.dis_const = params['dist_cons']
        self.eta_cat = self.eta_num = params['eta']
        self.n_bins_numeric = params['n_bins_numeric']  # ← PERBAIKAN
        self.auto_params_log = params.copy()
        if self.verbose: print(f"[AUTONOMOUS] Hybrid-tuned: {self.auto_params_log}")

        print("[Phase 0] Computing initializing distances...")
        # self._compute_entropy_weights(X)
        self._init_distances(X)
        
        print("[Phase 1] Building jNNG with LMD distance...")
        D = self._compute_distance_matrix(X)
        n = len(X)
        median_dist = np.median(D[np.triu_indices(n, k=1)])
        D_scaled = D / (median_dist + 1e-12)
        
        jnng_sparse, geo_dist, jnng_adj = self._build_jnng(D_scaled)
        
        # Menggunakan Local Scaling di sini
        rho_gjnn = self._gaussian_kernel_density(geo_dist)
        rho_tilde = self._two_round_sigmoid_correction(rho_gjnn)
        
        print("[Phase 2] Running Bombing process (Automatic K detection)...")
        K_before, labels_bombing = self._run_bombing_phase(X, D_scaled, geo_dist, rho_tilde, jnng_adj)
        print(f"   -> Detected K = {K_before}")
        
        self.labels_ = labels_bombing
        self.centers_ = self._compute_initial_centroids(X, labels_bombing)
        
        print("[Phase 2.5] Evaluating structural coherence for conditional merging...")
        metrics = compute_merge_metrics(labels_bombing, jnng_adj, rho_tilde)
        trigger = should_merge_parameter_free(metrics, T2=self.T2)
        
        if trigger:
            print("   -> Trigger activated. Initializing Post-Hoc Merger...")
            merger = PostHocMerger(self, X, D_scaled)
            merge_results = merger.smart_merge_stable(min_K=2, max_merge_ratio=1.0)
            labels_refined = merge_results['labels_merged']
            K_after = merge_results['K_final']
            print(f"   -> Merge Result: K {merge_results['K_before']} → {K_after}")
        else:
            print("   -> Structure coherent. Skipping merging phase.")
            labels_refined = labels_bombing
            K_after = K_before
            
        self.labels_ = labels_refined
        self.K_auto_ = K_after
        
        print("[Phase 3] Computing density-excluded core centroids...")
        if trigger:
            for lbl in np.unique(labels_refined):
                orig_lbls = np.unique(labels_bombing[labels_refined == lbl])
                r_idx = []
                for ol in orig_lbls:
                    r_idx.extend(self._reachable_indices.get(ol, []))
                self._reachable_indices[lbl] = r_idx
                
        self.centers_ = self._compute_initial_centroids(X, labels_refined)
        
        print("[Phase 4-6] LMD Iterative Refinement Loop...")
        prev_labels = self.labels_.copy()
        best_silhouette = -1.0
        best_labels = self.labels_.copy()
        restart_count = 0
        
        for step in range(self.max_iter_refine):
            current_alpha = 0.5 if step < 2 else self.alpha_reg
            orig_alpha = self.alpha_reg
            self.alpha_reg = current_alpha
            
            self._update_nominal_ordinal_phi(X, self.labels_)
            self._update_numeric_phi(X, self.labels_)
            self.alpha_reg = orig_alpha
            
            self.labels_ = self._assign_refinement(X, jnng_adj)
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
            if step >= 2:
                ari_internal = adjusted_rand_score(prev_labels, self.labels_)
                
                # === PERBAIKAN: TRAP DETECTION via Silhouette ===
                sample_idx = np.random.choice(len(X), min(1000, len(X)), replace=False) if len(X) > 1000 else np.arange(len(X))
                D_sample = np.zeros((len(sample_idx), len(sample_idx)))
                for i, idx_i in enumerate(sample_idx):
                    for j, idx_j in enumerate(sample_idx):
                        # Menggunakan fungsi distance yang sudah ada
                        D_sample[i,j] = self.distance(X.iloc[idx_i], X.iloc[idx_j])
                
                try:
                    current_sil = silhouette_score(D_sample, self.labels_[sample_idx], metric='precomputed')
                except ValueError:
                    current_sil = 0.0
                    
                if current_sil > best_silhouette:
                    best_silhouette = current_sil
                    best_labels = self.labels_.copy()
                    
                print(f"   Iter {step+1}: Internal ARI = {ari_internal:.4f} | Silhouette = {current_sil:.4f}")
                
                if ari_internal > self.conv_threshold:
                    if current_sil < 0.1 and restart_count < 2:
                        print(f"   -> [Trap Detected] Silhouette < 0.1. Perturbing labels to escape local optimum...")
                        noise_mask = np.random.rand(len(X)) < 0.2
                        self.labels_[noise_mask] = self._rng.integers(0, self.K_auto_, size=np.sum(noise_mask))
                        self.centers_ = self._update_centers_refinement(X, self.labels_)
                        restart_count += 1
                        prev_labels = self.labels_.copy()
                        continue
                    else:
                        print("   -> Convergence reached.")
                        break
                        
            prev_labels = self.labels_.copy()

        if best_silhouette > current_sil:
            if self.verbose: print(f"   -> [Recovery] Restoring best labels from Iter with Silhouette = {best_silhouette:.4f}")
            self.labels_ = best_labels
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
        return self

### V2 Bombing + EW-LMD Algorithm without Density-Excluded Init

In [ ]:
# ==============================================================================
# MAIN HYBRID CLASS (PURE BOMBING ARCHITECTURE)
# ==============================================================================
class EWLMDBombingHybrid_V2:
    def __init__(self, attr_types: dict, ordinal_orders: dict = None,
                 max_iter_refine: int = 20, random_state: int = 42, verbose: bool = True):
        
        self.attr_types = attr_types.copy()
        self.ordinal_orders = (ordinal_orders or {}).copy()
        self.dis_const = 0.3
        self.eta_cat = 20.0
        self.eta_num = 20.0 
        self.alpha_reg = 0.8
        self.tau_weight = 1e-3
        self.n_bins_numeric = 10
        self.max_pairs_per_cluster = 200
        self.max_iter_refine = max_iter_refine
        self.conv_threshold = 0.98
        self._rng = np.random.default_rng(random_state)
        self.verbose = verbose
        
        # Bombing Parameters
        self.k1, self.k2 = 2, 10
        self.dc, self.tau, self.T1, self.T2 = 0.15, 0.6, 0.9, 0.2
        
        self.attr_weights = {}
        self.nominal_dist, self.ordinal_adj_dist = {}, {}
        self.num_min, self.num_max = {}, {}
        self.num_bin_psi = {}
        self.nominal_phi, self.ordinal_phi, self.numeric_phi = {}, {}, {}
        
        self.labels_ = None
        self.centers_ = None
        self.K_auto_ = None
        self._reachable_indices = {} 

    def _compute_structure(self, X: pd.DataFrame) -> dict:
        counts = {"Numeric": 0, "Ordinal": 0, "Nominal": 0}
        for t in self.attr_types.values():
            counts["Nominal" if t not in ["Numeric", "Ordinal"] else t] += 1
        max_nom = max([X[c].nunique() for c, t in self.attr_types.items() if t == "Nominal"] or [0])
        return {
            'n_rows': len(X), 'n_cols': len(X.columns),
            'n_numeric': counts['Numeric'], 'n_ordinal': counts['Ordinal'], 'n_nominal': counts['Nominal'],
            'max_nominal_card': max_nom,
            'is_wide': len(X.columns) > 20,
            'is_numeric_heavy': counts['Numeric'] > counts['Nominal'],
            'is_nominal_heavy': counts['Nominal'] > counts['Numeric']
        }
    # --- Distance & Weight Helpers ---
    def _compute_entropy_weights(self, X: pd.DataFrame):
        eps = 1e-12
        raw_weights: Dict[str, float] = {}
        
        for col in X.columns:
            t = self.attr_types[col]
            if t in ("Nominal", "Ordinal"):
                probs = X[col].value_counts(normalize=True)
                H = -np.sum(probs * np.log(probs + eps))
                H_max = np.log(len(probs)) if len(probs) > 1 else 1.0
                informativeness = 1.0 - (H / (H_max + eps))
            elif t == "Numeric":
                vals = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy()
                vmin, vmax = vals.min(), vals.max()
                if vmax > vmin:
                    z = (vals - vmin) / (vmax - vmin)
                    n = len(z)
                    max_pairs = min(self.max_pairs_per_cluster, n * (n - 1) // 2)
                    if max_pairs > 0:
                        idx_i = self._rng.integers(0, n, size=max_pairs)
                        idx_j = self._rng.integers(0, n, size=max_pairs)
                        delta = np.abs(z[idx_i] - z[idx_j])
                        hist, _ = np.histogram(delta, bins=self.n_bins_numeric, range=(0.0, 1.0))
                        probs = hist.astype(float) / (hist.sum() + eps)
                        H = -np.sum(probs * np.log(probs + eps))
                        H_max = np.log(self.n_bins_numeric)
                        informativeness = 1.0 - (H / (H_max + eps))
                    else: informativeness = 0.0
                else: informativeness = 0.0
            else: informativeness = 0.0
            raw_weights[col] = max(informativeness, self.tau_weight)
        
        type_groups = {"Nominal": [], "Ordinal": [], "Numeric": []}
        for col in X.columns:
            t = self.attr_types[col]
            if t in type_groups: type_groups[t].append(col)
            
        n_active_types = len([t for t, cols in type_groups.items() if cols])
        weight_per_type = 1.0 / n_active_types if n_active_types > 0 else 1.0
        
        self.attr_weights = {}
        for t, cols in type_groups.items():
            if not cols: continue
            total_raw = sum(raw_weights[col] for col in cols)
            for col in cols:
                self.attr_weights[col] = weight_per_type * (raw_weights[col] / (total_raw + eps))

    def _init_distances(self, X: pd.DataFrame):
        for col in X.columns:
            t = self.attr_types[col]
            if t == "Nominal":
                vals = list(pd.unique(X[col]))
                self.nominal_dist[col] = {(a,b): self.dis_const for a in vals for b in vals if a!=b}
                self.nominal_phi[col] = {(a,b): 0.0 for a in vals for b in vals if a!=b}
            elif t == "Ordinal":
                order = list(self.ordinal_orders[col])
                self.ordinal_adj_dist[col] = {order[i]: self.dis_const for i in range(len(order)-1)}
                self.ordinal_phi[col] = {order[i]: 0.0 for i in range(len(order)-1)}
            elif t == "Numeric":
                v = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())
                self.num_min[col], self.num_max[col] = float(v.min()), float(v.max())
                self.num_bin_psi[col] = np.zeros(self.n_bins_numeric, dtype=float)
                self.numeric_phi[col] = np.zeros(self.n_bins_numeric, dtype=float)

    def _dist_attr(self, col: str, a: Any, b: Any) -> float:
        t = self.attr_types[col]
        if t == "Nominal":
            if a == b: return 0.0
            mp = self.nominal_dist[col]
            return mp.get((a,b), mp.get((b,a), self.dis_const))
        elif t == "Ordinal":
            if a == b: return 0.0
            order = list(self.ordinal_orders[col])
            try: ia, ib = order.index(a), order.index(b)
            except ValueError: return 0.0
            if ia > ib: ia, ib = ib, ia
            return sum(self.ordinal_adj_dist[col].get(order[i], self.dis_const) for i in range(ia, ib))
        elif t == "Numeric":
            vmin, vmax = self.num_min[col], self.num_max[col]
            if vmax <= vmin: return 0.0
            try: va, vb = float(a), float(b)
            except Exception: va, vb = (vmin+vmax)/2.0, (vmin+vmax)/2.0
            
            z_a = np.clip((va - vmin) / (vmax - vmin), 0.0, 1.0)
            z_b = np.clip((vb - vmin) / (vmax - vmin), 0.0, 1.0)
            
            # PERBAIKAN MURNI BOMBING: Absolute Difference (BUKAN Squared)
            # Ini menjaga konektivitas manifold jNNG agar tidak terputus (sparse)
            delta = abs(z_a - z_b)
            
            psi_vec = self.num_bin_psi[col]
            if psi_vec is None or len(psi_vec) == 0: return delta
            bidx = int(np.clip(delta * self.n_bins_numeric, 0, self.n_bins_numeric - 1))
            scale = max(0.1, 1.0 + psi_vec[bidx])
            return delta * scale
        return 0.0

    def distance(self, x: pd.Series, y: pd.Series) -> float:
        d = 0.0
        for col in y.index:
            w = self.attr_weights.get(col, 1.0)
            d += w * self._dist_attr(col, x[col], y[col])
        return float(d)

    def _compute_distance_matrix(self, X: pd.DataFrame) -> np.ndarray:
        n = len(X)
        D = np.zeros((n, n))
        cols = list(X.columns)
        for i in range(n):
            for j in range(i+1, n):
                d = sum(self.attr_weights[col] * self._dist_attr(col, X.iloc[i][col], X.iloc[j][col]) for col in cols)
                D[i, j] = D[j, i] = d
        return D

    # --- Bombing Core Mechanics ---
    def _build_jnng(self, D_scaled: np.ndarray):
        n = len(D_scaled)
        
        k1_idx = np.argpartition(D_scaled, self.k1 + 1, axis=1)[:, 1:self.k1 + 1] # +1 untuk skip diri sendiri
        k1_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k1_idx[i]:
                k1_adj[i, j] = k1_adj[j, i] = True
                
        # --- k2 (MUTUAL k-NN FIX) ---
        # Ambil k2+1, lalu slice [:, 1:] untuk membuang jarak ke diri sendiri (yang bernilai 0)
        k2_idx = np.argpartition(D_scaled, self.k2 + 1, axis=1)[:, 1:self.k2 + 1] 
        k2_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k2_idx[i]:
                # Cek mutualitas yang sebenarnya (j != i sudah dijamin oleh slice [1:])
                if i in k2_idx[j]: 
                    k2_adj[i, j] = k2_adj[j, i] = True
                
        jnng_adj = k1_adj | k2_adj
        jnng_sparse = csr_matrix(jnng_adj.astype(float) * D_scaled)
        
        # Geodesic distance
        geo_dist = shortest_path(jnng_sparse, method='D', directed=False)
        geo_dist[np.isinf(geo_dist)] = geo_dist[np.isfinite(geo_dist)].max() * 1.5
        
        return jnng_sparse, geo_dist, csr_matrix(jnng_adj.astype(float))

    def _gaussian_kernel_density_old(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        PERBAIKAN MURNI BOMBING: Local Scaling (Zelnik-Perona style).
        Mencegah density flattening dengan membuat bandwidth adaptif terhadap kepadatan lokal.
        """
        n = geo_dist.shape[0]
        k_local = min(7, n - 1)
        
        sigmas = np.partition(geo_dist, k_local, axis=1)[:, k_local]
        sigmas[sigmas == 0] = 1e-12
        
        sigma_matrix = np.outer(sigmas, sigmas)
        rho = np.sum(np.exp(-geo_dist**2 / (2 * sigma_matrix + 1e-12)), axis=1)
        return rho

    def _gaussian_kernel_density(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        Mengikuti Eq. 9 dari Paper Fei et al. (2025).
        Menggunakan varians tetap (dc) untuk menjaga validitas Two-Round Sigmoid.
        """
        # self.dc = 0.15 (atau sesuai parameter default paper)
        rho = np.sum(np.exp(-(geo_dist ** 2) / (2 * (self.dc ** 2))), axis=1)
        return rho

    def _two_round_sigmoid_correction(self, rho_gjnn: np.ndarray) -> np.ndarray:
        rho_max = rho_gjnn.max()
        slope, delta = 10/rho_max, rho_max/2
        rho_tilde = 1 / (1 + np.exp(-slope * (rho_gjnn - delta)))
        mask = rho_tilde < self.T1
        if np.any(mask):
            rho_max_below = rho_gjnn[mask].max()
            s2, d2 = 10/rho_max_below, rho_max_below/2
            rho_tilde[mask] = 1 / (1 + np.exp(-s2 * (rho_tilde[mask] - d2)))
        return rho_tilde

    def _local_neighbor_density(self, D_scaled: np.ndarray, r: float, valid_mask: np.ndarray) -> np.ndarray:
        """Menghitung rho_r (Eq. 16) hanya berdasarkan titik-titik yang valid (belum di-cluster)."""
        n = len(D_scaled)
        rho_r = np.zeros(n)
        for i in range(n):
            if not valid_mask[i]: continue
            # Hitung tetangga dalam radius r dari titik i
            neighbors_in_r = np.sum((D_scaled[i] <= r) & valid_mask)
            rho_r[i] = neighbors_in_r / self.k2
        return rho_r

    def _bfs_generalized_cluster(self, adj_mat: csr_matrix, core_idx: int, P: np.ndarray, visited_mask: np.ndarray) -> Tuple[List[int], List[int]]:
        R_t, B_t = [], []
        queue = [core_idx]
        visited_mask[core_idx] = True
        R_t.append(core_idx)
        adj_list = adj_mat.tolil().rows
        first_boundary = set()
        
        while queue:
            u = queue.pop(0)
            for v in adj_list[u]:
                if visited_mask[v]: continue
                if P[v] >= self.tau:
                    visited_mask[v] = True
                    R_t.append(v)
                    queue.append(v)
                else:
                    first_boundary.add(v)
                    
        second_boundary = set()
        for v in first_boundary:
            for w in adj_list[v]:
                if not visited_mask[w] and w not in first_boundary and w not in R_t:
                    second_boundary.add(w)
        B_t = list(first_boundary | second_boundary)
        return R_t, B_t

    def _degree_of_adjacency(self, adj_mat: csr_matrix, R_t: List[int], B_t: List[int], existing_clusters: dict) -> Tuple[float, int]:
        if not existing_clusters: return 0.0, -1
        C_star = len(R_t) + len(B_t)
        if C_star == 0: return 0.0, -1
        adj_list = adj_mat.tolil().rows
        max_nap, best_s = 0, -1
        for s, (_, B_s) in existing_clusters.items():
            nap = sum(1 for u in R_t for v in adj_list[u] if v in B_s)
            da = nap / C_star
            if da > max_nap: max_nap, best_s = da, s
        return max_nap, best_s

    def _run_bombing_phase(self, X: pd.DataFrame, D_scaled: np.ndarray, geo_dist: np.ndarray, rho_tilde: np.ndarray, jnng_adj: csr_matrix):
        n = len(X)
        labels = np.full(n, -1, dtype=int)
        visited = np.zeros(n, dtype=bool)
        existing_clusters = {}
        t = 0
        self._reachable_indices = {}
        
        # Copy rho_tilde agar bisa dimanipulasi per iterasi
        current_rho_tilde = rho_tilde.copy()
        
        while not np.all(visited):
            cand = ~visited
            if not np.any(cand): break
            
            # 1. Tentukan Core Point dari titik yang BELUM dikunjungi
            core_idx = np.argmax(current_rho_tilde * cand)
            if current_rho_tilde[core_idx] < 1e-6: break

            # 2. Hitung Ulang Local Neighbor Density (rho_r) berdasarkan sisa titik
            # Paper: r adalah jarak ke k-th nearest neighbor dari core point
            unvisited_indices = np.where(cand)[0]
            if len(unvisited_indices) <= self.k2:
                r = np.max(D_scaled[core_idx, unvisited_indices])
            else:
                r = np.partition(D_scaled[core_idx, unvisited_indices], self.k2)[self.k2]
                
            rho_r = self._local_neighbor_density(D_scaled, r, cand)

            # 3. Mixture Density (Eq. 17)
            P = current_rho_tilde + rho_r
            P_min, P_max = P.min(), P.max()
            if P_max > P_min: 
                P = (P - P_min) / (P_max - P_min)

            # 4. BFS dengan TAU TETAP (0.6) - Ini adalah klaim utama paper!
            R_t, B_t = self._bfs_generalized_cluster(jnng_adj, core_idx, P, visited.copy())
            
            # Tandai titik yang di-cluster dan boundary sebagai visited (dihapus dari X)
            idxs = R_t + B_t
            for idx in idxs: 
                visited[idx] = True

            # 5. Hitung Degree of Adjacency (DA)
            da, best_s = self._degree_of_adjacency(jnng_adj, R_t, B_t, existing_clusters)

            # 6. Aturan Penggabungan (Generalized Cluster)
            if da >= self.T2 and best_s != -1:
                labels[idxs] = best_s
            else:
                labels[idxs] = t
                existing_clusters[t] = (R_t, B_t)
                self._reachable_indices[t] = R_t
                t += 1

            # 7. UPDATE DINAMIS: Nol-kan densitas titik yang sudah di-cluster
            # Agar iterasi berikutnya hanya mencari core point dari sisa dataset
            current_rho_tilde[visited] = 0.0
            
            # Opsional: Recalculate Sigmoid Round 1 & 2 untuk sisa titik jika diperlukan
            # (Paper melakukan recalculate rho_max dari sisa titik)
            rho_max_rem = current_rho_tilde.max()
            if rho_max_rem > 1e-6:
                slope, delta = 10 / rho_max_rem, rho_max_rem / 2
                # Terapkan ulang sigmoid ke titik yang belum visited
                current_rho_tilde[~visited] = 1 / (1 + np.exp(-slope * (current_rho_tilde[~visited] - delta)))

        # Final Clean-up (Assign unvisited/boundary ke nearest visited point)
        unvisited = np.where(labels == -1)[0]
        if len(unvisited) > 0:
            visited_pts = np.where(labels != -1)[0]
            if len(visited_pts) > 0:
                for u in unvisited:
                    labels[u] = labels[visited_pts[np.argmin(D_scaled[u, visited_pts])]]
            else:
                labels[:n//2] = 0
                labels[n//2:] = 1

        self.K_auto_ = len(np.unique(labels[labels != -1]))
        return self.K_auto_, labels

    def _compute_initial_centroids(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        """
        [ABLATION VAR-2] Tanpa Density-Excluded Init.
        Inisialisasi sentroid menggunakan SEMUA titik dalam klaster Bombing 
        (termasuk boundary/noise), bukan hanya reachable (core) points.
        """
        centers = []
        unique_labels = np.unique(labels)
        
        for lbl in unique_labels:
            # PERUBAHAN UTAMA: Mengabaikan self._reachable_indices.
            # Langsung ambil semua titik yang memiliki label lbl (termasuk boundary & noise).
            cl = X[labels == lbl]
            
            c = {}
            for col in X.columns:
                if self.attr_types[col] in ('Nominal', 'Ordinal'):
                    c[col] = _mode(cl[col])
                else:
                    c[col] = cl[col].mean()
            centers.append(c)
            
        return pd.DataFrame(centers, columns=X.columns)

    # --- EW-LMD Refinement Helpers ---
    def _update_nominal_ordinal_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] not in ('Nominal', 'Ordinal'): continue
            vals = list(pd.unique(X[col]))
            if len(vals) < 2: continue
            P = {}
            for a in vals:
                for b in vals:
                    if a == b: continue
                    val_sum = sum(((X.iloc[np.where(labels==c)[0]][col]==a).sum()/(len(X[labels==c])+eps)) * 
                                  ((X.iloc[np.where(labels==c)[0]][col]==b).sum()/(len(X[labels==c])+eps)) 
                                  for c in range(k) if len(X[labels==c]) > 0)
                    P[(a,b)] = val_sum
            if not P: continue
            P_vals = list(P.values())
            P_max, P_min = max(P_vals), min(P_vals)
            if abs(P_max-P_min) < eps: continue
            
            phi_raw = {key: (-P_max/self.eta_cat if v==P_max else (1-P_min)/self.eta_cat if v==P_min else 0.0) for key, v in P.items()}
            phi_old = self.nominal_phi[col] if self.attr_types[col]=='Nominal' else self.ordinal_phi[col]
            phi_new = {key: np.clip(self.alpha_reg*phi_old.get(key,0.0) + (1-self.alpha_reg)*phi_raw[key], -1, 1) for key in phi_raw}
            
            if self.attr_types[col]=='Nominal':
                self.nominal_phi[col] = phi_new
                self.nominal_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}
            else:
                self.ordinal_phi[col] = phi_new
                self.ordinal_adj_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}

    def _update_numeric_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        B = self.n_bins_numeric
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] != 'Numeric': continue
            if self.num_min[col] >= self.num_max[col]: continue
            z_full = np.clip((pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy() - self.num_min[col]) / (self.num_max[col] - self.num_min[col]), 0.0, 1.0)
            P_sum = np.zeros(B)
            
            for c in range(k):
                idx = np.where(labels==c)[0]
                m = len(idx)
                if m < 2: continue
                if m > self.max_pairs_per_cluster:
                    idx_sample = self._rng.choice(idx, size=self.max_pairs_per_cluster, replace=False)
                else:
                    idx_sample = idx
                m_s = len(idx_sample)
                if m_s < 2: continue
                
                bin_counts = np.zeros(B, dtype=float)
                for i in range(m_s):
                    for j in range(i + 1, m_s):
                        delta = abs(z_full[idx_sample[i]] - z_full[idx_sample[j]])
                        bidx = int(np.clip(delta * B, 0, B - 1))
                        bin_counts[bidx] += 1.0
                total_pairs = bin_counts.sum()
                if total_pairs <= 0: continue
                probs = bin_counts / (total_pairs + eps)
                P_sum += probs * probs

            if np.all(P_sum <= eps): continue
            P_max, P_min = P_sum.max(), P_sum.min()
            if abs(P_max - P_min) < eps: continue

            phi_raw_vec = np.zeros(B, dtype=float)
            for bidx in range(B):
                val = P_sum[bidx]
                if val == P_max: phi = -P_max / self.eta_num
                elif val == P_min: phi = (1.0 - P_min) / self.eta_num
                else: phi = 0.0
                phi_raw_vec[bidx] = phi

            phi_old_vec = self.numeric_phi[col]
            if phi_old_vec.shape[0] != B: phi_old_vec = np.zeros(B, dtype=float)
            phi_new_vec = np.clip(self.alpha_reg * phi_old_vec + (1.0 - self.alpha_reg) * phi_raw_vec, -1.0, 1.0)
            self.numeric_phi[col] = phi_new_vec
            self.num_bin_psi[col] = phi_new_vec.copy()

    def _assign_refinement(self, X: pd.DataFrame, jnng_adj: csr_matrix) -> np.ndarray:
        """
        REFINEMENT DENGAN KENDALA KONEKTIVITAS jNNG (FIX CELAH #1)
        Titik hanya boleh pindah klaster jika memiliki edge jNNG dengan anggota klaster target.
        """
        n = len(X)
        labels = np.empty(n, dtype=int)
        
        # 1. Pre-compute anggota klaster saat ini untuk lookup O(1)
        cluster_members = {c: set(np.where(self.labels_ == c)[0]) for c in range(self.K_auto_)}
        
        # 2. Konversi adjacency matrix ke list of sets untuk akses tetangga yang cepat
        adj_list = jnng_adj.tolil().rows

        for i in range(n):
            # Hitung jarak EW-LMD ke semua sentroid
            dists = [
                sum(self.attr_weights[col] * self._dist_attr(col, X.iloc[i][col], self.centers_.iloc[c][col]) 
                    for col in X.columns) 
                for c in range(self.K_auto_)
            ]
            best_c = int(np.argmin(dists)) # Klaster terdekat secara geometris
            
            # Cek konektivitas topologi jNNG
            neighbors_i = set(adj_list[i])
            
            # Cari klaster mana saja yang memiliki anggota terhubung dengan titik i
            connected_clusters = [
                c for c, members in cluster_members.items() 
                if neighbors_i.intersection(members)
            ]
            
            # 3. Keputusan Alokasi
            if connected_clusters:
                # Jika ada koneksi, pilih klaster yang TERHUBUNG yang jaraknya paling minimum
                labels[i] = min(connected_clusters, key=lambda c: dists[c])
            else:
                # Fallback: Titik terisolasi (tidak ada edge ke klaster manapun)
                # Gunakan jarak geometris murni agar tidak error
                labels[i] = best_c

        return labels

    def _update_centers_refinement(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        for c in range(self.K_auto_):
            cl = X[labels==c]
            if cl.empty: 
                centers.append(self.centers_.iloc[c].to_dict())
                continue
            c_dict = {col: _mode(cl[col]) if self.attr_types[col] in ('Nominal','Ordinal') else cl[col].mean() for col in X.columns}
            centers.append(c_dict)
        return pd.DataFrame(centers)

    def fit(self, df: pd.DataFrame):
        X = df.drop(columns=["Label"], errors="ignore").copy()
        for col in X.columns:
            if self.attr_types[col] == "Numeric":
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())

        # 1. HYBRID PARAMETER TUNING
        structure = self._compute_structure(X)
        params = hybrid_selection_conservative(X, self.attr_types, structure)
        self.dis_const = params['dist_cons']
        self.eta_cat = self.eta_num = params['eta']
        self.n_bins_numeric = params['n_bins_numeric']  # ← PERBAIKAN
        self.auto_params_log = params.copy()
        if self.verbose: print(f"[AUTONOMOUS] Hybrid-tuned: {self.auto_params_log}")
        # if not hasattr(self, 'dis_const') or self.dis_const < 0.2:
        #     # Estimasi median jarak numerik, lalu set dis_const ~30-50% dari median
        #     # (Opsional: bisa hardcode 0.35 untuk sintetik mixed-type)
        #     self.dis_const = 0.30
        #     if self.verbose: print(f"[AUTONOMOUS] _dist_const-tuned: {self.dis_const}")
        # print(f"[MANUAL] _dist_const: {self.dis_const}, _eta: {self.eta_cat}, _n_bins_numeric: {self.n_bins_numeric}")

        print("[Phase 0] Computing entropy weights & initializing distances...")
        self._compute_entropy_weights(X)
        self._init_distances(X)
        
        print("[Phase 1] Building jNNG with EW-LMD distance...")
        D = self._compute_distance_matrix(X)
        n = len(X)
        median_dist = np.median(D[np.triu_indices(n, k=1)])
        D_scaled = D / (median_dist + 1e-12)
        
        jnng_sparse, geo_dist, jnng_adj = self._build_jnng(D_scaled)
        
        # Menggunakan Local Scaling di sini
        rho_gjnn = self._gaussian_kernel_density(geo_dist)
        rho_tilde = self._two_round_sigmoid_correction(rho_gjnn)
        
        print("[Phase 2] Running Bombing process (Automatic K detection)...")
        K_before, labels_bombing = self._run_bombing_phase(X, D_scaled, geo_dist, rho_tilde, jnng_adj)
        print(f"   -> Detected K = {K_before}")
        
        self.labels_ = labels_bombing
        self.centers_ = self._compute_initial_centroids(X, labels_bombing)
        
        print("[Phase 2.5] Evaluating structural coherence for conditional merging...")
        metrics = compute_merge_metrics(labels_bombing, jnng_adj, rho_tilde)
        trigger = should_merge_parameter_free(metrics, T2=self.T2)
        
        if trigger:
            print("   -> Trigger activated. Initializing Post-Hoc Merger...")
            merger = PostHocMerger(self, X, D_scaled)
            merge_results = merger.smart_merge_stable(min_K=2, max_merge_ratio=1.0)
            labels_refined = merge_results['labels_merged']
            K_after = merge_results['K_final']
            print(f"   -> Merge Result: K {merge_results['K_before']} → {K_after}")
        else:
            print("   -> Structure coherent. Skipping merging phase.")
            labels_refined = labels_bombing
            K_after = K_before
            
        self.labels_ = labels_refined
        self.K_auto_ = K_after
        
        print("[Phase 3] Computing core centroids...")
        if trigger:
            for lbl in np.unique(labels_refined):
                orig_lbls = np.unique(labels_bombing[labels_refined == lbl])
                r_idx = []
                for ol in orig_lbls:
                    r_idx.extend(self._reachable_indices.get(ol, []))
                self._reachable_indices[lbl] = r_idx
                
        self.centers_ = self._compute_initial_centroids(X, labels_refined)
        
        print("[Phase 4-6] EW-LMD Iterative Refinement Loop...")
        prev_labels = self.labels_.copy()
        best_silhouette = -1.0
        best_labels = self.labels_.copy()
        restart_count = 0
        
        for step in range(self.max_iter_refine):
            current_alpha = 0.5 if step < 2 else self.alpha_reg
            orig_alpha = self.alpha_reg
            self.alpha_reg = current_alpha
            
            self._update_nominal_ordinal_phi(X, self.labels_)
            self._update_numeric_phi(X, self.labels_)
            self.alpha_reg = orig_alpha
            
            self.labels_ = self._assign_refinement(X, jnng_adj)
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
            if step >= 2:
                ari_internal = adjusted_rand_score(prev_labels, self.labels_)
                
                # === PERBAIKAN: TRAP DETECTION via Silhouette ===
                sample_idx = np.random.choice(len(X), min(1000, len(X)), replace=False) if len(X) > 1000 else np.arange(len(X))
                D_sample = np.zeros((len(sample_idx), len(sample_idx)))
                for i, idx_i in enumerate(sample_idx):
                    for j, idx_j in enumerate(sample_idx):
                        # Menggunakan fungsi distance yang sudah ada
                        D_sample[i,j] = self.distance(X.iloc[idx_i], X.iloc[idx_j])
                
                try:
                    current_sil = silhouette_score(D_sample, self.labels_[sample_idx], metric='precomputed')
                except ValueError:
                    current_sil = 0.0
                    
                if current_sil > best_silhouette:
                    best_silhouette = current_sil
                    best_labels = self.labels_.copy()
                    
                print(f"   Iter {step+1}: Internal ARI = {ari_internal:.4f} | Silhouette = {current_sil:.4f}")
                
                if ari_internal > self.conv_threshold:
                    if current_sil < 0.1 and restart_count < 2:
                        print(f"   -> [Trap Detected] Silhouette < 0.1. Perturbing labels to escape local optimum...")
                        noise_mask = np.random.rand(len(X)) < 0.2
                        self.labels_[noise_mask] = self._rng.integers(0, self.K_auto_, size=np.sum(noise_mask))
                        self.centers_ = self._update_centers_refinement(X, self.labels_)
                        restart_count += 1
                        prev_labels = self.labels_.copy()
                        continue
                    else:
                        print("   -> Convergence reached.")
                        break
                        
            prev_labels = self.labels_.copy()

        if best_silhouette > current_sil:
            if self.verbose: print(f"   -> [Recovery] Restoring best labels from Iter with Silhouette = {best_silhouette:.4f}")
            self.labels_ = best_labels
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
        return self

### V3 Bombing + EW-LMD Algorithm without Post-Hoc Merging

In [ ]:
class EWLMDBombingHybrid_V3:
    def __init__(self, attr_types: dict, ordinal_orders: dict = None,
                 max_iter_refine: int = 20, random_state: int = 42, verbose: bool = True):
        
        self.attr_types = attr_types.copy()
        self.ordinal_orders = (ordinal_orders or {}).copy()
        self.dis_const = 0.3
        self.eta_cat = 20.0
        self.eta_num = 20.0 
        self.alpha_reg = 0.8
        self.tau_weight = 1e-3
        self.n_bins_numeric = 10
        self.max_pairs_per_cluster = 200
        self.max_iter_refine = max_iter_refine
        self.conv_threshold = 0.98
        self._rng = np.random.default_rng(random_state)
        self.verbose = verbose
        
        # Bombing Parameters
        self.k1, self.k2 = 2, 10
        self.dc, self.tau, self.T1, self.T2 = 0.15, 0.6, 0.9, 0.2
        
        self.attr_weights = {}
        self.nominal_dist, self.ordinal_adj_dist = {}, {}
        self.num_min, self.num_max = {}, {}
        self.num_bin_psi = {}
        self.nominal_phi, self.ordinal_phi, self.numeric_phi = {}, {}, {}
        
        self.labels_ = None
        self.centers_ = None
        self.K_auto_ = None
        self._reachable_indices = {} 

    def _compute_structure(self, X: pd.DataFrame) -> dict:
        counts = {"Numeric": 0, "Ordinal": 0, "Nominal": 0}
        for t in self.attr_types.values():
            counts["Nominal" if t not in ["Numeric", "Ordinal"] else t] += 1
        max_nom = max([X[c].nunique() for c, t in self.attr_types.items() if t == "Nominal"] or [0])
        return {
            'n_rows': len(X), 'n_cols': len(X.columns),
            'n_numeric': counts['Numeric'], 'n_ordinal': counts['Ordinal'], 'n_nominal': counts['Nominal'],
            'max_nominal_card': max_nom,
            'is_wide': len(X.columns) > 20,
            'is_numeric_heavy': counts['Numeric'] > counts['Nominal'],
            'is_nominal_heavy': counts['Nominal'] > counts['Numeric']
        }
    # --- Distance & Weight Helpers ---
    def _compute_entropy_weights(self, X: pd.DataFrame):
        eps = 1e-12
        raw_weights: Dict[str, float] = {}
        
        for col in X.columns:
            t = self.attr_types[col]
            if t in ("Nominal", "Ordinal"):
                probs = X[col].value_counts(normalize=True)
                H = -np.sum(probs * np.log(probs + eps))
                H_max = np.log(len(probs)) if len(probs) > 1 else 1.0
                informativeness = 1.0 - (H / (H_max + eps))
            elif t == "Numeric":
                vals = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy()
                vmin, vmax = vals.min(), vals.max()
                if vmax > vmin:
                    z = (vals - vmin) / (vmax - vmin)
                    n = len(z)
                    max_pairs = min(self.max_pairs_per_cluster, n * (n - 1) // 2)
                    if max_pairs > 0:
                        idx_i = self._rng.integers(0, n, size=max_pairs)
                        idx_j = self._rng.integers(0, n, size=max_pairs)
                        delta = np.abs(z[idx_i] - z[idx_j])
                        hist, _ = np.histogram(delta, bins=self.n_bins_numeric, range=(0.0, 1.0))
                        probs = hist.astype(float) / (hist.sum() + eps)
                        H = -np.sum(probs * np.log(probs + eps))
                        H_max = np.log(self.n_bins_numeric)
                        informativeness = 1.0 - (H / (H_max + eps))
                    else: informativeness = 0.0
                else: informativeness = 0.0
            else: informativeness = 0.0
            raw_weights[col] = max(informativeness, self.tau_weight)
        
        type_groups = {"Nominal": [], "Ordinal": [], "Numeric": []}
        for col in X.columns:
            t = self.attr_types[col]
            if t in type_groups: type_groups[t].append(col)
            
        n_active_types = len([t for t, cols in type_groups.items() if cols])
        weight_per_type = 1.0 / n_active_types if n_active_types > 0 else 1.0
        
        self.attr_weights = {}
        for t, cols in type_groups.items():
            if not cols: continue
            total_raw = sum(raw_weights[col] for col in cols)
            for col in cols:
                self.attr_weights[col] = weight_per_type * (raw_weights[col] / (total_raw + eps))

    def _init_distances(self, X: pd.DataFrame):
        for col in X.columns:
            t = self.attr_types[col]
            if t == "Nominal":
                vals = list(pd.unique(X[col]))
                self.nominal_dist[col] = {(a,b): self.dis_const for a in vals for b in vals if a!=b}
                self.nominal_phi[col] = {(a,b): 0.0 for a in vals for b in vals if a!=b}
            elif t == "Ordinal":
                order = list(self.ordinal_orders[col])
                self.ordinal_adj_dist[col] = {order[i]: self.dis_const for i in range(len(order)-1)}
                self.ordinal_phi[col] = {order[i]: 0.0 for i in range(len(order)-1)}
            elif t == "Numeric":
                v = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())
                self.num_min[col], self.num_max[col] = float(v.min()), float(v.max())
                self.num_bin_psi[col] = np.zeros(self.n_bins_numeric, dtype=float)
                self.numeric_phi[col] = np.zeros(self.n_bins_numeric, dtype=float)

    def _dist_attr(self, col: str, a: Any, b: Any) -> float:
        t = self.attr_types[col]
        if t == "Nominal":
            if a == b: return 0.0
            mp = self.nominal_dist[col]
            return mp.get((a,b), mp.get((b,a), self.dis_const))
        elif t == "Ordinal":
            if a == b: return 0.0
            order = list(self.ordinal_orders[col])
            try: ia, ib = order.index(a), order.index(b)
            except ValueError: return 0.0
            if ia > ib: ia, ib = ib, ia
            return sum(self.ordinal_adj_dist[col].get(order[i], self.dis_const) for i in range(ia, ib))
        elif t == "Numeric":
            vmin, vmax = self.num_min[col], self.num_max[col]
            if vmax <= vmin: return 0.0
            try: va, vb = float(a), float(b)
            except Exception: va, vb = (vmin+vmax)/2.0, (vmin+vmax)/2.0
            
            z_a = np.clip((va - vmin) / (vmax - vmin), 0.0, 1.0)
            z_b = np.clip((vb - vmin) / (vmax - vmin), 0.0, 1.0)
            
            # PERBAIKAN MURNI BOMBING: Absolute Difference (BUKAN Squared)
            # Ini menjaga konektivitas manifold jNNG agar tidak terputus (sparse)
            delta = abs(z_a - z_b)
            
            psi_vec = self.num_bin_psi[col]
            if psi_vec is None or len(psi_vec) == 0: return delta
            bidx = int(np.clip(delta * self.n_bins_numeric, 0, self.n_bins_numeric - 1))
            scale = max(0.1, 1.0 + psi_vec[bidx])
            return delta * scale
        return 0.0

    def distance(self, x: pd.Series, y: pd.Series) -> float:
        d = 0.0
        for col in y.index:
            w = self.attr_weights.get(col, 1.0)
            d += w * self._dist_attr(col, x[col], y[col])
        return float(d)

    def _compute_distance_matrix(self, X: pd.DataFrame) -> np.ndarray:
        n = len(X)
        D = np.zeros((n, n))
        cols = list(X.columns)
        for i in range(n):
            for j in range(i+1, n):
                d = sum(self.attr_weights[col] * self._dist_attr(col, X.iloc[i][col], X.iloc[j][col]) for col in cols)
                D[i, j] = D[j, i] = d
        return D

    # --- Bombing Core Mechanics ---
    def _build_jnng(self, D_scaled: np.ndarray):
        n = len(D_scaled)
        
        k1_idx = np.argpartition(D_scaled, self.k1 + 1, axis=1)[:, 1:self.k1 + 1] # +1 untuk skip diri sendiri
        k1_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k1_idx[i]:
                k1_adj[i, j] = k1_adj[j, i] = True
                
        # --- k2 (MUTUAL k-NN FIX) ---
        # Ambil k2+1, lalu slice [:, 1:] untuk membuang jarak ke diri sendiri (yang bernilai 0)
        k2_idx = np.argpartition(D_scaled, self.k2 + 1, axis=1)[:, 1:self.k2 + 1] 
        k2_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k2_idx[i]:
                # Cek mutualitas yang sebenarnya (j != i sudah dijamin oleh slice [1:])
                if i in k2_idx[j]: 
                    k2_adj[i, j] = k2_adj[j, i] = True
                
        jnng_adj = k1_adj | k2_adj
        jnng_sparse = csr_matrix(jnng_adj.astype(float) * D_scaled)
        
        # Geodesic distance
        geo_dist = shortest_path(jnng_sparse, method='D', directed=False)
        geo_dist[np.isinf(geo_dist)] = geo_dist[np.isfinite(geo_dist)].max() * 1.5
        
        return jnng_sparse, geo_dist, csr_matrix(jnng_adj.astype(float))

    def _gaussian_kernel_density_old(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        PERBAIKAN MURNI BOMBING: Local Scaling (Zelnik-Perona style).
        Mencegah density flattening dengan membuat bandwidth adaptif terhadap kepadatan lokal.
        """
        n = geo_dist.shape[0]
        k_local = min(7, n - 1)
        
        sigmas = np.partition(geo_dist, k_local, axis=1)[:, k_local]
        sigmas[sigmas == 0] = 1e-12
        
        sigma_matrix = np.outer(sigmas, sigmas)
        rho = np.sum(np.exp(-geo_dist**2 / (2 * sigma_matrix + 1e-12)), axis=1)
        return rho

    def _gaussian_kernel_density(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        Mengikuti Eq. 9 dari Paper Fei et al. (2025).
        Menggunakan varians tetap (dc) untuk menjaga validitas Two-Round Sigmoid.
        """
        # self.dc = 0.15 (atau sesuai parameter default paper)
        rho = np.sum(np.exp(-(geo_dist ** 2) / (2 * (self.dc ** 2))), axis=1)
        return rho

    def _two_round_sigmoid_correction(self, rho_gjnn: np.ndarray) -> np.ndarray:
        rho_max = rho_gjnn.max()
        slope, delta = 10/rho_max, rho_max/2
        rho_tilde = 1 / (1 + np.exp(-slope * (rho_gjnn - delta)))
        mask = rho_tilde < self.T1
        if np.any(mask):
            rho_max_below = rho_gjnn[mask].max()
            s2, d2 = 10/rho_max_below, rho_max_below/2
            rho_tilde[mask] = 1 / (1 + np.exp(-s2 * (rho_tilde[mask] - d2)))
        return rho_tilde

    def _local_neighbor_density(self, D_scaled: np.ndarray, r: float, valid_mask: np.ndarray) -> np.ndarray:
        """Menghitung rho_r (Eq. 16) hanya berdasarkan titik-titik yang valid (belum di-cluster)."""
        n = len(D_scaled)
        rho_r = np.zeros(n)
        for i in range(n):
            if not valid_mask[i]: continue
            # Hitung tetangga dalam radius r dari titik i
            neighbors_in_r = np.sum((D_scaled[i] <= r) & valid_mask)
            rho_r[i] = neighbors_in_r / self.k2
        return rho_r

    def _bfs_generalized_cluster(self, adj_mat: csr_matrix, core_idx: int, P: np.ndarray, visited_mask: np.ndarray) -> Tuple[List[int], List[int]]:
        R_t, B_t = [], []
        queue = [core_idx]
        visited_mask[core_idx] = True
        R_t.append(core_idx)
        adj_list = adj_mat.tolil().rows
        first_boundary = set()
        
        while queue:
            u = queue.pop(0)
            for v in adj_list[u]:
                if visited_mask[v]: continue
                if P[v] >= self.tau:
                    visited_mask[v] = True
                    R_t.append(v)
                    queue.append(v)
                else:
                    first_boundary.add(v)
                    
        second_boundary = set()
        for v in first_boundary:
            for w in adj_list[v]:
                if not visited_mask[w] and w not in first_boundary and w not in R_t:
                    second_boundary.add(w)
        B_t = list(first_boundary | second_boundary)
        return R_t, B_t

    def _degree_of_adjacency(self, adj_mat: csr_matrix, R_t: List[int], B_t: List[int], existing_clusters: dict) -> Tuple[float, int]:
        if not existing_clusters: return 0.0, -1
        C_star = len(R_t) + len(B_t)
        if C_star == 0: return 0.0, -1
        adj_list = adj_mat.tolil().rows
        max_nap, best_s = 0, -1
        for s, (_, B_s) in existing_clusters.items():
            nap = sum(1 for u in R_t for v in adj_list[u] if v in B_s)
            da = nap / C_star
            if da > max_nap: max_nap, best_s = da, s
        return max_nap, best_s

    def _run_bombing_phase(self, X: pd.DataFrame, D_scaled: np.ndarray, geo_dist: np.ndarray, rho_tilde: np.ndarray, jnng_adj: csr_matrix):
        n = len(X)
        labels = np.full(n, -1, dtype=int)
        visited = np.zeros(n, dtype=bool)
        existing_clusters = {}
        t = 0
        self._reachable_indices = {}
        
        # Copy rho_tilde agar bisa dimanipulasi per iterasi
        current_rho_tilde = rho_tilde.copy()
        
        while not np.all(visited):
            cand = ~visited
            if not np.any(cand): break
            
            # 1. Tentukan Core Point dari titik yang BELUM dikunjungi
            core_idx = np.argmax(current_rho_tilde * cand)
            if current_rho_tilde[core_idx] < 1e-6: break

            # 2. Hitung Ulang Local Neighbor Density (rho_r) berdasarkan sisa titik
            # Paper: r adalah jarak ke k-th nearest neighbor dari core point
            unvisited_indices = np.where(cand)[0]
            if len(unvisited_indices) <= self.k2:
                r = np.max(D_scaled[core_idx, unvisited_indices])
            else:
                r = np.partition(D_scaled[core_idx, unvisited_indices], self.k2)[self.k2]
                
            rho_r = self._local_neighbor_density(D_scaled, r, cand)

            # 3. Mixture Density (Eq. 17)
            P = current_rho_tilde + rho_r
            P_min, P_max = P.min(), P.max()
            if P_max > P_min: 
                P = (P - P_min) / (P_max - P_min)

            # 4. BFS dengan TAU TETAP (0.6) - Ini adalah klaim utama paper!
            R_t, B_t = self._bfs_generalized_cluster(jnng_adj, core_idx, P, visited.copy())
            
            # Tandai titik yang di-cluster dan boundary sebagai visited (dihapus dari X)
            idxs = R_t + B_t
            for idx in idxs: 
                visited[idx] = True

            # 5. Hitung Degree of Adjacency (DA)
            da, best_s = self._degree_of_adjacency(jnng_adj, R_t, B_t, existing_clusters)

            # 6. Aturan Penggabungan (Generalized Cluster)
            if da >= self.T2 and best_s != -1:
                labels[idxs] = best_s
            else:
                labels[idxs] = t
                existing_clusters[t] = (R_t, B_t)
                self._reachable_indices[t] = R_t
                t += 1

            # 7. UPDATE DINAMIS: Nol-kan densitas titik yang sudah di-cluster
            # Agar iterasi berikutnya hanya mencari core point dari sisa dataset
            current_rho_tilde[visited] = 0.0
            
            # Opsional: Recalculate Sigmoid Round 1 & 2 untuk sisa titik jika diperlukan
            # (Paper melakukan recalculate rho_max dari sisa titik)
            rho_max_rem = current_rho_tilde.max()
            if rho_max_rem > 1e-6:
                slope, delta = 10 / rho_max_rem, rho_max_rem / 2
                # Terapkan ulang sigmoid ke titik yang belum visited
                current_rho_tilde[~visited] = 1 / (1 + np.exp(-slope * (current_rho_tilde[~visited] - delta)))

        # Final Clean-up (Assign unvisited/boundary ke nearest visited point)
        unvisited = np.where(labels == -1)[0]
        if len(unvisited) > 0:
            visited_pts = np.where(labels != -1)[0]
            if len(visited_pts) > 0:
                for u in unvisited:
                    labels[u] = labels[visited_pts[np.argmin(D_scaled[u, visited_pts])]]
            else:
                labels[:n//2] = 0
                labels[n//2:] = 1

        self.K_auto_ = len(np.unique(labels[labels != -1]))
        return self.K_auto_, labels

    def _compute_initial_centroids(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        unique_labels = np.unique(labels)
        reachable_dict = getattr(self, '_reachable_indices', {})
        n_points = len(X)
        
        for lbl in unique_labels:
            r_indices = reachable_dict.get(lbl, [])
            if len(r_indices) > 0:
                core_mask = np.zeros(n_points, dtype=bool)
                core_mask[r_indices] = True
                cl = X[core_mask] # Density-Excluded Initialization
            else:
                cl = X[labels == lbl]
                
            c = {}
            for col in X.columns:
                if self.attr_types[col] in ('Nominal', 'Ordinal'):
                    c[col] = _mode(cl[col])
                else:
                    c[col] = cl[col].mean()
            centers.append(c)
        return pd.DataFrame(centers, columns=X.columns)

    # --- EW-LMD Refinement Helpers ---
    def _update_nominal_ordinal_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] not in ('Nominal', 'Ordinal'): continue
            vals = list(pd.unique(X[col]))
            if len(vals) < 2: continue
            P = {}
            for a in vals:
                for b in vals:
                    if a == b: continue
                    val_sum = sum(((X.iloc[np.where(labels==c)[0]][col]==a).sum()/(len(X[labels==c])+eps)) * 
                                  ((X.iloc[np.where(labels==c)[0]][col]==b).sum()/(len(X[labels==c])+eps)) 
                                  for c in range(k) if len(X[labels==c]) > 0)
                    P[(a,b)] = val_sum
            if not P: continue
            P_vals = list(P.values())
            P_max, P_min = max(P_vals), min(P_vals)
            if abs(P_max-P_min) < eps: continue
            
            phi_raw = {key: (-P_max/self.eta_cat if v==P_max else (1-P_min)/self.eta_cat if v==P_min else 0.0) for key, v in P.items()}
            phi_old = self.nominal_phi[col] if self.attr_types[col]=='Nominal' else self.ordinal_phi[col]
            phi_new = {key: np.clip(self.alpha_reg*phi_old.get(key,0.0) + (1-self.alpha_reg)*phi_raw[key], -1, 1) for key in phi_raw}
            
            if self.attr_types[col]=='Nominal':
                self.nominal_phi[col] = phi_new
                self.nominal_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}
            else:
                self.ordinal_phi[col] = phi_new
                self.ordinal_adj_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}

    def _update_numeric_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        B = self.n_bins_numeric
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] != 'Numeric': continue
            if self.num_min[col] >= self.num_max[col]: continue
            z_full = np.clip((pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy() - self.num_min[col]) / (self.num_max[col] - self.num_min[col]), 0.0, 1.0)
            P_sum = np.zeros(B)
            
            for c in range(k):
                idx = np.where(labels==c)[0]
                m = len(idx)
                if m < 2: continue
                if m > self.max_pairs_per_cluster:
                    idx_sample = self._rng.choice(idx, size=self.max_pairs_per_cluster, replace=False)
                else:
                    idx_sample = idx
                m_s = len(idx_sample)
                if m_s < 2: continue
                
                bin_counts = np.zeros(B, dtype=float)
                for i in range(m_s):
                    for j in range(i + 1, m_s):
                        delta = abs(z_full[idx_sample[i]] - z_full[idx_sample[j]])
                        bidx = int(np.clip(delta * B, 0, B - 1))
                        bin_counts[bidx] += 1.0
                total_pairs = bin_counts.sum()
                if total_pairs <= 0: continue
                probs = bin_counts / (total_pairs + eps)
                P_sum += probs * probs

            if np.all(P_sum <= eps): continue
            P_max, P_min = P_sum.max(), P_sum.min()
            if abs(P_max - P_min) < eps: continue

            phi_raw_vec = np.zeros(B, dtype=float)
            for bidx in range(B):
                val = P_sum[bidx]
                if val == P_max: phi = -P_max / self.eta_num
                elif val == P_min: phi = (1.0 - P_min) / self.eta_num
                else: phi = 0.0
                phi_raw_vec[bidx] = phi

            phi_old_vec = self.numeric_phi[col]
            if phi_old_vec.shape[0] != B: phi_old_vec = np.zeros(B, dtype=float)
            phi_new_vec = np.clip(self.alpha_reg * phi_old_vec + (1.0 - self.alpha_reg) * phi_raw_vec, -1.0, 1.0)
            self.numeric_phi[col] = phi_new_vec
            self.num_bin_psi[col] = phi_new_vec.copy()

    def _assign_refinement(self, X: pd.DataFrame, jnng_adj: csr_matrix) -> np.ndarray:
        """
        REFINEMENT DENGAN KENDALA KONEKTIVITAS jNNG (FIX CELAH #1)
        Titik hanya boleh pindah klaster jika memiliki edge jNNG dengan anggota klaster target.
        """
        n = len(X)
        labels = np.empty(n, dtype=int)
        
        # 1. Pre-compute anggota klaster saat ini untuk lookup O(1)
        cluster_members = {c: set(np.where(self.labels_ == c)[0]) for c in range(self.K_auto_)}
        
        # 2. Konversi adjacency matrix ke list of sets untuk akses tetangga yang cepat
        adj_list = jnng_adj.tolil().rows

        for i in range(n):
            # Hitung jarak EW-LMD ke semua sentroid
            dists = [
                sum(self.attr_weights[col] * self._dist_attr(col, X.iloc[i][col], self.centers_.iloc[c][col]) 
                    for col in X.columns) 
                for c in range(self.K_auto_)
            ]
            best_c = int(np.argmin(dists)) # Klaster terdekat secara geometris
            
            # Cek konektivitas topologi jNNG
            neighbors_i = set(adj_list[i])
            
            # Cari klaster mana saja yang memiliki anggota terhubung dengan titik i
            connected_clusters = [
                c for c, members in cluster_members.items() 
                if neighbors_i.intersection(members)
            ]
            
            # 3. Keputusan Alokasi
            if connected_clusters:
                # Jika ada koneksi, pilih klaster yang TERHUBUNG yang jaraknya paling minimum
                labels[i] = min(connected_clusters, key=lambda c: dists[c])
            else:
                # Fallback: Titik terisolasi (tidak ada edge ke klaster manapun)
                # Gunakan jarak geometris murni agar tidak error
                labels[i] = best_c

        return labels

    def _update_centers_refinement(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        for c in range(self.K_auto_):
            cl = X[labels==c]
            if cl.empty: 
                centers.append(self.centers_.iloc[c].to_dict())
                continue
            c_dict = {col: _mode(cl[col]) if self.attr_types[col] in ('Nominal','Ordinal') else cl[col].mean() for col in X.columns}
            centers.append(c_dict)
        return pd.DataFrame(centers)

    def fit(self, df: pd.DataFrame):
        X = df.drop(columns=["Label"], errors="ignore").copy()
        for col in X.columns:
            if self.attr_types[col] == "Numeric":
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())

        # 1. HYBRID PARAMETER TUNING
        structure = self._compute_structure(X)
        params = hybrid_selection_conservative(X, self.attr_types, structure)
        self.dis_const = params['dist_cons']
        self.eta_cat = self.eta_num = params['eta']
        self.n_bins_numeric = params['n_bins_numeric']  # ← PERBAIKAN
        self.auto_params_log = params.copy()
        if self.verbose: print(f"[AUTONOMOUS] Hybrid-tuned: {self.auto_params_log}")
        # if not hasattr(self, 'dis_const') or self.dis_const < 0.2:
        #     # Estimasi median jarak numerik, lalu set dis_const ~30-50% dari median
        #     # (Opsional: bisa hardcode 0.35 untuk sintetik mixed-type)
        #     self.dis_const = 0.30
        #     if self.verbose: print(f"[AUTONOMOUS] _dist_const-tuned: {self.dis_const}")
        # print(f"[MANUAL] _dist_const: {self.dis_const}, _eta: {self.eta_cat}, _n_bins_numeric: {self.n_bins_numeric}")

        print("[Phase 0] Computing entropy weights & initializing distances...")
        self._compute_entropy_weights(X)
        self._init_distances(X)
        
        print("[Phase 1] Building jNNG with EW-LMD distance...")
        D = self._compute_distance_matrix(X)
        n = len(X)
        median_dist = np.median(D[np.triu_indices(n, k=1)])
        D_scaled = D / (median_dist + 1e-12)
        
        jnng_sparse, geo_dist, jnng_adj = self._build_jnng(D_scaled)
        
        # Menggunakan Local Scaling di sini
        rho_gjnn = self._gaussian_kernel_density(geo_dist)
        rho_tilde = self._two_round_sigmoid_correction(rho_gjnn)
        
        print("[Phase 2] Running Bombing process (Automatic K detection)...")
        K_before, labels_bombing = self._run_bombing_phase(X, D_scaled, geo_dist, rho_tilde, jnng_adj)
        print(f"   -> Detected K = {K_before}")
        
        self.labels_ = labels_bombing
        self.centers_ = self._compute_initial_centroids(X, labels_bombing)
        self.K_auto_ = K_before
        
        # print("[Phase 3] Computing density-excluded core centroids...")
        # self.centers_ = self._compute_initial_centroids(X, labels_refined)
        
        print("[Phase 4-6] EW-LMD Iterative Refinement Loop...")
        prev_labels = self.labels_.copy()
        best_silhouette = -1.0
        best_labels = self.labels_.copy()
        restart_count = 0
        
        for step in range(self.max_iter_refine):
            current_alpha = 0.5 if step < 2 else self.alpha_reg
            orig_alpha = self.alpha_reg
            self.alpha_reg = current_alpha
            
            self._update_nominal_ordinal_phi(X, self.labels_)
            self._update_numeric_phi(X, self.labels_)
            self.alpha_reg = orig_alpha
            
            self.labels_ = self._assign_refinement(X, jnng_adj)
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
            if step >= 2:
                ari_internal = adjusted_rand_score(prev_labels, self.labels_)
                
                # === PERBAIKAN: TRAP DETECTION via Silhouette ===
                sample_idx = np.random.choice(len(X), min(1000, len(X)), replace=False) if len(X) > 1000 else np.arange(len(X))
                D_sample = np.zeros((len(sample_idx), len(sample_idx)))
                for i, idx_i in enumerate(sample_idx):
                    for j, idx_j in enumerate(sample_idx):
                        # Menggunakan fungsi distance yang sudah ada
                        D_sample[i,j] = self.distance(X.iloc[idx_i], X.iloc[idx_j])
                
                try:
                    current_sil = silhouette_score(D_sample, self.labels_[sample_idx], metric='precomputed')
                except ValueError:
                    current_sil = 0.0
                    
                if current_sil > best_silhouette:
                    best_silhouette = current_sil
                    best_labels = self.labels_.copy()
                    
                print(f"   Iter {step+1}: Internal ARI = {ari_internal:.4f} | Silhouette = {current_sil:.4f}")
                
                if ari_internal > self.conv_threshold:
                    if current_sil < 0.1 and restart_count < 2:
                        print(f"   -> [Trap Detected] Silhouette < 0.1. Perturbing labels to escape local optimum...")
                        noise_mask = np.random.rand(len(X)) < 0.2
                        self.labels_[noise_mask] = self._rng.integers(0, self.K_auto_, size=np.sum(noise_mask))
                        self.centers_ = self._update_centers_refinement(X, self.labels_)
                        restart_count += 1
                        prev_labels = self.labels_.copy()
                        continue
                    else:
                        print("   -> Convergence reached.")
                        break
                        
            prev_labels = self.labels_.copy()

        if best_silhouette > current_sil:
            if self.verbose: print(f"   -> [Recovery] Restoring best labels from Iter with Silhouette = {best_silhouette:.4f}")
            self.labels_ = best_labels
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
        return self

### V4 Bombing + EW-LMD Algorithm without Refinement 

In [ ]:
class EWLMDBombingHybrid_V4:
    def __init__(self, attr_types: dict, ordinal_orders: dict = None,
                 max_iter_refine: int = 20, random_state: int = 42, verbose: bool = True):
        
        self.attr_types = attr_types.copy()
        self.ordinal_orders = (ordinal_orders or {}).copy()
        self.dis_const = 0.3
        self.eta_cat = 20.0
        self.eta_num = 20.0 
        self.alpha_reg = 0.8
        self.tau_weight = 1e-3
        self.n_bins_numeric = 10
        self.max_pairs_per_cluster = 200
        self.max_iter_refine = max_iter_refine
        self.conv_threshold = 0.98
        self._rng = np.random.default_rng(random_state)
        self.verbose = verbose
        
        # Bombing Parameters
        self.k1, self.k2 = 2, 10
        self.dc, self.tau, self.T1, self.T2 = 0.15, 0.6, 0.9, 0.2
        
        self.attr_weights = {}
        self.nominal_dist, self.ordinal_adj_dist = {}, {}
        self.num_min, self.num_max = {}, {}
        self.num_bin_psi = {}
        self.nominal_phi, self.ordinal_phi, self.numeric_phi = {}, {}, {}
        
        self.labels_ = None
        self.centers_ = None
        self.K_auto_ = None
        self._reachable_indices = {} 

    def _compute_structure(self, X: pd.DataFrame) -> dict:
        counts = {"Numeric": 0, "Ordinal": 0, "Nominal": 0}
        for t in self.attr_types.values():
            counts["Nominal" if t not in ["Numeric", "Ordinal"] else t] += 1
        max_nom = max([X[c].nunique() for c, t in self.attr_types.items() if t == "Nominal"] or [0])
        return {
            'n_rows': len(X), 'n_cols': len(X.columns),
            'n_numeric': counts['Numeric'], 'n_ordinal': counts['Ordinal'], 'n_nominal': counts['Nominal'],
            'max_nominal_card': max_nom,
            'is_wide': len(X.columns) > 20,
            'is_numeric_heavy': counts['Numeric'] > counts['Nominal'],
            'is_nominal_heavy': counts['Nominal'] > counts['Numeric']
        }
    # --- Distance & Weight Helpers ---
    def _compute_entropy_weights(self, X: pd.DataFrame):
        eps = 1e-12
        raw_weights: Dict[str, float] = {}
        
        for col in X.columns:
            t = self.attr_types[col]
            if t in ("Nominal", "Ordinal"):
                probs = X[col].value_counts(normalize=True)
                H = -np.sum(probs * np.log(probs + eps))
                H_max = np.log(len(probs)) if len(probs) > 1 else 1.0
                informativeness = 1.0 - (H / (H_max + eps))
            elif t == "Numeric":
                vals = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy()
                vmin, vmax = vals.min(), vals.max()
                if vmax > vmin:
                    z = (vals - vmin) / (vmax - vmin)
                    n = len(z)
                    max_pairs = min(self.max_pairs_per_cluster, n * (n - 1) // 2)
                    if max_pairs > 0:
                        idx_i = self._rng.integers(0, n, size=max_pairs)
                        idx_j = self._rng.integers(0, n, size=max_pairs)
                        delta = np.abs(z[idx_i] - z[idx_j])
                        hist, _ = np.histogram(delta, bins=self.n_bins_numeric, range=(0.0, 1.0))
                        probs = hist.astype(float) / (hist.sum() + eps)
                        H = -np.sum(probs * np.log(probs + eps))
                        H_max = np.log(self.n_bins_numeric)
                        informativeness = 1.0 - (H / (H_max + eps))
                    else: informativeness = 0.0
                else: informativeness = 0.0
            else: informativeness = 0.0
            raw_weights[col] = max(informativeness, self.tau_weight)
        
        type_groups = {"Nominal": [], "Ordinal": [], "Numeric": []}
        for col in X.columns:
            t = self.attr_types[col]
            if t in type_groups: type_groups[t].append(col)
            
        n_active_types = len([t for t, cols in type_groups.items() if cols])
        weight_per_type = 1.0 / n_active_types if n_active_types > 0 else 1.0
        
        self.attr_weights = {}
        for t, cols in type_groups.items():
            if not cols: continue
            total_raw = sum(raw_weights[col] for col in cols)
            for col in cols:
                self.attr_weights[col] = weight_per_type * (raw_weights[col] / (total_raw + eps))

    def _init_distances(self, X: pd.DataFrame):
        for col in X.columns:
            t = self.attr_types[col]
            if t == "Nominal":
                vals = list(pd.unique(X[col]))
                self.nominal_dist[col] = {(a,b): self.dis_const for a in vals for b in vals if a!=b}
                self.nominal_phi[col] = {(a,b): 0.0 for a in vals for b in vals if a!=b}
            elif t == "Ordinal":
                order = list(self.ordinal_orders[col])
                self.ordinal_adj_dist[col] = {order[i]: self.dis_const for i in range(len(order)-1)}
                self.ordinal_phi[col] = {order[i]: 0.0 for i in range(len(order)-1)}
            elif t == "Numeric":
                v = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())
                self.num_min[col], self.num_max[col] = float(v.min()), float(v.max())
                self.num_bin_psi[col] = np.zeros(self.n_bins_numeric, dtype=float)
                self.numeric_phi[col] = np.zeros(self.n_bins_numeric, dtype=float)

    def _dist_attr(self, col: str, a: Any, b: Any) -> float:
        t = self.attr_types[col]
        if t == "Nominal":
            if a == b: return 0.0
            mp = self.nominal_dist[col]
            return mp.get((a,b), mp.get((b,a), self.dis_const))
        elif t == "Ordinal":
            if a == b: return 0.0
            order = list(self.ordinal_orders[col])
            try: ia, ib = order.index(a), order.index(b)
            except ValueError: return 0.0
            if ia > ib: ia, ib = ib, ia
            return sum(self.ordinal_adj_dist[col].get(order[i], self.dis_const) for i in range(ia, ib))
        elif t == "Numeric":
            vmin, vmax = self.num_min[col], self.num_max[col]
            if vmax <= vmin: return 0.0
            try: va, vb = float(a), float(b)
            except Exception: va, vb = (vmin+vmax)/2.0, (vmin+vmax)/2.0
            
            z_a = np.clip((va - vmin) / (vmax - vmin), 0.0, 1.0)
            z_b = np.clip((vb - vmin) / (vmax - vmin), 0.0, 1.0)
            
            # PERBAIKAN MURNI BOMBING: Absolute Difference (BUKAN Squared)
            # Ini menjaga konektivitas manifold jNNG agar tidak terputus (sparse)
            delta = abs(z_a - z_b)
            
            psi_vec = self.num_bin_psi[col]
            if psi_vec is None or len(psi_vec) == 0: return delta
            bidx = int(np.clip(delta * self.n_bins_numeric, 0, self.n_bins_numeric - 1))
            scale = max(0.1, 1.0 + psi_vec[bidx])
            return delta * scale
        return 0.0

    def distance(self, x: pd.Series, y: pd.Series) -> float:
        d = 0.0
        for col in y.index:
            w = self.attr_weights.get(col, 1.0)
            d += w * self._dist_attr(col, x[col], y[col])
        return float(d)

    def _compute_distance_matrix(self, X: pd.DataFrame) -> np.ndarray:
        n = len(X)
        D = np.zeros((n, n))
        cols = list(X.columns)
        for i in range(n):
            for j in range(i+1, n):
                d = sum(self.attr_weights[col] * self._dist_attr(col, X.iloc[i][col], X.iloc[j][col]) for col in cols)
                D[i, j] = D[j, i] = d
        return D

    # --- Bombing Core Mechanics ---
    def _build_jnng(self, D_scaled: np.ndarray):
        n = len(D_scaled)
        
        k1_idx = np.argpartition(D_scaled, self.k1 + 1, axis=1)[:, 1:self.k1 + 1] # +1 untuk skip diri sendiri
        k1_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k1_idx[i]:
                k1_adj[i, j] = k1_adj[j, i] = True
                
        # --- k2 (MUTUAL k-NN FIX) ---
        # Ambil k2+1, lalu slice [:, 1:] untuk membuang jarak ke diri sendiri (yang bernilai 0)
        k2_idx = np.argpartition(D_scaled, self.k2 + 1, axis=1)[:, 1:self.k2 + 1] 
        k2_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k2_idx[i]:
                # Cek mutualitas yang sebenarnya (j != i sudah dijamin oleh slice [1:])
                if i in k2_idx[j]: 
                    k2_adj[i, j] = k2_adj[j, i] = True
                
        jnng_adj = k1_adj | k2_adj
        jnng_sparse = csr_matrix(jnng_adj.astype(float) * D_scaled)
        
        # Geodesic distance
        geo_dist = shortest_path(jnng_sparse, method='D', directed=False)
        geo_dist[np.isinf(geo_dist)] = geo_dist[np.isfinite(geo_dist)].max() * 1.5
        
        return jnng_sparse, geo_dist, csr_matrix(jnng_adj.astype(float))

    def _gaussian_kernel_density_old(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        PERBAIKAN MURNI BOMBING: Local Scaling (Zelnik-Perona style).
        Mencegah density flattening dengan membuat bandwidth adaptif terhadap kepadatan lokal.
        """
        n = geo_dist.shape[0]
        k_local = min(7, n - 1)
        
        sigmas = np.partition(geo_dist, k_local, axis=1)[:, k_local]
        sigmas[sigmas == 0] = 1e-12
        
        sigma_matrix = np.outer(sigmas, sigmas)
        rho = np.sum(np.exp(-geo_dist**2 / (2 * sigma_matrix + 1e-12)), axis=1)
        return rho

    def _gaussian_kernel_density(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        Mengikuti Eq. 9 dari Paper Fei et al. (2025).
        Menggunakan varians tetap (dc) untuk menjaga validitas Two-Round Sigmoid.
        """
        # self.dc = 0.15 (atau sesuai parameter default paper)
        rho = np.sum(np.exp(-(geo_dist ** 2) / (2 * (self.dc ** 2))), axis=1)
        return rho

    def _two_round_sigmoid_correction(self, rho_gjnn: np.ndarray) -> np.ndarray:
        rho_max = rho_gjnn.max()
        slope, delta = 10/rho_max, rho_max/2
        rho_tilde = 1 / (1 + np.exp(-slope * (rho_gjnn - delta)))
        mask = rho_tilde < self.T1
        if np.any(mask):
            rho_max_below = rho_gjnn[mask].max()
            s2, d2 = 10/rho_max_below, rho_max_below/2
            rho_tilde[mask] = 1 / (1 + np.exp(-s2 * (rho_tilde[mask] - d2)))
        return rho_tilde

    def _local_neighbor_density(self, D_scaled: np.ndarray, r: float, valid_mask: np.ndarray) -> np.ndarray:
        """Menghitung rho_r (Eq. 16) hanya berdasarkan titik-titik yang valid (belum di-cluster)."""
        n = len(D_scaled)
        rho_r = np.zeros(n)
        for i in range(n):
            if not valid_mask[i]: continue
            # Hitung tetangga dalam radius r dari titik i
            neighbors_in_r = np.sum((D_scaled[i] <= r) & valid_mask)
            rho_r[i] = neighbors_in_r / self.k2
        return rho_r

    def _get_adaptive_tau_elbow(self, unvisited_P: np.ndarray) -> float:
        """
        PERBAIKAN MURNI BOMBING: Mencari 'tebing' kepadatan secara otomatis.
        Menggantikan np.percentile statis yang gagal saat distribusi P datar.
        """
        n_points = len(unvisited_P)
        if n_points < 10:
            return np.percentile(unvisited_P, 75) if n_points > 0 else 0.6
            
        sorted_P = np.sort(unvisited_P)[::-1]
        gradients = np.diff(sorted_P)
        
        n_grad = len(gradients)
        start_idx = max(1, int(n_grad * 0.1))
        end_idx = min(n_grad - 1, int(n_grad * 0.9))
        
        if start_idx >= end_idx:
            return np.percentile(unvisited_P, 75)
            
        valid_gradients = gradients[start_idx:end_idx]
        elbow_idx_in_valid = np.argmin(valid_gradients)
        elbow_idx = start_idx + elbow_idx_in_valid
        
        tau_adaptive = float(sorted_P[elbow_idx])
        return float(np.clip(tau_adaptive, 0.2, 0.8))

    def _bfs_generalized_cluster(self, adj_mat: csr_matrix, core_idx: int, P: np.ndarray, visited_mask: np.ndarray) -> Tuple[List[int], List[int]]:
        R_t, B_t = [], []
        queue = [core_idx]
        visited_mask[core_idx] = True
        R_t.append(core_idx)
        adj_list = adj_mat.tolil().rows
        first_boundary = set()
        
        while queue:
            u = queue.pop(0)
            for v in adj_list[u]:
                if visited_mask[v]: continue
                if P[v] >= self.tau:
                    visited_mask[v] = True
                    R_t.append(v)
                    queue.append(v)
                else:
                    first_boundary.add(v)
                    
        second_boundary = set()
        for v in first_boundary:
            for w in adj_list[v]:
                if not visited_mask[w] and w not in first_boundary and w not in R_t:
                    second_boundary.add(w)
        B_t = list(first_boundary | second_boundary)
        return R_t, B_t

    def _degree_of_adjacency(self, adj_mat: csr_matrix, R_t: List[int], B_t: List[int], existing_clusters: dict) -> Tuple[float, int]:
        if not existing_clusters: return 0.0, -1
        C_star = len(R_t) + len(B_t)
        if C_star == 0: return 0.0, -1
        adj_list = adj_mat.tolil().rows
        max_nap, best_s = 0, -1
        for s, (_, B_s) in existing_clusters.items():
            nap = sum(1 for u in R_t for v in adj_list[u] if v in B_s)
            da = nap / C_star
            if da > max_nap: max_nap, best_s = da, s
        return max_nap, best_s

    def _run_bombing_phase(self, X: pd.DataFrame, D_scaled: np.ndarray, geo_dist: np.ndarray, rho_tilde: np.ndarray, jnng_adj: csr_matrix):
        n = len(X)
        labels = np.full(n, -1, dtype=int)
        visited = np.zeros(n, dtype=bool)
        existing_clusters = {}
        t = 0
        self._reachable_indices = {}
        
        # Copy rho_tilde agar bisa dimanipulasi per iterasi
        current_rho_tilde = rho_tilde.copy()
        
        while not np.all(visited):
            cand = ~visited
            if not np.any(cand): break
            
            # 1. Tentukan Core Point dari titik yang BELUM dikunjungi
            core_idx = np.argmax(current_rho_tilde * cand)
            if current_rho_tilde[core_idx] < 1e-6: break

            # 2. Hitung Ulang Local Neighbor Density (rho_r) berdasarkan sisa titik
            # Paper: r adalah jarak ke k-th nearest neighbor dari core point
            unvisited_indices = np.where(cand)[0]
            if len(unvisited_indices) <= self.k2:
                r = np.max(D_scaled[core_idx, unvisited_indices])
            else:
                r = np.partition(D_scaled[core_idx, unvisited_indices], self.k2)[self.k2]
                
            rho_r = self._local_neighbor_density(D_scaled, r, cand)

            # 3. Mixture Density (Eq. 17)
            P = current_rho_tilde + rho_r
            P_min, P_max = P.min(), P.max()
            if P_max > P_min: 
                P = (P - P_min) / (P_max - P_min)

            # 4. BFS dengan TAU TETAP (0.6) - Ini adalah klaim utama paper!
            R_t, B_t = self._bfs_generalized_cluster(jnng_adj, core_idx, P, visited.copy())
            
            # Tandai titik yang di-cluster dan boundary sebagai visited (dihapus dari X)
            idxs = R_t + B_t
            for idx in idxs: 
                visited[idx] = True

            # 5. Hitung Degree of Adjacency (DA)
            da, best_s = self._degree_of_adjacency(jnng_adj, R_t, B_t, existing_clusters)

            # 6. Aturan Penggabungan (Generalized Cluster)
            if da >= self.T2 and best_s != -1:
                labels[idxs] = best_s
            else:
                labels[idxs] = t
                existing_clusters[t] = (R_t, B_t)
                self._reachable_indices[t] = R_t
                t += 1

            # 7. UPDATE DINAMIS: Nol-kan densitas titik yang sudah di-cluster
            # Agar iterasi berikutnya hanya mencari core point dari sisa dataset
            current_rho_tilde[visited] = 0.0
            
            # Opsional: Recalculate Sigmoid Round 1 & 2 untuk sisa titik jika diperlukan
            # (Paper melakukan recalculate rho_max dari sisa titik)
            rho_max_rem = current_rho_tilde.max()
            if rho_max_rem > 1e-6:
                slope, delta = 10 / rho_max_rem, rho_max_rem / 2
                # Terapkan ulang sigmoid ke titik yang belum visited
                current_rho_tilde[~visited] = 1 / (1 + np.exp(-slope * (current_rho_tilde[~visited] - delta)))

        # Final Clean-up (Assign unvisited/boundary ke nearest visited point)
        unvisited = np.where(labels == -1)[0]
        if len(unvisited) > 0:
            visited_pts = np.where(labels != -1)[0]
            if len(visited_pts) > 0:
                for u in unvisited:
                    labels[u] = labels[visited_pts[np.argmin(D_scaled[u, visited_pts])]]
            else:
                labels[:n//2] = 0
                labels[n//2:] = 1

        self.K_auto_ = len(np.unique(labels[labels != -1]))
        return self.K_auto_, labels

    def _compute_initial_centroids(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        unique_labels = np.unique(labels)
        reachable_dict = getattr(self, '_reachable_indices', {})
        n_points = len(X)
        
        for lbl in unique_labels:
            r_indices = reachable_dict.get(lbl, [])
            if len(r_indices) > 0:
                core_mask = np.zeros(n_points, dtype=bool)
                core_mask[r_indices] = True
                cl = X[core_mask] # Density-Excluded Initialization
            else:
                cl = X[labels == lbl]
                
            c = {}
            for col in X.columns:
                if self.attr_types[col] in ('Nominal', 'Ordinal'):
                    c[col] = _mode(cl[col])
                else:
                    c[col] = cl[col].mean()
            centers.append(c)
        return pd.DataFrame(centers, columns=X.columns)

    # --- EW-LMD Refinement Helpers ---
    def _update_nominal_ordinal_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] not in ('Nominal', 'Ordinal'): continue
            vals = list(pd.unique(X[col]))
            if len(vals) < 2: continue
            P = {}
            for a in vals:
                for b in vals:
                    if a == b: continue
                    val_sum = sum(((X.iloc[np.where(labels==c)[0]][col]==a).sum()/(len(X[labels==c])+eps)) * 
                                  ((X.iloc[np.where(labels==c)[0]][col]==b).sum()/(len(X[labels==c])+eps)) 
                                  for c in range(k) if len(X[labels==c]) > 0)
                    P[(a,b)] = val_sum
            if not P: continue
            P_vals = list(P.values())
            P_max, P_min = max(P_vals), min(P_vals)
            if abs(P_max-P_min) < eps: continue
            
            phi_raw = {key: (-P_max/self.eta_cat if v==P_max else (1-P_min)/self.eta_cat if v==P_min else 0.0) for key, v in P.items()}
            phi_old = self.nominal_phi[col] if self.attr_types[col]=='Nominal' else self.ordinal_phi[col]
            phi_new = {key: np.clip(self.alpha_reg*phi_old.get(key,0.0) + (1-self.alpha_reg)*phi_raw[key], -1, 1) for key in phi_raw}
            
            if self.attr_types[col]=='Nominal':
                self.nominal_phi[col] = phi_new
                self.nominal_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}
            else:
                self.ordinal_phi[col] = phi_new
                self.ordinal_adj_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}

    def _update_numeric_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        B = self.n_bins_numeric
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] != 'Numeric': continue
            if self.num_min[col] >= self.num_max[col]: continue
            z_full = np.clip((pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy() - self.num_min[col]) / (self.num_max[col] - self.num_min[col]), 0.0, 1.0)
            P_sum = np.zeros(B)
            
            for c in range(k):
                idx = np.where(labels==c)[0]
                m = len(idx)
                if m < 2: continue
                if m > self.max_pairs_per_cluster:
                    idx_sample = self._rng.choice(idx, size=self.max_pairs_per_cluster, replace=False)
                else:
                    idx_sample = idx
                m_s = len(idx_sample)
                if m_s < 2: continue
                
                bin_counts = np.zeros(B, dtype=float)
                for i in range(m_s):
                    for j in range(i + 1, m_s):
                        delta = abs(z_full[idx_sample[i]] - z_full[idx_sample[j]])
                        bidx = int(np.clip(delta * B, 0, B - 1))
                        bin_counts[bidx] += 1.0
                total_pairs = bin_counts.sum()
                if total_pairs <= 0: continue
                probs = bin_counts / (total_pairs + eps)
                P_sum += probs * probs

            if np.all(P_sum <= eps): continue
            P_max, P_min = P_sum.max(), P_sum.min()
            if abs(P_max - P_min) < eps: continue

            phi_raw_vec = np.zeros(B, dtype=float)
            for bidx in range(B):
                val = P_sum[bidx]
                if val == P_max: phi = -P_max / self.eta_num
                elif val == P_min: phi = (1.0 - P_min) / self.eta_num
                else: phi = 0.0
                phi_raw_vec[bidx] = phi

            phi_old_vec = self.numeric_phi[col]
            if phi_old_vec.shape[0] != B: phi_old_vec = np.zeros(B, dtype=float)
            phi_new_vec = np.clip(self.alpha_reg * phi_old_vec + (1.0 - self.alpha_reg) * phi_raw_vec, -1.0, 1.0)
            self.numeric_phi[col] = phi_new_vec
            self.num_bin_psi[col] = phi_new_vec.copy()

    def _assign_refinement(self, X: pd.DataFrame, jnng_adj: csr_matrix) -> np.ndarray:
        """
        REFINEMENT DENGAN KENDALA KONEKTIVITAS jNNG (FIX CELAH #1)
        Titik hanya boleh pindah klaster jika memiliki edge jNNG dengan anggota klaster target.
        """
        n = len(X)
        labels = np.empty(n, dtype=int)
        
        # 1. Pre-compute anggota klaster saat ini untuk lookup O(1)
        cluster_members = {c: set(np.where(self.labels_ == c)[0]) for c in range(self.K_auto_)}
        
        # 2. Konversi adjacency matrix ke list of sets untuk akses tetangga yang cepat
        adj_list = jnng_adj.tolil().rows

        for i in range(n):
            # Hitung jarak EW-LMD ke semua sentroid
            dists = [
                sum(self.attr_weights[col] * self._dist_attr(col, X.iloc[i][col], self.centers_.iloc[c][col]) 
                    for col in X.columns) 
                for c in range(self.K_auto_)
            ]
            best_c = int(np.argmin(dists)) # Klaster terdekat secara geometris
            
            # Cek konektivitas topologi jNNG
            neighbors_i = set(adj_list[i])
            
            # Cari klaster mana saja yang memiliki anggota terhubung dengan titik i
            connected_clusters = [
                c for c, members in cluster_members.items() 
                if neighbors_i.intersection(members)
            ]
            
            # 3. Keputusan Alokasi
            if connected_clusters:
                # Jika ada koneksi, pilih klaster yang TERHUBUNG yang jaraknya paling minimum
                labels[i] = min(connected_clusters, key=lambda c: dists[c])
            else:
                # Fallback: Titik terisolasi (tidak ada edge ke klaster manapun)
                # Gunakan jarak geometris murni agar tidak error
                labels[i] = best_c

        return labels

    def _update_centers_refinement(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        for c in range(self.K_auto_):
            cl = X[labels==c]
            if cl.empty: 
                centers.append(self.centers_.iloc[c].to_dict())
                continue
            c_dict = {col: _mode(cl[col]) if self.attr_types[col] in ('Nominal','Ordinal') else cl[col].mean() for col in X.columns}
            centers.append(c_dict)
        return pd.DataFrame(centers)

    def fit(self, df: pd.DataFrame):
        X = df.drop(columns=["Label"], errors="ignore").copy()
        for col in X.columns:
            if self.attr_types[col] == "Numeric":
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())

        # 1. HYBRID PARAMETER TUNING
        structure = self._compute_structure(X)
        params = hybrid_selection_conservative(X, self.attr_types, structure)
        self.dis_const = params['dist_cons']
        self.eta_cat = self.eta_num = params['eta']
        self.n_bins_numeric = params['n_bins_numeric']  # ← PERBAIKAN
        self.auto_params_log = params.copy()
        if self.verbose: print(f"[AUTONOMOUS] Hybrid-tuned: {self.auto_params_log}")

        print("[Phase 0] Computing entropy weights & initializing distances...")
        self._compute_entropy_weights(X)
        self._init_distances(X)
        
        print("[Phase 1] Building jNNG with EW-LMD distance...")
        D = self._compute_distance_matrix(X)
        n = len(X)
        median_dist = np.median(D[np.triu_indices(n, k=1)])
        D_scaled = D / (median_dist + 1e-12)
        
        jnng_sparse, geo_dist, jnng_adj = self._build_jnng(D_scaled)
        
        # Menggunakan Local Scaling di sini
        rho_gjnn = self._gaussian_kernel_density(geo_dist)
        rho_tilde = self._two_round_sigmoid_correction(rho_gjnn)
        
        print("[Phase 2] Running Bombing process (Automatic K detection)...")
        K_before, labels_bombing = self._run_bombing_phase(X, D_scaled, geo_dist, rho_tilde, jnng_adj)
        print(f"   -> Detected K = {K_before}")
        
        self.labels_ = labels_bombing
        self.centers_ = self._compute_initial_centroids(X, labels_bombing)
        
        print("[Phase 2.5] Evaluating structural coherence for conditional merging...")
        metrics = compute_merge_metrics(labels_bombing, jnng_adj, rho_tilde)
        trigger = should_merge_parameter_free(metrics, T2=self.T2)
        
        if trigger:
            print("   -> Trigger activated. Initializing Post-Hoc Merger...")
            merger = PostHocMerger(self, X, D_scaled)
            merge_results = merger.smart_merge_stable(min_K=2, max_merge_ratio=1.0)
            labels_refined = merge_results['labels_merged']
            K_after = merge_results['K_final']
            print(f"   -> Merge Result: K {merge_results['K_before']} → {K_after}")
        else:
            print("   -> Structure coherent. Skipping merging phase.")
            labels_refined = labels_bombing
            K_after = K_before
            
        self.labels_ = labels_refined
        self.K_auto_ = K_after
        
        print("[Phase 3] Computing density-excluded core centroids...")
        if trigger:
            for lbl in np.unique(labels_refined):
                orig_lbls = np.unique(labels_bombing[labels_refined == lbl])
                r_idx = []
                for ol in orig_lbls:
                    r_idx.extend(self._reachable_indices.get(ol, []))
                self._reachable_indices[lbl] = r_idx
                
        self.centers_ = self._compute_initial_centroids(X, labels_refined)
            
        return self

### V5 Bombing + EW-LMD Algorithm Full ver

In [ ]:
class EWLMDBombingHybrid:
    def __init__(self, attr_types: dict, ordinal_orders: dict = None,
                 max_iter_refine: int = 20, random_state: int = 42, verbose: bool = True):
        
        self.attr_types = attr_types.copy()
        self.ordinal_orders = (ordinal_orders or {}).copy()
        self.dis_const = 0.3
        self.eta_cat = 20.0
        self.eta_num = 20.0 
        self.alpha_reg = 0.8
        self.tau_weight = 1e-3
        self.n_bins_numeric = 10
        self.max_pairs_per_cluster = 200
        self.max_iter_refine = max_iter_refine
        self.conv_threshold = 0.98
        self._rng = np.random.default_rng(random_state)
        self.verbose = verbose
        
        # Bombing Parameters
        self.k1, self.k2 = 2, 10
        self.dc, self.tau, self.T1, self.T2 = 0.15, 0.6, 0.9, 0.2
        
        self.attr_weights = {}
        self.nominal_dist, self.ordinal_adj_dist = {}, {}
        self.num_min, self.num_max = {}, {}
        self.num_bin_psi = {}
        self.nominal_phi, self.ordinal_phi, self.numeric_phi = {}, {}, {}
        
        self.labels_ = None
        self.centers_ = None
        self.K_auto_ = None
        self._reachable_indices = {} 

    def _compute_structure(self, X: pd.DataFrame) -> dict:
        counts = {"Numeric": 0, "Ordinal": 0, "Nominal": 0}
        for t in self.attr_types.values():
            counts["Nominal" if t not in ["Numeric", "Ordinal"] else t] += 1
        max_nom = max([X[c].nunique() for c, t in self.attr_types.items() if t == "Nominal"] or [0])
        return {
            'n_rows': len(X), 'n_cols': len(X.columns),
            'n_numeric': counts['Numeric'], 'n_ordinal': counts['Ordinal'], 'n_nominal': counts['Nominal'],
            'max_nominal_card': max_nom,
            'is_wide': len(X.columns) > 20,
            'is_numeric_heavy': counts['Numeric'] > counts['Nominal'],
            'is_nominal_heavy': counts['Nominal'] > counts['Numeric']
        }
    # --- Distance & Weight Helpers ---
    def _compute_entropy_weights(self, X: pd.DataFrame):
        eps = 1e-12
        raw_weights: Dict[str, float] = {}
        
        for col in X.columns:
            t = self.attr_types[col]
            if t in ("Nominal", "Ordinal"):
                probs = X[col].value_counts(normalize=True)
                H = -np.sum(probs * np.log(probs + eps))
                H_max = np.log(len(probs)) if len(probs) > 1 else 1.0
                informativeness = 1.0 - (H / (H_max + eps))
            elif t == "Numeric":
                vals = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy()
                vmin, vmax = vals.min(), vals.max()
                if vmax > vmin:
                    z = (vals - vmin) / (vmax - vmin)
                    n = len(z)
                    max_pairs = min(self.max_pairs_per_cluster, n * (n - 1) // 2)
                    if max_pairs > 0:
                        idx_i = self._rng.integers(0, n, size=max_pairs)
                        idx_j = self._rng.integers(0, n, size=max_pairs)
                        delta = np.abs(z[idx_i] - z[idx_j])
                        hist, _ = np.histogram(delta, bins=self.n_bins_numeric, range=(0.0, 1.0))
                        probs = hist.astype(float) / (hist.sum() + eps)
                        H = -np.sum(probs * np.log(probs + eps))
                        H_max = np.log(self.n_bins_numeric)
                        informativeness = 1.0 - (H / (H_max + eps))
                    else: informativeness = 0.0
                else: informativeness = 0.0
            else: informativeness = 0.0
            raw_weights[col] = max(informativeness, self.tau_weight)
        
        type_groups = {"Nominal": [], "Ordinal": [], "Numeric": []}
        for col in X.columns:
            t = self.attr_types[col]
            if t in type_groups: type_groups[t].append(col)
            
        n_active_types = len([t for t, cols in type_groups.items() if cols])
        weight_per_type = 1.0 / n_active_types if n_active_types > 0 else 1.0
        
        self.attr_weights = {}
        for t, cols in type_groups.items():
            if not cols: continue
            total_raw = sum(raw_weights[col] for col in cols)
            for col in cols:
                self.attr_weights[col] = weight_per_type * (raw_weights[col] / (total_raw + eps))

    def _init_distances(self, X: pd.DataFrame):
        for col in X.columns:
            t = self.attr_types[col]
            if t == "Nominal":
                vals = list(pd.unique(X[col]))
                self.nominal_dist[col] = {(a,b): self.dis_const for a in vals for b in vals if a!=b}
                self.nominal_phi[col] = {(a,b): 0.0 for a in vals for b in vals if a!=b}
            elif t == "Ordinal":
                order = list(self.ordinal_orders[col])
                self.ordinal_adj_dist[col] = {order[i]: self.dis_const for i in range(len(order)-1)}
                self.ordinal_phi[col] = {order[i]: 0.0 for i in range(len(order)-1)}
            elif t == "Numeric":
                v = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())
                self.num_min[col], self.num_max[col] = float(v.min()), float(v.max())
                self.num_bin_psi[col] = np.zeros(self.n_bins_numeric, dtype=float)
                self.numeric_phi[col] = np.zeros(self.n_bins_numeric, dtype=float)

    def _dist_attr(self, col: str, a: Any, b: Any) -> float:
        t = self.attr_types[col]
        if t == "Nominal":
            if a == b: return 0.0
            mp = self.nominal_dist[col]
            return mp.get((a,b), mp.get((b,a), self.dis_const))
        elif t == "Ordinal":
            if a == b: return 0.0
            order = list(self.ordinal_orders[col])
            try: ia, ib = order.index(a), order.index(b)
            except ValueError: return 0.0
            if ia > ib: ia, ib = ib, ia
            return sum(self.ordinal_adj_dist[col].get(order[i], self.dis_const) for i in range(ia, ib))
        elif t == "Numeric":
            vmin, vmax = self.num_min[col], self.num_max[col]
            if vmax <= vmin: return 0.0
            try: va, vb = float(a), float(b)
            except Exception: va, vb = (vmin+vmax)/2.0, (vmin+vmax)/2.0
            
            z_a = np.clip((va - vmin) / (vmax - vmin), 0.0, 1.0)
            z_b = np.clip((vb - vmin) / (vmax - vmin), 0.0, 1.0)
            
            # PERBAIKAN MURNI BOMBING: Absolute Difference (BUKAN Squared)
            # Ini menjaga konektivitas manifold jNNG agar tidak terputus (sparse)
            delta = abs(z_a - z_b)
            
            psi_vec = self.num_bin_psi[col]
            if psi_vec is None or len(psi_vec) == 0: return delta
            bidx = int(np.clip(delta * self.n_bins_numeric, 0, self.n_bins_numeric - 1))
            scale = max(0.1, 1.0 + psi_vec[bidx])
            return delta * scale
        return 0.0

    def distance(self, x: pd.Series, y: pd.Series) -> float:
        d = 0.0
        for col in y.index:
            w = self.attr_weights.get(col, 1.0)
            d += w * self._dist_attr(col, x[col], y[col])
        return float(d)

    def _compute_distance_matrix(self, X: pd.DataFrame) -> np.ndarray:
        n = len(X)
        D = np.zeros((n, n))
        cols = list(X.columns)
        for i in range(n):
            for j in range(i+1, n):
                d = sum(self.attr_weights[col] * self._dist_attr(col, X.iloc[i][col], X.iloc[j][col]) for col in cols)
                D[i, j] = D[j, i] = d
        return D

    # --- Bombing Core Mechanics ---
    def _build_jnng(self, D_scaled: np.ndarray):
        n = len(D_scaled)
        
        k1_idx = np.argpartition(D_scaled, self.k1 + 1, axis=1)[:, 1:self.k1 + 1] # +1 untuk skip diri sendiri
        k1_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k1_idx[i]:
                k1_adj[i, j] = k1_adj[j, i] = True
                
        # --- k2 (MUTUAL k-NN FIX) ---
        # Ambil k2+1, lalu slice [:, 1:] untuk membuang jarak ke diri sendiri (yang bernilai 0)
        k2_idx = np.argpartition(D_scaled, self.k2 + 1, axis=1)[:, 1:self.k2 + 1] 
        k2_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k2_idx[i]:
                # Cek mutualitas yang sebenarnya (j != i sudah dijamin oleh slice [1:])
                if i in k2_idx[j]: 
                    k2_adj[i, j] = k2_adj[j, i] = True
                
        jnng_adj = k1_adj | k2_adj
        jnng_sparse = csr_matrix(jnng_adj.astype(float) * D_scaled)
        
        # Geodesic distance
        geo_dist = shortest_path(jnng_sparse, method='D', directed=False)
        geo_dist[np.isinf(geo_dist)] = geo_dist[np.isfinite(geo_dist)].max() * 1.5
        
        return jnng_sparse, geo_dist, csr_matrix(jnng_adj.astype(float))

    def _gaussian_kernel_density_old(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        PERBAIKAN MURNI BOMBING: Local Scaling (Zelnik-Perona style).
        Mencegah density flattening dengan membuat bandwidth adaptif terhadap kepadatan lokal.
        """
        n = geo_dist.shape[0]
        k_local = min(7, n - 1)
        
        sigmas = np.partition(geo_dist, k_local, axis=1)[:, k_local]
        sigmas[sigmas == 0] = 1e-12
        
        sigma_matrix = np.outer(sigmas, sigmas)
        rho = np.sum(np.exp(-geo_dist**2 / (2 * sigma_matrix + 1e-12)), axis=1)
        return rho

    def _gaussian_kernel_density(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        Mengikuti Eq. 9 dari Paper Fei et al. (2025).
        Menggunakan varians tetap (dc) untuk menjaga validitas Two-Round Sigmoid.
        """
        # self.dc = 0.15 (atau sesuai parameter default paper)
        rho = np.sum(np.exp(-(geo_dist ** 2) / (2 * (self.dc ** 2))), axis=1)
        return rho

    def _two_round_sigmoid_correction(self, rho_gjnn: np.ndarray) -> np.ndarray:
        rho_max = rho_gjnn.max()
        slope, delta = 10/rho_max, rho_max/2
        rho_tilde = 1 / (1 + np.exp(-slope * (rho_gjnn - delta)))
        mask = rho_tilde < self.T1
        if np.any(mask):
            rho_max_below = rho_gjnn[mask].max()
            s2, d2 = 10/rho_max_below, rho_max_below/2
            rho_tilde[mask] = 1 / (1 + np.exp(-s2 * (rho_tilde[mask] - d2)))
        return rho_tilde

    def _local_neighbor_density(self, D_scaled: np.ndarray, r: float, valid_mask: np.ndarray) -> np.ndarray:
        """Menghitung rho_r (Eq. 16) hanya berdasarkan titik-titik yang valid (belum di-cluster)."""
        n = len(D_scaled)
        rho_r = np.zeros(n)
        for i in range(n):
            if not valid_mask[i]: continue
            # Hitung tetangga dalam radius r dari titik i
            neighbors_in_r = np.sum((D_scaled[i] <= r) & valid_mask)
            rho_r[i] = neighbors_in_r / self.k2
        return rho_r

    def _get_adaptive_tau_elbow(self, unvisited_P: np.ndarray) -> float:
        """
        PERBAIKAN MURNI BOMBING: Mencari 'tebing' kepadatan secara otomatis.
        Menggantikan np.percentile statis yang gagal saat distribusi P datar.
        """
        n_points = len(unvisited_P)
        if n_points < 10:
            return np.percentile(unvisited_P, 75) if n_points > 0 else 0.6
            
        sorted_P = np.sort(unvisited_P)[::-1]
        gradients = np.diff(sorted_P)
        
        n_grad = len(gradients)
        start_idx = max(1, int(n_grad * 0.1))
        end_idx = min(n_grad - 1, int(n_grad * 0.9))
        
        if start_idx >= end_idx:
            return np.percentile(unvisited_P, 75)
            
        valid_gradients = gradients[start_idx:end_idx]
        elbow_idx_in_valid = np.argmin(valid_gradients)
        elbow_idx = start_idx + elbow_idx_in_valid
        
        tau_adaptive = float(sorted_P[elbow_idx])
        return float(np.clip(tau_adaptive, 0.2, 0.8))

    def _bfs_generalized_cluster(self, adj_mat: csr_matrix, core_idx: int, P: np.ndarray, visited_mask: np.ndarray) -> Tuple[List[int], List[int]]:
        R_t, B_t = [], []
        queue = [core_idx]
        visited_mask[core_idx] = True
        R_t.append(core_idx)
        adj_list = adj_mat.tolil().rows
        first_boundary = set()
        
        while queue:
            u = queue.pop(0)
            for v in adj_list[u]:
                if visited_mask[v]: continue
                if P[v] >= self.tau:
                    visited_mask[v] = True
                    R_t.append(v)
                    queue.append(v)
                else:
                    first_boundary.add(v)
                    
        second_boundary = set()
        for v in first_boundary:
            for w in adj_list[v]:
                if not visited_mask[w] and w not in first_boundary and w not in R_t:
                    second_boundary.add(w)
        B_t = list(first_boundary | second_boundary)
        return R_t, B_t

    def _degree_of_adjacency(self, adj_mat: csr_matrix, R_t: List[int], B_t: List[int], existing_clusters: dict) -> Tuple[float, int]:
        if not existing_clusters: return 0.0, -1
        C_star = len(R_t) + len(B_t)
        if C_star == 0: return 0.0, -1
        adj_list = adj_mat.tolil().rows
        max_nap, best_s = 0, -1
        for s, (_, B_s) in existing_clusters.items():
            nap = sum(1 for u in R_t for v in adj_list[u] if v in B_s)
            da = nap / C_star
            if da > max_nap: max_nap, best_s = da, s
        return max_nap, best_s

    def _run_bombing_phase(self, X: pd.DataFrame, D_scaled: np.ndarray, geo_dist: np.ndarray, rho_tilde: np.ndarray, jnng_adj: csr_matrix):
        n = len(X)
        labels = np.full(n, -1, dtype=int)
        visited = np.zeros(n, dtype=bool)
        existing_clusters = {}
        t = 0
        self._reachable_indices = {}
        
        # Copy rho_tilde agar bisa dimanipulasi per iterasi
        current_rho_tilde = rho_tilde.copy()
        
        while not np.all(visited):
            cand = ~visited
            if not np.any(cand): break
            
            # 1. Tentukan Core Point dari titik yang BELUM dikunjungi
            core_idx = np.argmax(current_rho_tilde * cand)
            if current_rho_tilde[core_idx] < 1e-6: break

            # 2. Hitung Ulang Local Neighbor Density (rho_r) berdasarkan sisa titik
            # Paper: r adalah jarak ke k-th nearest neighbor dari core point
            unvisited_indices = np.where(cand)[0]
            if len(unvisited_indices) <= self.k2:
                r = np.max(D_scaled[core_idx, unvisited_indices])
            else:
                r = np.partition(D_scaled[core_idx, unvisited_indices], self.k2)[self.k2]
                
            rho_r = self._local_neighbor_density(D_scaled, r, cand)

            # 3. Mixture Density (Eq. 17)
            P = current_rho_tilde + rho_r
            P_min, P_max = P.min(), P.max()
            if P_max > P_min: 
                P = (P - P_min) / (P_max - P_min)

            # 4. BFS dengan TAU TETAP (0.6) - Ini adalah klaim utama paper!
            R_t, B_t = self._bfs_generalized_cluster(jnng_adj, core_idx, P, visited.copy())
            
            # Tandai titik yang di-cluster dan boundary sebagai visited (dihapus dari X)
            idxs = R_t + B_t
            for idx in idxs: 
                visited[idx] = True

            # 5. Hitung Degree of Adjacency (DA)
            da, best_s = self._degree_of_adjacency(jnng_adj, R_t, B_t, existing_clusters)

            # 6. Aturan Penggabungan (Generalized Cluster)
            if da >= self.T2 and best_s != -1:
                labels[idxs] = best_s
            else:
                labels[idxs] = t
                existing_clusters[t] = (R_t, B_t)
                self._reachable_indices[t] = R_t
                t += 1

            # 7. UPDATE DINAMIS: Nol-kan densitas titik yang sudah di-cluster
            # Agar iterasi berikutnya hanya mencari core point dari sisa dataset
            current_rho_tilde[visited] = 0.0
            
            # Opsional: Recalculate Sigmoid Round 1 & 2 untuk sisa titik jika diperlukan
            # (Paper melakukan recalculate rho_max dari sisa titik)
            rho_max_rem = current_rho_tilde.max()
            if rho_max_rem > 1e-6:
                slope, delta = 10 / rho_max_rem, rho_max_rem / 2
                # Terapkan ulang sigmoid ke titik yang belum visited
                current_rho_tilde[~visited] = 1 / (1 + np.exp(-slope * (current_rho_tilde[~visited] - delta)))

        # Final Clean-up (Assign unvisited/boundary ke nearest visited point)
        unvisited = np.where(labels == -1)[0]
        if len(unvisited) > 0:
            visited_pts = np.where(labels != -1)[0]
            if len(visited_pts) > 0:
                for u in unvisited:
                    labels[u] = labels[visited_pts[np.argmin(D_scaled[u, visited_pts])]]
            else:
                labels[:n//2] = 0
                labels[n//2:] = 1

        self.K_auto_ = len(np.unique(labels[labels != -1]))
        return self.K_auto_, labels

    def _compute_initial_centroids(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        unique_labels = np.unique(labels)
        reachable_dict = getattr(self, '_reachable_indices', {})
        n_points = len(X)
        
        for lbl in unique_labels:
            r_indices = reachable_dict.get(lbl, [])
            if len(r_indices) > 0:
                core_mask = np.zeros(n_points, dtype=bool)
                core_mask[r_indices] = True
                cl = X[core_mask] # Density-Excluded Initialization
            else:
                cl = X[labels == lbl]
                
            c = {}
            for col in X.columns:
                if self.attr_types[col] in ('Nominal', 'Ordinal'):
                    c[col] = _mode(cl[col])
                else:
                    c[col] = cl[col].mean()
            centers.append(c)
        return pd.DataFrame(centers, columns=X.columns)

    # --- EW-LMD Refinement Helpers ---
    def _update_nominal_ordinal_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] not in ('Nominal', 'Ordinal'): continue
            vals = list(pd.unique(X[col]))
            if len(vals) < 2: continue
            P = {}
            for a in vals:
                for b in vals:
                    if a == b: continue
                    val_sum = sum(((X.iloc[np.where(labels==c)[0]][col]==a).sum()/(len(X[labels==c])+eps)) * 
                                  ((X.iloc[np.where(labels==c)[0]][col]==b).sum()/(len(X[labels==c])+eps)) 
                                  for c in range(k) if len(X[labels==c]) > 0)
                    P[(a,b)] = val_sum
            if not P: continue
            P_vals = list(P.values())
            P_max, P_min = max(P_vals), min(P_vals)
            if abs(P_max-P_min) < eps: continue
            
            phi_raw = {key: (-P_max/self.eta_cat if v==P_max else (1-P_min)/self.eta_cat if v==P_min else 0.0) for key, v in P.items()}
            phi_old = self.nominal_phi[col] if self.attr_types[col]=='Nominal' else self.ordinal_phi[col]
            phi_new = {key: np.clip(self.alpha_reg*phi_old.get(key,0.0) + (1-self.alpha_reg)*phi_raw[key], -1, 1) for key in phi_raw}
            
            if self.attr_types[col]=='Nominal':
                self.nominal_phi[col] = phi_new
                self.nominal_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}
            else:
                self.ordinal_phi[col] = phi_new
                self.ordinal_adj_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}

    def _update_numeric_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        B = self.n_bins_numeric
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] != 'Numeric': continue
            if self.num_min[col] >= self.num_max[col]: continue
            z_full = np.clip((pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy() - self.num_min[col]) / (self.num_max[col] - self.num_min[col]), 0.0, 1.0)
            P_sum = np.zeros(B)
            
            for c in range(k):
                idx = np.where(labels==c)[0]
                m = len(idx)
                if m < 2: continue
                if m > self.max_pairs_per_cluster:
                    idx_sample = self._rng.choice(idx, size=self.max_pairs_per_cluster, replace=False)
                else:
                    idx_sample = idx
                m_s = len(idx_sample)
                if m_s < 2: continue
                
                bin_counts = np.zeros(B, dtype=float)
                for i in range(m_s):
                    for j in range(i + 1, m_s):
                        delta = abs(z_full[idx_sample[i]] - z_full[idx_sample[j]])
                        bidx = int(np.clip(delta * B, 0, B - 1))
                        bin_counts[bidx] += 1.0
                total_pairs = bin_counts.sum()
                if total_pairs <= 0: continue
                probs = bin_counts / (total_pairs + eps)
                P_sum += probs * probs

            if np.all(P_sum <= eps): continue
            P_max, P_min = P_sum.max(), P_sum.min()
            if abs(P_max - P_min) < eps: continue

            phi_raw_vec = np.zeros(B, dtype=float)
            for bidx in range(B):
                val = P_sum[bidx]
                if val == P_max: phi = -P_max / self.eta_num
                elif val == P_min: phi = (1.0 - P_min) / self.eta_num
                else: phi = 0.0
                phi_raw_vec[bidx] = phi

            phi_old_vec = self.numeric_phi[col]
            if phi_old_vec.shape[0] != B: phi_old_vec = np.zeros(B, dtype=float)
            phi_new_vec = np.clip(self.alpha_reg * phi_old_vec + (1.0 - self.alpha_reg) * phi_raw_vec, -1.0, 1.0)
            self.numeric_phi[col] = phi_new_vec
            self.num_bin_psi[col] = phi_new_vec.copy()

    def _assign_refinement(self, X: pd.DataFrame, jnng_adj: csr_matrix) -> np.ndarray:
        """
        REFINEMENT DENGAN KENDALA KONEKTIVITAS jNNG (FIX CELAH #1)
        Titik hanya boleh pindah klaster jika memiliki edge jNNG dengan anggota klaster target.
        """
        n = len(X)
        labels = np.empty(n, dtype=int)
        
        # 1. Pre-compute anggota klaster saat ini untuk lookup O(1)
        cluster_members = {c: set(np.where(self.labels_ == c)[0]) for c in range(self.K_auto_)}
        
        # 2. Konversi adjacency matrix ke list of sets untuk akses tetangga yang cepat
        adj_list = jnng_adj.tolil().rows

        for i in range(n):
            # Hitung jarak EW-LMD ke semua sentroid
            dists = [
                sum(self.attr_weights[col] * self._dist_attr(col, X.iloc[i][col], self.centers_.iloc[c][col]) 
                    for col in X.columns) 
                for c in range(self.K_auto_)
            ]
            best_c = int(np.argmin(dists)) # Klaster terdekat secara geometris
            
            # Cek konektivitas topologi jNNG
            neighbors_i = set(adj_list[i])
            
            # Cari klaster mana saja yang memiliki anggota terhubung dengan titik i
            connected_clusters = [
                c for c, members in cluster_members.items() 
                if neighbors_i.intersection(members)
            ]
            
            # 3. Keputusan Alokasi
            if connected_clusters:
                # Jika ada koneksi, pilih klaster yang TERHUBUNG yang jaraknya paling minimum
                labels[i] = min(connected_clusters, key=lambda c: dists[c])
            else:
                # Fallback: Titik terisolasi (tidak ada edge ke klaster manapun)
                # Gunakan jarak geometris murni agar tidak error
                labels[i] = best_c

        return labels

    def _update_centers_refinement(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        for c in range(self.K_auto_):
            cl = X[labels==c]
            if cl.empty: 
                centers.append(self.centers_.iloc[c].to_dict())
                continue
            c_dict = {col: _mode(cl[col]) if self.attr_types[col] in ('Nominal','Ordinal') else cl[col].mean() for col in X.columns}
            centers.append(c_dict)
        return pd.DataFrame(centers)

    def fit(self, df: pd.DataFrame):
        X = df.drop(columns=["Label"], errors="ignore").copy()
        for col in X.columns:
            if self.attr_types[col] == "Numeric":
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())

        # 1. HYBRID PARAMETER TUNING
        structure = self._compute_structure(X)
        params = hybrid_selection_conservative(X, self.attr_types, structure)
        self.dis_const = params['dist_cons']
        self.eta_cat = self.eta_num = params['eta']
        self.n_bins_numeric = params['n_bins_numeric']  # ← PERBAIKAN
        self.auto_params_log = params.copy()
        if self.verbose: print(f"[AUTONOMOUS] Hybrid-tuned: {self.auto_params_log}")
        # if not hasattr(self, 'dis_const') or self.dis_const < 0.2:
        #     # Estimasi median jarak numerik, lalu set dis_const ~30-50% dari median
        #     # (Opsional: bisa hardcode 0.35 untuk sintetik mixed-type)
        #     self.dis_const = 0.30
        #     if self.verbose: print(f"[AUTONOMOUS] _dist_const-tuned: {self.dis_const}")
        # print(f"[MANUAL] _dist_const: {self.dis_const}, _eta: {self.eta_cat}, _n_bins_numeric: {self.n_bins_numeric}")

        print("[Phase 0] Computing entropy weights & initializing distances...")
        self._compute_entropy_weights(X)
        self._init_distances(X)
        
        print("[Phase 1] Building jNNG with EW-LMD distance...")
        D = self._compute_distance_matrix(X)
        n = len(X)
        median_dist = np.median(D[np.triu_indices(n, k=1)])
        D_scaled = D / (median_dist + 1e-12)
        
        jnng_sparse, geo_dist, jnng_adj = self._build_jnng(D_scaled)
        
        # Menggunakan Local Scaling di sini
        rho_gjnn = self._gaussian_kernel_density(geo_dist)
        rho_tilde = self._two_round_sigmoid_correction(rho_gjnn)
        
        print("[Phase 2] Running Bombing process (Automatic K detection)...")
        K_before, labels_bombing = self._run_bombing_phase(X, D_scaled, geo_dist, rho_tilde, jnng_adj)
        print(f"   -> Detected K = {K_before}")
        
        self.labels_ = labels_bombing
        self.centers_ = self._compute_initial_centroids(X, labels_bombing)
        
        print("[Phase 2.5] Evaluating structural coherence for conditional merging...")
        metrics = compute_merge_metrics(labels_bombing, jnng_adj, rho_tilde)
        trigger = should_merge_parameter_free(metrics, T2=self.T2)
        
        if trigger:
            print("   -> Trigger activated. Initializing Post-Hoc Merger...")
            merger = PostHocMerger(self, X, D_scaled)
            merge_results = merger.smart_merge_stable(min_K=2, max_merge_ratio=1.0)
            labels_refined = merge_results['labels_merged']
            K_after = merge_results['K_final']
            print(f"   -> Merge Result: K {merge_results['K_before']} → {K_after}")
        else:
            print("   -> Structure coherent. Skipping merging phase.")
            labels_refined = labels_bombing
            K_after = K_before
            
        self.labels_ = labels_refined
        self.K_auto_ = K_after
        
        print("[Phase 3] Computing density-excluded core centroids...")
        if trigger:
            for lbl in np.unique(labels_refined):
                orig_lbls = np.unique(labels_bombing[labels_refined == lbl])
                r_idx = []
                for ol in orig_lbls:
                    r_idx.extend(self._reachable_indices.get(ol, []))
                self._reachable_indices[lbl] = r_idx
                
        self.centers_ = self._compute_initial_centroids(X, labels_refined)
        
        print("[Phase 4-6] EW-LMD Iterative Refinement Loop...")
        prev_labels = self.labels_.copy()
        best_silhouette = -1.0
        best_labels = self.labels_.copy()
        restart_count = 0
        
        for step in range(self.max_iter_refine):
            current_alpha = 0.5 if step < 2 else self.alpha_reg
            orig_alpha = self.alpha_reg
            self.alpha_reg = current_alpha
            
            self._update_nominal_ordinal_phi(X, self.labels_)
            self._update_numeric_phi(X, self.labels_)
            self.alpha_reg = orig_alpha
            
            self.labels_ = self._assign_refinement(X, jnng_adj)
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
            if step >= 2:
                ari_internal = adjusted_rand_score(prev_labels, self.labels_)
                
                # === PERBAIKAN: TRAP DETECTION via Silhouette ===
                sample_idx = np.random.choice(len(X), min(1000, len(X)), replace=False) if len(X) > 1000 else np.arange(len(X))
                D_sample = np.zeros((len(sample_idx), len(sample_idx)))
                for i, idx_i in enumerate(sample_idx):
                    for j, idx_j in enumerate(sample_idx):
                        # Menggunakan fungsi distance yang sudah ada
                        D_sample[i,j] = self.distance(X.iloc[idx_i], X.iloc[idx_j])
                
                try:
                    current_sil = silhouette_score(D_sample, self.labels_[sample_idx], metric='precomputed')
                except ValueError:
                    current_sil = 0.0
                    
                if current_sil > best_silhouette:
                    best_silhouette = current_sil
                    best_labels = self.labels_.copy()
                    
                print(f"   Iter {step+1}: Internal ARI = {ari_internal:.4f} | Silhouette = {current_sil:.4f}")
                
                if ari_internal > self.conv_threshold:
                    if current_sil < 0.1 and restart_count < 2:
                        print(f"   -> [Trap Detected] Silhouette < 0.1. Perturbing labels to escape local optimum...")
                        noise_mask = np.random.rand(len(X)) < 0.2
                        self.labels_[noise_mask] = self._rng.integers(0, self.K_auto_, size=np.sum(noise_mask))
                        self.centers_ = self._update_centers_refinement(X, self.labels_)
                        restart_count += 1
                        prev_labels = self.labels_.copy()
                        continue
                    else:
                        print("   -> Convergence reached.")
                        break
                        
            prev_labels = self.labels_.copy()

        if best_silhouette > current_sil:
            if self.verbose: print(f"   -> [Recovery] Restoring best labels from Iter with Silhouette = {best_silhouette:.4f}")
            self.labels_ = best_labels
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
        return self

### V6 Bombing + LMD without PM

In [ ]:
class LMDBombingHybrid_V2:
    def __init__(self, attr_types: dict, ordinal_orders: dict = None,
                 max_iter_refine: int = 20, random_state: int = 42, verbose: bool = True):
        
        self.attr_types = attr_types.copy()
        self.ordinal_orders = (ordinal_orders or {}).copy()
        self.dis_const = 0.3
        self.eta_cat = 20.0
        self.eta_num = 20.0 
        self.alpha_reg = 0.8
        self.tau_weight = 1e-3
        self.n_bins_numeric = 10
        self.max_pairs_per_cluster = 200
        self.max_iter_refine = max_iter_refine
        self.conv_threshold = 0.98
        self._rng = np.random.default_rng(random_state)
        self.verbose = verbose
        
        # Bombing Parameters
        self.k1, self.k2 = 2, 10
        self.dc, self.tau, self.T1, self.T2 = 0.15, 0.6, 0.9, 0.2
        
        # self.attr_weights = {}
        self.nominal_dist, self.ordinal_adj_dist = {}, {}
        self.num_min, self.num_max = {}, {}
        self.num_bin_psi = {}
        self.nominal_phi, self.ordinal_phi, self.numeric_phi = {}, {}, {}
        
        self.labels_ = None
        self.centers_ = None
        self.K_auto_ = None
        self._reachable_indices = {} 

    def _compute_structure(self, X: pd.DataFrame) -> dict:
        counts = {"Numeric": 0, "Ordinal": 0, "Nominal": 0}
        for t in self.attr_types.values():
            counts["Nominal" if t not in ["Numeric", "Ordinal"] else t] += 1
        max_nom = max([X[c].nunique() for c, t in self.attr_types.items() if t == "Nominal"] or [0])
        return {
            'n_rows': len(X), 'n_cols': len(X.columns),
            'n_numeric': counts['Numeric'], 'n_ordinal': counts['Ordinal'], 'n_nominal': counts['Nominal'],
            'max_nominal_card': max_nom,
            'is_wide': len(X.columns) > 20,
            'is_numeric_heavy': counts['Numeric'] > counts['Nominal'],
            'is_nominal_heavy': counts['Nominal'] > counts['Numeric']
        }
    # --- Distance Helpers ---
    def _init_distances(self, X: pd.DataFrame):
        for col in X.columns:
            t = self.attr_types[col]
            if t == "Nominal":
                vals = list(pd.unique(X[col]))
                self.nominal_dist[col] = {(a,b): self.dis_const for a in vals for b in vals if a!=b}
                self.nominal_phi[col] = {(a,b): 0.0 for a in vals for b in vals if a!=b}
            elif t == "Ordinal":
                order = list(self.ordinal_orders[col])
                self.ordinal_adj_dist[col] = {order[i]: self.dis_const for i in range(len(order)-1)}
                self.ordinal_phi[col] = {order[i]: 0.0 for i in range(len(order)-1)}
            elif t == "Numeric":
                v = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())
                self.num_min[col], self.num_max[col] = float(v.min()), float(v.max())
                self.num_bin_psi[col] = np.zeros(self.n_bins_numeric, dtype=float)
                self.numeric_phi[col] = np.zeros(self.n_bins_numeric, dtype=float)

    def _dist_attr(self, col: str, a: Any, b: Any) -> float:
        t = self.attr_types[col]
        if t == "Nominal":
            if a == b: return 0.0
            mp = self.nominal_dist[col]
            return mp.get((a,b), mp.get((b,a), self.dis_const))
        elif t == "Ordinal":
            if a == b: return 0.0
            order = list(self.ordinal_orders[col])
            try: ia, ib = order.index(a), order.index(b)
            except ValueError: return 0.0
            if ia > ib: ia, ib = ib, ia
            return sum(self.ordinal_adj_dist[col].get(order[i], self.dis_const) for i in range(ia, ib))
        elif t == "Numeric":
            vmin, vmax = self.num_min[col], self.num_max[col]
            if vmax <= vmin: return 0.0
            try: va, vb = float(a), float(b)
            except Exception: va, vb = (vmin+vmax)/2.0, (vmin+vmax)/2.0
            
            z_a = np.clip((va - vmin) / (vmax - vmin), 0.0, 1.0)
            z_b = np.clip((vb - vmin) / (vmax - vmin), 0.0, 1.0)
            
            # PERBAIKAN MURNI BOMBING: Absolute Difference (BUKAN Squared)
            # Ini menjaga konektivitas manifold jNNG agar tidak terputus (sparse)
            delta = abs(z_a - z_b)
            
            psi_vec = self.num_bin_psi[col]
            if psi_vec is None or len(psi_vec) == 0: return delta
            bidx = int(np.clip(delta * self.n_bins_numeric, 0, self.n_bins_numeric - 1))
            scale = max(0.1, 1.0 + psi_vec[bidx])
            return delta * scale
        return 0.0

    def distance(self, x: pd.Series, y: pd.Series) -> float:
        d = 0.0
        for col in y.index:
            d += self._dist_attr(col, x[col], y[col])
        return float(d)

    def _compute_distance_matrix(self, X: pd.DataFrame) -> np.ndarray:
        n = len(X)
        D = np.zeros((n, n))
        cols = list(X.columns)
        for i in range(n):
            for j in range(i+1, n):
                d = sum(self._dist_attr(col, X.iloc[i][col], X.iloc[j][col]) for col in cols)
                D[i, j] = D[j, i] = d
        return D

    # --- Bombing Core Mechanics ---
    def _build_jnng(self, D_scaled: np.ndarray):
        n = len(D_scaled)
        
        k1_idx = np.argpartition(D_scaled, self.k1 + 1, axis=1)[:, 1:self.k1 + 1] # +1 untuk skip diri sendiri
        k1_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k1_idx[i]:
                k1_adj[i, j] = k1_adj[j, i] = True
                
        # --- k2 (MUTUAL k-NN FIX) ---
        # Ambil k2+1, lalu slice [:, 1:] untuk membuang jarak ke diri sendiri (yang bernilai 0)
        k2_idx = np.argpartition(D_scaled, self.k2 + 1, axis=1)[:, 1:self.k2 + 1] 
        k2_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k2_idx[i]:
                # Cek mutualitas yang sebenarnya (j != i sudah dijamin oleh slice [1:])
                if i in k2_idx[j]: 
                    k2_adj[i, j] = k2_adj[j, i] = True
                
        jnng_adj = k1_adj | k2_adj
        jnng_sparse = csr_matrix(jnng_adj.astype(float) * D_scaled)
        
        # Geodesic distance
        geo_dist = shortest_path(jnng_sparse, method='D', directed=False)
        geo_dist[np.isinf(geo_dist)] = geo_dist[np.isfinite(geo_dist)].max() * 1.5
        
        return jnng_sparse, geo_dist, csr_matrix(jnng_adj.astype(float))

    def _gaussian_kernel_density(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        Mengikuti Eq. 9 dari Paper Fei et al. (2025).
        Menggunakan varians tetap (dc) untuk menjaga validitas Two-Round Sigmoid.
        """
        # self.dc = 0.15 (atau sesuai parameter default paper)
        rho = np.sum(np.exp(-(geo_dist ** 2) / (2 * (self.dc ** 2))), axis=1)
        return rho

    def _two_round_sigmoid_correction(self, rho_gjnn: np.ndarray) -> np.ndarray:
        rho_max = rho_gjnn.max()
        slope, delta = 10/rho_max, rho_max/2
        rho_tilde = 1 / (1 + np.exp(-slope * (rho_gjnn - delta)))
        mask = rho_tilde < self.T1
        if np.any(mask):
            rho_max_below = rho_gjnn[mask].max()
            s2, d2 = 10/rho_max_below, rho_max_below/2
            rho_tilde[mask] = 1 / (1 + np.exp(-s2 * (rho_tilde[mask] - d2)))
        return rho_tilde

    def _local_neighbor_density(self, D_scaled: np.ndarray, r: float, valid_mask: np.ndarray) -> np.ndarray:
        """Menghitung rho_r (Eq. 16) hanya berdasarkan titik-titik yang valid (belum di-cluster)."""
        n = len(D_scaled)
        rho_r = np.zeros(n)
        for i in range(n):
            if not valid_mask[i]: continue
            # Hitung tetangga dalam radius r dari titik i
            neighbors_in_r = np.sum((D_scaled[i] <= r) & valid_mask)
            rho_r[i] = neighbors_in_r / self.k2
        return rho_r

    def _bfs_generalized_cluster(self, adj_mat: csr_matrix, core_idx: int, P: np.ndarray, visited_mask: np.ndarray) -> Tuple[List[int], List[int]]:
        R_t, B_t = [], []
        queue = [core_idx]
        visited_mask[core_idx] = True
        R_t.append(core_idx)
        adj_list = adj_mat.tolil().rows
        first_boundary = set()
        
        while queue:
            u = queue.pop(0)
            for v in adj_list[u]:
                if visited_mask[v]: continue
                if P[v] >= self.tau:
                    visited_mask[v] = True
                    R_t.append(v)
                    queue.append(v)
                else:
                    first_boundary.add(v)
                    
        second_boundary = set()
        for v in first_boundary:
            for w in adj_list[v]:
                if not visited_mask[w] and w not in first_boundary and w not in R_t:
                    second_boundary.add(w)
        B_t = list(first_boundary | second_boundary)
        return R_t, B_t

    def _degree_of_adjacency(self, adj_mat: csr_matrix, R_t: List[int], B_t: List[int], existing_clusters: dict) -> Tuple[float, int]:
        if not existing_clusters: return 0.0, -1
        C_star = len(R_t) + len(B_t)
        if C_star == 0: return 0.0, -1
        adj_list = adj_mat.tolil().rows
        max_nap, best_s = 0, -1
        for s, (_, B_s) in existing_clusters.items():
            nap = sum(1 for u in R_t for v in adj_list[u] if v in B_s)
            da = nap / C_star
            if da > max_nap: max_nap, best_s = da, s
        return max_nap, best_s

    def _run_bombing_phase(self, X: pd.DataFrame, D_scaled: np.ndarray, geo_dist: np.ndarray, rho_tilde: np.ndarray, jnng_adj: csr_matrix):
        n = len(X)
        labels = np.full(n, -1, dtype=int)
        visited = np.zeros(n, dtype=bool)
        existing_clusters = {}
        t = 0
        self._reachable_indices = {}
        
        # Copy rho_tilde agar bisa dimanipulasi per iterasi
        current_rho_tilde = rho_tilde.copy()
        
        while not np.all(visited):
            cand = ~visited
            if not np.any(cand): break
            
            # 1. Tentukan Core Point dari titik yang BELUM dikunjungi
            core_idx = np.argmax(current_rho_tilde * cand)
            if current_rho_tilde[core_idx] < 1e-6: break

            # 2. Hitung Ulang Local Neighbor Density (rho_r) berdasarkan sisa titik
            # Paper: r adalah jarak ke k-th nearest neighbor dari core point
            unvisited_indices = np.where(cand)[0]
            if len(unvisited_indices) <= self.k2:
                r = np.max(D_scaled[core_idx, unvisited_indices])
            else:
                r = np.partition(D_scaled[core_idx, unvisited_indices], self.k2)[self.k2]
                
            rho_r = self._local_neighbor_density(D_scaled, r, cand)

            # 3. Mixture Density (Eq. 17)
            P = current_rho_tilde + rho_r
            P_min, P_max = P.min(), P.max()
            if P_max > P_min: 
                P = (P - P_min) / (P_max - P_min)

            # 4. BFS dengan TAU TETAP (0.6) - Ini adalah klaim utama paper!
            R_t, B_t = self._bfs_generalized_cluster(jnng_adj, core_idx, P, visited.copy())
            
            # Tandai titik yang di-cluster dan boundary sebagai visited (dihapus dari X)
            idxs = R_t + B_t
            for idx in idxs: 
                visited[idx] = True

            # 5. Hitung Degree of Adjacency (DA)
            da, best_s = self._degree_of_adjacency(jnng_adj, R_t, B_t, existing_clusters)

            # 6. Aturan Penggabungan (Generalized Cluster)
            if da >= self.T2 and best_s != -1:
                labels[idxs] = best_s
            else:
                labels[idxs] = t
                existing_clusters[t] = (R_t, B_t)
                self._reachable_indices[t] = R_t
                t += 1

            # 7. UPDATE DINAMIS: Nol-kan densitas titik yang sudah di-cluster
            # Agar iterasi berikutnya hanya mencari core point dari sisa dataset
            current_rho_tilde[visited] = 0.0
            
            # Opsional: Recalculate Sigmoid Round 1 & 2 untuk sisa titik jika diperlukan
            # (Paper melakukan recalculate rho_max dari sisa titik)
            rho_max_rem = current_rho_tilde.max()
            if rho_max_rem > 1e-6:
                slope, delta = 10 / rho_max_rem, rho_max_rem / 2
                # Terapkan ulang sigmoid ke titik yang belum visited
                current_rho_tilde[~visited] = 1 / (1 + np.exp(-slope * (current_rho_tilde[~visited] - delta)))

        # Final Clean-up (Assign unvisited/boundary ke nearest visited point)
        unvisited = np.where(labels == -1)[0]
        if len(unvisited) > 0:
            visited_pts = np.where(labels != -1)[0]
            if len(visited_pts) > 0:
                for u in unvisited:
                    labels[u] = labels[visited_pts[np.argmin(D_scaled[u, visited_pts])]]
            else:
                labels[:n//2] = 0
                labels[n//2:] = 1

        self.K_auto_ = len(np.unique(labels[labels != -1]))
        return self.K_auto_, labels

    def _compute_initial_centroids(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        unique_labels = np.unique(labels)
        reachable_dict = getattr(self, '_reachable_indices', {})
        n_points = len(X)
        
        for lbl in unique_labels:
            r_indices = reachable_dict.get(lbl, [])
            if len(r_indices) > 0:
                core_mask = np.zeros(n_points, dtype=bool)
                core_mask[r_indices] = True
                cl = X[core_mask] # Density-Excluded Initialization
            else:
                cl = X[labels == lbl]
                
            c = {}
            for col in X.columns:
                if self.attr_types[col] in ('Nominal', 'Ordinal'):
                    c[col] = _mode(cl[col])
                else:
                    c[col] = cl[col].mean()
            centers.append(c)
        return pd.DataFrame(centers, columns=X.columns)

    # --- EW-LMD Refinement Helpers ---
    def _update_nominal_ordinal_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] not in ('Nominal', 'Ordinal'): continue
            vals = list(pd.unique(X[col]))
            if len(vals) < 2: continue
            P = {}
            for a in vals:
                for b in vals:
                    if a == b: continue
                    val_sum = sum(((X.iloc[np.where(labels==c)[0]][col]==a).sum()/(len(X[labels==c])+eps)) * 
                                  ((X.iloc[np.where(labels==c)[0]][col]==b).sum()/(len(X[labels==c])+eps)) 
                                  for c in range(k) if len(X[labels==c]) > 0)
                    P[(a,b)] = val_sum
            if not P: continue
            P_vals = list(P.values())
            P_max, P_min = max(P_vals), min(P_vals)
            if abs(P_max-P_min) < eps: continue
            
            phi_raw = {key: (-P_max/self.eta_cat if v==P_max else (1-P_min)/self.eta_cat if v==P_min else 0.0) for key, v in P.items()}
            phi_old = self.nominal_phi[col] if self.attr_types[col]=='Nominal' else self.ordinal_phi[col]
            phi_new = {key: np.clip(self.alpha_reg*phi_old.get(key,0.0) + (1-self.alpha_reg)*phi_raw[key], -1, 1) for key in phi_raw}
            
            if self.attr_types[col]=='Nominal':
                self.nominal_phi[col] = phi_new
                self.nominal_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}
            else:
                self.ordinal_phi[col] = phi_new
                self.ordinal_adj_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}

    def _update_numeric_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        B = self.n_bins_numeric
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] != 'Numeric': continue
            if self.num_min[col] >= self.num_max[col]: continue
            z_full = np.clip((pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy() - self.num_min[col]) / (self.num_max[col] - self.num_min[col]), 0.0, 1.0)
            P_sum = np.zeros(B)
            
            for c in range(k):
                idx = np.where(labels==c)[0]
                m = len(idx)
                if m < 2: continue
                if m > self.max_pairs_per_cluster:
                    idx_sample = self._rng.choice(idx, size=self.max_pairs_per_cluster, replace=False)
                else:
                    idx_sample = idx
                m_s = len(idx_sample)
                if m_s < 2: continue
                
                bin_counts = np.zeros(B, dtype=float)
                for i in range(m_s):
                    for j in range(i + 1, m_s):
                        delta = abs(z_full[idx_sample[i]] - z_full[idx_sample[j]])
                        bidx = int(np.clip(delta * B, 0, B - 1))
                        bin_counts[bidx] += 1.0
                total_pairs = bin_counts.sum()
                if total_pairs <= 0: continue
                probs = bin_counts / (total_pairs + eps)
                P_sum += probs * probs

            if np.all(P_sum <= eps): continue
            P_max, P_min = P_sum.max(), P_sum.min()
            if abs(P_max - P_min) < eps: continue

            phi_raw_vec = np.zeros(B, dtype=float)
            for bidx in range(B):
                val = P_sum[bidx]
                if val == P_max: phi = -P_max / self.eta_num
                elif val == P_min: phi = (1.0 - P_min) / self.eta_num
                else: phi = 0.0
                phi_raw_vec[bidx] = phi

            phi_old_vec = self.numeric_phi[col]
            if phi_old_vec.shape[0] != B: phi_old_vec = np.zeros(B, dtype=float)
            phi_new_vec = np.clip(self.alpha_reg * phi_old_vec + (1.0 - self.alpha_reg) * phi_raw_vec, -1.0, 1.0)
            self.numeric_phi[col] = phi_new_vec
            self.num_bin_psi[col] = phi_new_vec.copy()

    def _assign_refinement(self, X: pd.DataFrame, jnng_adj: csr_matrix) -> np.ndarray:
        """
        REFINEMENT DENGAN KENDALA KONEKTIVITAS jNNG (FIX CELAH #1)
        Titik hanya boleh pindah klaster jika memiliki edge jNNG dengan anggota klaster target.
        """
        n = len(X)
        labels = np.empty(n, dtype=int)
        
        # 1. Pre-compute anggota klaster saat ini untuk lookup O(1)
        cluster_members = {c: set(np.where(self.labels_ == c)[0]) for c in range(self.K_auto_)}
        
        # 2. Konversi adjacency matrix ke list of sets untuk akses tetangga yang cepat
        adj_list = jnng_adj.tolil().rows

        for i in range(n):
            # Hitung jarak EW-LMD ke semua sentroid
            dists = [
                sum(self._dist_attr(col, X.iloc[i][col], self.centers_.iloc[c][col]) 
                    for col in X.columns) 
                for c in range(self.K_auto_)
            ]
            best_c = int(np.argmin(dists)) # Klaster terdekat secara geometris
            
            # Cek konektivitas topologi jNNG
            neighbors_i = set(adj_list[i])
            
            # Cari klaster mana saja yang memiliki anggota terhubung dengan titik i
            connected_clusters = [
                c for c, members in cluster_members.items() 
                if neighbors_i.intersection(members)
            ]
            
            # 3. Keputusan Alokasi
            if connected_clusters:
                # Jika ada koneksi, pilih klaster yang TERHUBUNG yang jaraknya paling minimum
                labels[i] = min(connected_clusters, key=lambda c: dists[c])
            else:
                # Fallback: Titik terisolasi (tidak ada edge ke klaster manapun)
                # Gunakan jarak geometris murni agar tidak error
                labels[i] = best_c

        return labels

    def _update_centers_refinement(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        for c in range(self.K_auto_):
            cl = X[labels==c]
            if cl.empty: 
                centers.append(self.centers_.iloc[c].to_dict())
                continue
            c_dict = {col: _mode(cl[col]) if self.attr_types[col] in ('Nominal','Ordinal') else cl[col].mean() for col in X.columns}
            centers.append(c_dict)
        return pd.DataFrame(centers)

    def fit(self, df: pd.DataFrame):
        X = df.drop(columns=["Label"], errors="ignore").copy()
        for col in X.columns:
            if self.attr_types[col] == "Numeric":
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())

        # 1. HYBRID PARAMETER TUNING
        structure = self._compute_structure(X)
        params = hybrid_selection_conservative(X, self.attr_types, structure)
        self.dis_const = params['dist_cons']
        self.eta_cat = self.eta_num = params['eta']
        self.n_bins_numeric = params['n_bins_numeric']  # ← PERBAIKAN
        self.auto_params_log = params.copy()
        if self.verbose: print(f"[AUTONOMOUS] Hybrid-tuned: {self.auto_params_log}")

        print("[Phase 0] Computing initializing distances...")
        # self._compute_entropy_weights(X)
        self._init_distances(X)
        
        print("[Phase 1] Building jNNG with LMD distance...")
        D = self._compute_distance_matrix(X)
        n = len(X)
        median_dist = np.median(D[np.triu_indices(n, k=1)])
        D_scaled = D / (median_dist + 1e-12)
        
        jnng_sparse, geo_dist, jnng_adj = self._build_jnng(D_scaled)
        
        # Menggunakan Local Scaling di sini
        rho_gjnn = self._gaussian_kernel_density(geo_dist)
        rho_tilde = self._two_round_sigmoid_correction(rho_gjnn)
        
        print("[Phase 2] Running Bombing process (Automatic K detection)...")
        K_before, labels_bombing = self._run_bombing_phase(X, D_scaled, geo_dist, rho_tilde, jnng_adj)
        print(f"   -> Detected K = {K_before}")
        
        self.labels_ = labels_bombing
        self.centers_ = self._compute_initial_centroids(X, labels_bombing)
        self.K_auto_ = K_before
        
        # print("[Phase 3] Computing density-excluded core centroids...")
        # self.centers_ = self._compute_initial_centroids(X, labels_refined)
        
        print("[Phase 4-6] EW-LMD Iterative Refinement Loop...")
        prev_labels = self.labels_.copy()
        best_silhouette = -1.0
        best_labels = self.labels_.copy()
        restart_count = 0
        
        for step in range(self.max_iter_refine):
            current_alpha = 0.5 if step < 2 else self.alpha_reg
            orig_alpha = self.alpha_reg
            self.alpha_reg = current_alpha
            
            self._update_nominal_ordinal_phi(X, self.labels_)
            self._update_numeric_phi(X, self.labels_)
            self.alpha_reg = orig_alpha
            
            self.labels_ = self._assign_refinement(X, jnng_adj)
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
            if step >= 2:
                ari_internal = adjusted_rand_score(prev_labels, self.labels_)
                
                # === PERBAIKAN: TRAP DETECTION via Silhouette ===
                sample_idx = np.random.choice(len(X), min(1000, len(X)), replace=False) if len(X) > 1000 else np.arange(len(X))
                D_sample = np.zeros((len(sample_idx), len(sample_idx)))
                for i, idx_i in enumerate(sample_idx):
                    for j, idx_j in enumerate(sample_idx):
                        # Menggunakan fungsi distance yang sudah ada
                        D_sample[i,j] = self.distance(X.iloc[idx_i], X.iloc[idx_j])
                
                try:
                    current_sil = silhouette_score(D_sample, self.labels_[sample_idx], metric='precomputed')
                except ValueError:
                    current_sil = 0.0
                    
                if current_sil > best_silhouette:
                    best_silhouette = current_sil
                    best_labels = self.labels_.copy()
                    
                print(f"   Iter {step+1}: Internal ARI = {ari_internal:.4f} | Silhouette = {current_sil:.4f}")
                
                if ari_internal > self.conv_threshold:
                    if current_sil < 0.1 and restart_count < 2:
                        print(f"   -> [Trap Detected] Silhouette < 0.1. Perturbing labels to escape local optimum...")
                        noise_mask = np.random.rand(len(X)) < 0.2
                        self.labels_[noise_mask] = self._rng.integers(0, self.K_auto_, size=np.sum(noise_mask))
                        self.centers_ = self._update_centers_refinement(X, self.labels_)
                        restart_count += 1
                        prev_labels = self.labels_.copy()
                        continue
                    else:
                        print("   -> Convergence reached.")
                        break
                        
            prev_labels = self.labels_.copy()

        if best_silhouette > current_sil:
            if self.verbose: print(f"   -> [Recovery] Restoring best labels from Iter with Silhouette = {best_silhouette:.4f}")
            self.labels_ = best_labels
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
        return self

### V7 Bombing + LMD Algorithm without Density Excluded Init

In [ ]:
class LMDBombingHybrid_V3:
    def __init__(self, attr_types: dict, ordinal_orders: dict = None,
                 max_iter_refine: int = 20, random_state: int = 42, verbose: bool = True):
        
        self.attr_types = attr_types.copy()
        self.ordinal_orders = (ordinal_orders or {}).copy()
        self.dis_const = 0.3
        self.eta_cat = 20.0
        self.eta_num = 20.0 
        self.alpha_reg = 0.8
        self.tau_weight = 1e-3
        self.n_bins_numeric = 10
        self.max_pairs_per_cluster = 200
        self.max_iter_refine = max_iter_refine
        self.conv_threshold = 0.98
        self._rng = np.random.default_rng(random_state)
        self.verbose = verbose
        
        # Bombing Parameters
        self.k1, self.k2 = 2, 10
        self.dc, self.tau, self.T1, self.T2 = 0.15, 0.6, 0.9, 0.2
        
        # self.attr_weights = {}
        self.nominal_dist, self.ordinal_adj_dist = {}, {}
        self.num_min, self.num_max = {}, {}
        self.num_bin_psi = {}
        self.nominal_phi, self.ordinal_phi, self.numeric_phi = {}, {}, {}
        
        self.labels_ = None
        self.centers_ = None
        self.K_auto_ = None
        self._reachable_indices = {} 

    def _compute_structure(self, X: pd.DataFrame) -> dict:
        counts = {"Numeric": 0, "Ordinal": 0, "Nominal": 0}
        for t in self.attr_types.values():
            counts["Nominal" if t not in ["Numeric", "Ordinal"] else t] += 1
        max_nom = max([X[c].nunique() for c, t in self.attr_types.items() if t == "Nominal"] or [0])
        return {
            'n_rows': len(X), 'n_cols': len(X.columns),
            'n_numeric': counts['Numeric'], 'n_ordinal': counts['Ordinal'], 'n_nominal': counts['Nominal'],
            'max_nominal_card': max_nom,
            'is_wide': len(X.columns) > 20,
            'is_numeric_heavy': counts['Numeric'] > counts['Nominal'],
            'is_nominal_heavy': counts['Nominal'] > counts['Numeric']
        }
    # --- Distance Helpers ---
    def _init_distances(self, X: pd.DataFrame):
        for col in X.columns:
            t = self.attr_types[col]
            if t == "Nominal":
                vals = list(pd.unique(X[col]))
                self.nominal_dist[col] = {(a,b): self.dis_const for a in vals for b in vals if a!=b}
                self.nominal_phi[col] = {(a,b): 0.0 for a in vals for b in vals if a!=b}
            elif t == "Ordinal":
                order = list(self.ordinal_orders[col])
                self.ordinal_adj_dist[col] = {order[i]: self.dis_const for i in range(len(order)-1)}
                self.ordinal_phi[col] = {order[i]: 0.0 for i in range(len(order)-1)}
            elif t == "Numeric":
                v = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())
                self.num_min[col], self.num_max[col] = float(v.min()), float(v.max())
                self.num_bin_psi[col] = np.zeros(self.n_bins_numeric, dtype=float)
                self.numeric_phi[col] = np.zeros(self.n_bins_numeric, dtype=float)

    def _dist_attr(self, col: str, a: Any, b: Any) -> float:
        t = self.attr_types[col]
        if t == "Nominal":
            if a == b: return 0.0
            mp = self.nominal_dist[col]
            return mp.get((a,b), mp.get((b,a), self.dis_const))
        elif t == "Ordinal":
            if a == b: return 0.0
            order = list(self.ordinal_orders[col])
            try: ia, ib = order.index(a), order.index(b)
            except ValueError: return 0.0
            if ia > ib: ia, ib = ib, ia
            return sum(self.ordinal_adj_dist[col].get(order[i], self.dis_const) for i in range(ia, ib))
        elif t == "Numeric":
            vmin, vmax = self.num_min[col], self.num_max[col]
            if vmax <= vmin: return 0.0
            try: va, vb = float(a), float(b)
            except Exception: va, vb = (vmin+vmax)/2.0, (vmin+vmax)/2.0
            
            z_a = np.clip((va - vmin) / (vmax - vmin), 0.0, 1.0)
            z_b = np.clip((vb - vmin) / (vmax - vmin), 0.0, 1.0)
            
            # PERBAIKAN MURNI BOMBING: Absolute Difference (BUKAN Squared)
            # Ini menjaga konektivitas manifold jNNG agar tidak terputus (sparse)
            delta = abs(z_a - z_b)
            
            psi_vec = self.num_bin_psi[col]
            if psi_vec is None or len(psi_vec) == 0: return delta
            bidx = int(np.clip(delta * self.n_bins_numeric, 0, self.n_bins_numeric - 1))
            scale = max(0.1, 1.0 + psi_vec[bidx])
            return delta * scale
        return 0.0

    def distance(self, x: pd.Series, y: pd.Series) -> float:
        d = 0.0
        for col in y.index:
            d += self._dist_attr(col, x[col], y[col])
        return float(d)

    def _compute_distance_matrix(self, X: pd.DataFrame) -> np.ndarray:
        n = len(X)
        D = np.zeros((n, n))
        cols = list(X.columns)
        for i in range(n):
            for j in range(i+1, n):
                d = sum(self._dist_attr(col, X.iloc[i][col], X.iloc[j][col]) for col in cols)
                D[i, j] = D[j, i] = d
        return D

    # --- Bombing Core Mechanics ---
    # --- Bombing Core Mechanics ---
    def _build_jnng(self, D_scaled: np.ndarray):
        n = len(D_scaled)
        
        k1_idx = np.argpartition(D_scaled, self.k1 + 1, axis=1)[:, 1:self.k1 + 1] # +1 untuk skip diri sendiri
        k1_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k1_idx[i]:
                k1_adj[i, j] = k1_adj[j, i] = True
                
        # --- k2 (MUTUAL k-NN FIX) ---
        # Ambil k2+1, lalu slice [:, 1:] untuk membuang jarak ke diri sendiri (yang bernilai 0)
        k2_idx = np.argpartition(D_scaled, self.k2 + 1, axis=1)[:, 1:self.k2 + 1] 
        k2_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k2_idx[i]:
                # Cek mutualitas yang sebenarnya (j != i sudah dijamin oleh slice [1:])
                if i in k2_idx[j]: 
                    k2_adj[i, j] = k2_adj[j, i] = True
                
        jnng_adj = k1_adj | k2_adj
        jnng_sparse = csr_matrix(jnng_adj.astype(float) * D_scaled)
        
        # Geodesic distance
        geo_dist = shortest_path(jnng_sparse, method='D', directed=False)
        geo_dist[np.isinf(geo_dist)] = geo_dist[np.isfinite(geo_dist)].max() * 1.5
        
        return jnng_sparse, geo_dist, csr_matrix(jnng_adj.astype(float))

    def _gaussian_kernel_density_old(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        PERBAIKAN MURNI BOMBING: Local Scaling (Zelnik-Perona style).
        Mencegah density flattening dengan membuat bandwidth adaptif terhadap kepadatan lokal.
        """
        n = geo_dist.shape[0]
        k_local = min(7, n - 1)
        
        sigmas = np.partition(geo_dist, k_local, axis=1)[:, k_local]
        sigmas[sigmas == 0] = 1e-12
        
        sigma_matrix = np.outer(sigmas, sigmas)
        rho = np.sum(np.exp(-geo_dist**2 / (2 * sigma_matrix + 1e-12)), axis=1)
        return rho

    def _gaussian_kernel_density(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        Mengikuti Eq. 9 dari Paper Fei et al. (2025).
        Menggunakan varians tetap (dc) untuk menjaga validitas Two-Round Sigmoid.
        """
        # self.dc = 0.15 (atau sesuai parameter default paper)
        rho = np.sum(np.exp(-(geo_dist ** 2) / (2 * (self.dc ** 2))), axis=1)
        return rho

    def _two_round_sigmoid_correction(self, rho_gjnn: np.ndarray) -> np.ndarray:
        rho_max = rho_gjnn.max()
        slope, delta = 10/rho_max, rho_max/2
        rho_tilde = 1 / (1 + np.exp(-slope * (rho_gjnn - delta)))
        mask = rho_tilde < self.T1
        if np.any(mask):
            rho_max_below = rho_gjnn[mask].max()
            s2, d2 = 10/rho_max_below, rho_max_below/2
            rho_tilde[mask] = 1 / (1 + np.exp(-s2 * (rho_tilde[mask] - d2)))
        return rho_tilde

    def _local_neighbor_density(self, D_scaled: np.ndarray, r: float, valid_mask: np.ndarray) -> np.ndarray:
        """Menghitung rho_r (Eq. 16) hanya berdasarkan titik-titik yang valid (belum di-cluster)."""
        n = len(D_scaled)
        rho_r = np.zeros(n)
        for i in range(n):
            if not valid_mask[i]: continue
            # Hitung tetangga dalam radius r dari titik i
            neighbors_in_r = np.sum((D_scaled[i] <= r) & valid_mask)
            rho_r[i] = neighbors_in_r / self.k2
        return rho_r

    def _bfs_generalized_cluster(self, adj_mat: csr_matrix, core_idx: int, P: np.ndarray, visited_mask: np.ndarray) -> Tuple[List[int], List[int]]:
        R_t, B_t = [], []
        queue = [core_idx]
        visited_mask[core_idx] = True
        R_t.append(core_idx)
        adj_list = adj_mat.tolil().rows
        first_boundary = set()
        
        while queue:
            u = queue.pop(0)
            for v in adj_list[u]:
                if visited_mask[v]: continue
                if P[v] >= self.tau:
                    visited_mask[v] = True
                    R_t.append(v)
                    queue.append(v)
                else:
                    first_boundary.add(v)
                    
        second_boundary = set()
        for v in first_boundary:
            for w in adj_list[v]:
                if not visited_mask[w] and w not in first_boundary and w not in R_t:
                    second_boundary.add(w)
        B_t = list(first_boundary | second_boundary)
        return R_t, B_t

    def _degree_of_adjacency(self, adj_mat: csr_matrix, R_t: List[int], B_t: List[int], existing_clusters: dict) -> Tuple[float, int]:
        if not existing_clusters: return 0.0, -1
        C_star = len(R_t) + len(B_t)
        if C_star == 0: return 0.0, -1
        adj_list = adj_mat.tolil().rows
        max_nap, best_s = 0, -1
        for s, (_, B_s) in existing_clusters.items():
            nap = sum(1 for u in R_t for v in adj_list[u] if v in B_s)
            da = nap / C_star
            if da > max_nap: max_nap, best_s = da, s
        return max_nap, best_s

    def _run_bombing_phase(self, X: pd.DataFrame, D_scaled: np.ndarray, geo_dist: np.ndarray, rho_tilde: np.ndarray, jnng_adj: csr_matrix):
        n = len(X)
        labels = np.full(n, -1, dtype=int)
        visited = np.zeros(n, dtype=bool)
        existing_clusters = {}
        t = 0
        self._reachable_indices = {}
        
        # Copy rho_tilde agar bisa dimanipulasi per iterasi
        current_rho_tilde = rho_tilde.copy()
        
        while not np.all(visited):
            cand = ~visited
            if not np.any(cand): break
            
            # 1. Tentukan Core Point dari titik yang BELUM dikunjungi
            core_idx = np.argmax(current_rho_tilde * cand)
            if current_rho_tilde[core_idx] < 1e-6: break

            # 2. Hitung Ulang Local Neighbor Density (rho_r) berdasarkan sisa titik
            # Paper: r adalah jarak ke k-th nearest neighbor dari core point
            unvisited_indices = np.where(cand)[0]
            if len(unvisited_indices) <= self.k2:
                r = np.max(D_scaled[core_idx, unvisited_indices])
            else:
                r = np.partition(D_scaled[core_idx, unvisited_indices], self.k2)[self.k2]
                
            rho_r = self._local_neighbor_density(D_scaled, r, cand)

            # 3. Mixture Density (Eq. 17)
            P = current_rho_tilde + rho_r
            P_min, P_max = P.min(), P.max()
            if P_max > P_min: 
                P = (P - P_min) / (P_max - P_min)

            # 4. BFS dengan TAU TETAP (0.6) - Ini adalah klaim utama paper!
            R_t, B_t = self._bfs_generalized_cluster(jnng_adj, core_idx, P, visited.copy())
            
            # Tandai titik yang di-cluster dan boundary sebagai visited (dihapus dari X)
            idxs = R_t + B_t
            for idx in idxs: 
                visited[idx] = True

            # 5. Hitung Degree of Adjacency (DA)
            da, best_s = self._degree_of_adjacency(jnng_adj, R_t, B_t, existing_clusters)

            # 6. Aturan Penggabungan (Generalized Cluster)
            if da >= self.T2 and best_s != -1:
                labels[idxs] = best_s
            else:
                labels[idxs] = t
                existing_clusters[t] = (R_t, B_t)
                self._reachable_indices[t] = R_t
                t += 1

            # 7. UPDATE DINAMIS: Nol-kan densitas titik yang sudah di-cluster
            # Agar iterasi berikutnya hanya mencari core point dari sisa dataset
            current_rho_tilde[visited] = 0.0
            
            # Opsional: Recalculate Sigmoid Round 1 & 2 untuk sisa titik jika diperlukan
            # (Paper melakukan recalculate rho_max dari sisa titik)
            rho_max_rem = current_rho_tilde.max()
            if rho_max_rem > 1e-6:
                slope, delta = 10 / rho_max_rem, rho_max_rem / 2
                # Terapkan ulang sigmoid ke titik yang belum visited
                current_rho_tilde[~visited] = 1 / (1 + np.exp(-slope * (current_rho_tilde[~visited] - delta)))

        # Final Clean-up (Assign unvisited/boundary ke nearest visited point)
        unvisited = np.where(labels == -1)[0]
        if len(unvisited) > 0:
            visited_pts = np.where(labels != -1)[0]
            if len(visited_pts) > 0:
                for u in unvisited:
                    labels[u] = labels[visited_pts[np.argmin(D_scaled[u, visited_pts])]]
            else:
                labels[:n//2] = 0
                labels[n//2:] = 1

        self.K_auto_ = len(np.unique(labels[labels != -1]))
        return self.K_auto_, labels

    def _compute_initial_centroids(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        """
        [ABLATION VAR-2] Tanpa Density-Excluded Init.
        Inisialisasi sentroid menggunakan SEMUA titik dalam klaster Bombing 
        (termasuk boundary/noise), bukan hanya reachable (core) points.
        """
        centers = []
        unique_labels = np.unique(labels)
        
        for lbl in unique_labels:
            # PERUBAHAN UTAMA: Mengabaikan self._reachable_indices.
            # Langsung ambil semua titik yang memiliki label lbl (termasuk boundary & noise).
            cl = X[labels == lbl]
            
            c = {}
            for col in X.columns:
                if self.attr_types[col] in ('Nominal', 'Ordinal'):
                    c[col] = _mode(cl[col])
                else:
                    c[col] = cl[col].mean()
            centers.append(c)
            
        return pd.DataFrame(centers, columns=X.columns)

    # --- EW-LMD Refinement Helpers ---
    def _update_nominal_ordinal_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] not in ('Nominal', 'Ordinal'): continue
            vals = list(pd.unique(X[col]))
            if len(vals) < 2: continue
            P = {}
            for a in vals:
                for b in vals:
                    if a == b: continue
                    val_sum = sum(((X.iloc[np.where(labels==c)[0]][col]==a).sum()/(len(X[labels==c])+eps)) * 
                                  ((X.iloc[np.where(labels==c)[0]][col]==b).sum()/(len(X[labels==c])+eps)) 
                                  for c in range(k) if len(X[labels==c]) > 0)
                    P[(a,b)] = val_sum
            if not P: continue
            P_vals = list(P.values())
            P_max, P_min = max(P_vals), min(P_vals)
            if abs(P_max-P_min) < eps: continue
            
            phi_raw = {key: (-P_max/self.eta_cat if v==P_max else (1-P_min)/self.eta_cat if v==P_min else 0.0) for key, v in P.items()}
            phi_old = self.nominal_phi[col] if self.attr_types[col]=='Nominal' else self.ordinal_phi[col]
            phi_new = {key: np.clip(self.alpha_reg*phi_old.get(key,0.0) + (1-self.alpha_reg)*phi_raw[key], -1, 1) for key in phi_raw}
            
            if self.attr_types[col]=='Nominal':
                self.nominal_phi[col] = phi_new
                self.nominal_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}
            else:
                self.ordinal_phi[col] = phi_new
                self.ordinal_adj_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}

    def _update_numeric_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        B = self.n_bins_numeric
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] != 'Numeric': continue
            if self.num_min[col] >= self.num_max[col]: continue
            z_full = np.clip((pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy() - self.num_min[col]) / (self.num_max[col] - self.num_min[col]), 0.0, 1.0)
            P_sum = np.zeros(B)
            
            for c in range(k):
                idx = np.where(labels==c)[0]
                m = len(idx)
                if m < 2: continue
                if m > self.max_pairs_per_cluster:
                    idx_sample = self._rng.choice(idx, size=self.max_pairs_per_cluster, replace=False)
                else:
                    idx_sample = idx
                m_s = len(idx_sample)
                if m_s < 2: continue
                
                bin_counts = np.zeros(B, dtype=float)
                for i in range(m_s):
                    for j in range(i + 1, m_s):
                        delta = abs(z_full[idx_sample[i]] - z_full[idx_sample[j]])
                        bidx = int(np.clip(delta * B, 0, B - 1))
                        bin_counts[bidx] += 1.0
                total_pairs = bin_counts.sum()
                if total_pairs <= 0: continue
                probs = bin_counts / (total_pairs + eps)
                P_sum += probs * probs

            if np.all(P_sum <= eps): continue
            P_max, P_min = P_sum.max(), P_sum.min()
            if abs(P_max - P_min) < eps: continue

            phi_raw_vec = np.zeros(B, dtype=float)
            for bidx in range(B):
                val = P_sum[bidx]
                if val == P_max: phi = -P_max / self.eta_num
                elif val == P_min: phi = (1.0 - P_min) / self.eta_num
                else: phi = 0.0
                phi_raw_vec[bidx] = phi

            phi_old_vec = self.numeric_phi[col]
            if phi_old_vec.shape[0] != B: phi_old_vec = np.zeros(B, dtype=float)
            phi_new_vec = np.clip(self.alpha_reg * phi_old_vec + (1.0 - self.alpha_reg) * phi_raw_vec, -1.0, 1.0)
            self.numeric_phi[col] = phi_new_vec
            self.num_bin_psi[col] = phi_new_vec.copy()

    def _assign_refinement(self, X: pd.DataFrame, jnng_adj: csr_matrix) -> np.ndarray:
        """
        REFINEMENT DENGAN KENDALA KONEKTIVITAS jNNG (FIX CELAH #1)
        Titik hanya boleh pindah klaster jika memiliki edge jNNG dengan anggota klaster target.
        """
        n = len(X)
        labels = np.empty(n, dtype=int)
        
        # 1. Pre-compute anggota klaster saat ini untuk lookup O(1)
        cluster_members = {c: set(np.where(self.labels_ == c)[0]) for c in range(self.K_auto_)}
        
        # 2. Konversi adjacency matrix ke list of sets untuk akses tetangga yang cepat
        adj_list = jnng_adj.tolil().rows

        for i in range(n):
            # Hitung jarak EW-LMD ke semua sentroid
            dists = [
                sum(self._dist_attr(col, X.iloc[i][col], self.centers_.iloc[c][col]) 
                    for col in X.columns) 
                for c in range(self.K_auto_)
            ]
            best_c = int(np.argmin(dists)) # Klaster terdekat secara geometris
            
            # Cek konektivitas topologi jNNG
            neighbors_i = set(adj_list[i])
            
            # Cari klaster mana saja yang memiliki anggota terhubung dengan titik i
            connected_clusters = [
                c for c, members in cluster_members.items() 
                if neighbors_i.intersection(members)
            ]
            
            # 3. Keputusan Alokasi
            if connected_clusters:
                # Jika ada koneksi, pilih klaster yang TERHUBUNG yang jaraknya paling minimum
                labels[i] = min(connected_clusters, key=lambda c: dists[c])
            else:
                # Fallback: Titik terisolasi (tidak ada edge ke klaster manapun)
                # Gunakan jarak geometris murni agar tidak error
                labels[i] = best_c

        return labels

    def _update_centers_refinement(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        for c in range(self.K_auto_):
            cl = X[labels==c]
            if cl.empty: 
                centers.append(self.centers_.iloc[c].to_dict())
                continue
            c_dict = {col: _mode(cl[col]) if self.attr_types[col] in ('Nominal','Ordinal') else cl[col].mean() for col in X.columns}
            centers.append(c_dict)
        return pd.DataFrame(centers)

    def fit(self, df: pd.DataFrame):
        X = df.drop(columns=["Label"], errors="ignore").copy()
        for col in X.columns:
            if self.attr_types[col] == "Numeric":
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())

        # 1. HYBRID PARAMETER TUNING
        structure = self._compute_structure(X)
        params = hybrid_selection_conservative(X, self.attr_types, structure)
        self.dis_const = params['dist_cons']
        self.eta_cat = self.eta_num = params['eta']
        self.n_bins_numeric = params['n_bins_numeric']  # ← PERBAIKAN
        self.auto_params_log = params.copy()
        if self.verbose: print(f"[AUTONOMOUS] Hybrid-tuned: {self.auto_params_log}")
    
        print("[Phase 0] Computing entropy weights & initializing distances...")
        self._init_distances(X)
        
        print("[Phase 1] Building jNNG with EW-LMD distance...")
        D = self._compute_distance_matrix(X)
        n = len(X)
        median_dist = np.median(D[np.triu_indices(n, k=1)])
        D_scaled = D / (median_dist + 1e-12)
        
        jnng_sparse, geo_dist, jnng_adj = self._build_jnng(D_scaled)
        
        # Menggunakan Local Scaling di sini
        rho_gjnn = self._gaussian_kernel_density(geo_dist)
        rho_tilde = self._two_round_sigmoid_correction(rho_gjnn)
        
        print("[Phase 2] Running Bombing process (Automatic K detection)...")
        K_before, labels_bombing = self._run_bombing_phase(X, D_scaled, geo_dist, rho_tilde, jnng_adj)
        print(f"   -> Detected K = {K_before}")
        
        self.labels_ = labels_bombing
        self.centers_ = self._compute_initial_centroids(X, labels_bombing)
        
        print("[Phase 2.5] Evaluating structural coherence for conditional merging...")
        metrics = compute_merge_metrics(labels_bombing, jnng_adj, rho_tilde)
        trigger = should_merge_parameter_free(metrics, T2=self.T2)
        
        if trigger:
            print("   -> Trigger activated. Initializing Post-Hoc Merger...")
            merger = PostHocMerger(self, X, D_scaled)
            merge_results = merger.smart_merge_stable(min_K=2, max_merge_ratio=1.0)
            labels_refined = merge_results['labels_merged']
            K_after = merge_results['K_final']
            print(f"   -> Merge Result: K {merge_results['K_before']} → {K_after}")
        else:
            print("   -> Structure coherent. Skipping merging phase.")
            labels_refined = labels_bombing
            K_after = K_before
            
        self.labels_ = labels_refined
        self.K_auto_ = K_after
        
        print("[Phase 3] Computing core centroids...")
        if trigger:
            for lbl in np.unique(labels_refined):
                orig_lbls = np.unique(labels_bombing[labels_refined == lbl])
                r_idx = []
                for ol in orig_lbls:
                    r_idx.extend(self._reachable_indices.get(ol, []))
                self._reachable_indices[lbl] = r_idx
                
        self.centers_ = self._compute_initial_centroids(X, labels_refined)
        
        print("[Phase 4-6] EW-LMD Iterative Refinement Loop...")
        prev_labels = self.labels_.copy()
        best_silhouette = -1.0
        best_labels = self.labels_.copy()
        restart_count = 0
        
        for step in range(self.max_iter_refine):
            current_alpha = 0.5 if step < 2 else self.alpha_reg
            orig_alpha = self.alpha_reg
            self.alpha_reg = current_alpha
            
            self._update_nominal_ordinal_phi(X, self.labels_)
            self._update_numeric_phi(X, self.labels_)
            self.alpha_reg = orig_alpha
            
            self.labels_ = self._assign_refinement(X, jnng_adj)
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
            if step >= 2:
                ari_internal = adjusted_rand_score(prev_labels, self.labels_)
                
                # === PERBAIKAN: TRAP DETECTION via Silhouette ===
                sample_idx = np.random.choice(len(X), min(1000, len(X)), replace=False) if len(X) > 1000 else np.arange(len(X))
                D_sample = np.zeros((len(sample_idx), len(sample_idx)))
                for i, idx_i in enumerate(sample_idx):
                    for j, idx_j in enumerate(sample_idx):
                        # Menggunakan fungsi distance yang sudah ada
                        D_sample[i,j] = self.distance(X.iloc[idx_i], X.iloc[idx_j])
                
                try:
                    current_sil = silhouette_score(D_sample, self.labels_[sample_idx], metric='precomputed')
                except ValueError:
                    current_sil = 0.0
                    
                if current_sil > best_silhouette:
                    best_silhouette = current_sil
                    best_labels = self.labels_.copy()
                    
                print(f"   Iter {step+1}: Internal ARI = {ari_internal:.4f} | Silhouette = {current_sil:.4f}")
                
                if ari_internal > self.conv_threshold:
                    if current_sil < 0.1 and restart_count < 2:
                        print(f"   -> [Trap Detected] Silhouette < 0.1. Perturbing labels to escape local optimum...")
                        noise_mask = np.random.rand(len(X)) < 0.2
                        self.labels_[noise_mask] = self._rng.integers(0, self.K_auto_, size=np.sum(noise_mask))
                        self.centers_ = self._update_centers_refinement(X, self.labels_)
                        restart_count += 1
                        prev_labels = self.labels_.copy()
                        continue
                    else:
                        print("   -> Convergence reached.")
                        break
                        
            prev_labels = self.labels_.copy()

        if best_silhouette > current_sil:
            if self.verbose: print(f"   -> [Recovery] Restoring best labels from Iter with Silhouette = {best_silhouette:.4f}")
            self.labels_ = best_labels
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
        return self

### V8 Bombing + LMD Algorithm without DE + PM

In [ ]:
class LMDBombingHybrid_V4:
    def __init__(self, attr_types: dict, ordinal_orders: dict = None,
                 max_iter_refine: int = 20, random_state: int = 42, verbose: bool = True):
        
        self.attr_types = attr_types.copy()
        self.ordinal_orders = (ordinal_orders or {}).copy()
        self.dis_const = 0.3
        self.eta_cat = 20.0
        self.eta_num = 20.0 
        self.alpha_reg = 0.8
        self.tau_weight = 1e-3
        self.n_bins_numeric = 10
        self.max_pairs_per_cluster = 200
        self.max_iter_refine = max_iter_refine
        self.conv_threshold = 0.98
        self._rng = np.random.default_rng(random_state)
        self.verbose = verbose
        
        # Bombing Parameters
        self.k1, self.k2 = 2, 10
        self.dc, self.tau, self.T1, self.T2 = 0.15, 0.6, 0.9, 0.2
        
        # self.attr_weights = {}
        self.nominal_dist, self.ordinal_adj_dist = {}, {}
        self.num_min, self.num_max = {}, {}
        self.num_bin_psi = {}
        self.nominal_phi, self.ordinal_phi, self.numeric_phi = {}, {}, {}
        
        self.labels_ = None
        self.centers_ = None
        self.K_auto_ = None
        self._reachable_indices = {} 

    def _compute_structure(self, X: pd.DataFrame) -> dict:
        counts = {"Numeric": 0, "Ordinal": 0, "Nominal": 0}
        for t in self.attr_types.values():
            counts["Nominal" if t not in ["Numeric", "Ordinal"] else t] += 1
        max_nom = max([X[c].nunique() for c, t in self.attr_types.items() if t == "Nominal"] or [0])
        return {
            'n_rows': len(X), 'n_cols': len(X.columns),
            'n_numeric': counts['Numeric'], 'n_ordinal': counts['Ordinal'], 'n_nominal': counts['Nominal'],
            'max_nominal_card': max_nom,
            'is_wide': len(X.columns) > 20,
            'is_numeric_heavy': counts['Numeric'] > counts['Nominal'],
            'is_nominal_heavy': counts['Nominal'] > counts['Numeric']
        }
    # --- Distance Helpers ---
    def _init_distances(self, X: pd.DataFrame):
        for col in X.columns:
            t = self.attr_types[col]
            if t == "Nominal":
                vals = list(pd.unique(X[col]))
                self.nominal_dist[col] = {(a,b): self.dis_const for a in vals for b in vals if a!=b}
                self.nominal_phi[col] = {(a,b): 0.0 for a in vals for b in vals if a!=b}
            elif t == "Ordinal":
                order = list(self.ordinal_orders[col])
                self.ordinal_adj_dist[col] = {order[i]: self.dis_const for i in range(len(order)-1)}
                self.ordinal_phi[col] = {order[i]: 0.0 for i in range(len(order)-1)}
            elif t == "Numeric":
                v = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())
                self.num_min[col], self.num_max[col] = float(v.min()), float(v.max())
                self.num_bin_psi[col] = np.zeros(self.n_bins_numeric, dtype=float)
                self.numeric_phi[col] = np.zeros(self.n_bins_numeric, dtype=float)

    def _dist_attr(self, col: str, a: Any, b: Any) -> float:
        t = self.attr_types[col]
        if t == "Nominal":
            if a == b: return 0.0
            mp = self.nominal_dist[col]
            return mp.get((a,b), mp.get((b,a), self.dis_const))
        elif t == "Ordinal":
            if a == b: return 0.0
            order = list(self.ordinal_orders[col])
            try: ia, ib = order.index(a), order.index(b)
            except ValueError: return 0.0
            if ia > ib: ia, ib = ib, ia
            return sum(self.ordinal_adj_dist[col].get(order[i], self.dis_const) for i in range(ia, ib))
        elif t == "Numeric":
            vmin, vmax = self.num_min[col], self.num_max[col]
            if vmax <= vmin: return 0.0
            try: va, vb = float(a), float(b)
            except Exception: va, vb = (vmin+vmax)/2.0, (vmin+vmax)/2.0
            
            z_a = np.clip((va - vmin) / (vmax - vmin), 0.0, 1.0)
            z_b = np.clip((vb - vmin) / (vmax - vmin), 0.0, 1.0)
            
            # PERBAIKAN MURNI BOMBING: Absolute Difference (BUKAN Squared)
            # Ini menjaga konektivitas manifold jNNG agar tidak terputus (sparse)
            delta = abs(z_a - z_b)
            
            psi_vec = self.num_bin_psi[col]
            if psi_vec is None or len(psi_vec) == 0: return delta
            bidx = int(np.clip(delta * self.n_bins_numeric, 0, self.n_bins_numeric - 1))
            scale = max(0.1, 1.0 + psi_vec[bidx])
            return delta * scale
        return 0.0

    def distance(self, x: pd.Series, y: pd.Series) -> float:
        d = 0.0
        for col in y.index:
            d += self._dist_attr(col, x[col], y[col])
        return float(d)

    def _compute_distance_matrix(self, X: pd.DataFrame) -> np.ndarray:
        n = len(X)
        D = np.zeros((n, n))
        cols = list(X.columns)
        for i in range(n):
            for j in range(i+1, n):
                d = sum(self._dist_attr(col, X.iloc[i][col], X.iloc[j][col]) for col in cols)
                D[i, j] = D[j, i] = d
        return D

    # --- Bombing Core Mechanics ---
    def _build_jnng(self, D_scaled: np.ndarray):
        n = len(D_scaled)
        
        k1_idx = np.argpartition(D_scaled, self.k1 + 1, axis=1)[:, 1:self.k1 + 1] # +1 untuk skip diri sendiri
        k1_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k1_idx[i]:
                k1_adj[i, j] = k1_adj[j, i] = True
                
        # --- k2 (MUTUAL k-NN FIX) ---
        # Ambil k2+1, lalu slice [:, 1:] untuk membuang jarak ke diri sendiri (yang bernilai 0)
        k2_idx = np.argpartition(D_scaled, self.k2 + 1, axis=1)[:, 1:self.k2 + 1] 
        k2_adj = np.zeros((n, n), dtype=bool)
        for i in range(n):
            for j in k2_idx[i]:
                # Cek mutualitas yang sebenarnya (j != i sudah dijamin oleh slice [1:])
                if i in k2_idx[j]: 
                    k2_adj[i, j] = k2_adj[j, i] = True
                
        jnng_adj = k1_adj | k2_adj
        jnng_sparse = csr_matrix(jnng_adj.astype(float) * D_scaled)
        
        # Geodesic distance
        geo_dist = shortest_path(jnng_sparse, method='D', directed=False)
        geo_dist[np.isinf(geo_dist)] = geo_dist[np.isfinite(geo_dist)].max() * 1.5
        
        return jnng_sparse, geo_dist, csr_matrix(jnng_adj.astype(float))

    def _gaussian_kernel_density(self, geo_dist: np.ndarray) -> np.ndarray:
        """
        Mengikuti Eq. 9 dari Paper Fei et al. (2025).
        Menggunakan varians tetap (dc) untuk menjaga validitas Two-Round Sigmoid.
        """
        # self.dc = 0.15 (atau sesuai parameter default paper)
        rho = np.sum(np.exp(-(geo_dist ** 2) / (2 * (self.dc ** 2))), axis=1)
        return rho

    def _two_round_sigmoid_correction(self, rho_gjnn: np.ndarray) -> np.ndarray:
        rho_max = rho_gjnn.max()
        slope, delta = 10/rho_max, rho_max/2
        rho_tilde = 1 / (1 + np.exp(-slope * (rho_gjnn - delta)))
        mask = rho_tilde < self.T1
        if np.any(mask):
            rho_max_below = rho_gjnn[mask].max()
            s2, d2 = 10/rho_max_below, rho_max_below/2
            rho_tilde[mask] = 1 / (1 + np.exp(-s2 * (rho_tilde[mask] - d2)))
        return rho_tilde

    def _local_neighbor_density(self, D_scaled: np.ndarray, r: float, valid_mask: np.ndarray) -> np.ndarray:
        """Menghitung rho_r (Eq. 16) hanya berdasarkan titik-titik yang valid (belum di-cluster)."""
        n = len(D_scaled)
        rho_r = np.zeros(n)
        for i in range(n):
            if not valid_mask[i]: continue
            # Hitung tetangga dalam radius r dari titik i
            neighbors_in_r = np.sum((D_scaled[i] <= r) & valid_mask)
            rho_r[i] = neighbors_in_r / self.k2
        return rho_r

    def _bfs_generalized_cluster(self, adj_mat: csr_matrix, core_idx: int, P: np.ndarray, visited_mask: np.ndarray) -> Tuple[List[int], List[int]]:
        R_t, B_t = [], []
        queue = [core_idx]
        visited_mask[core_idx] = True
        R_t.append(core_idx)
        adj_list = adj_mat.tolil().rows
        first_boundary = set()
        
        while queue:
            u = queue.pop(0)
            for v in adj_list[u]:
                if visited_mask[v]: continue
                if P[v] >= self.tau:
                    visited_mask[v] = True
                    R_t.append(v)
                    queue.append(v)
                else:
                    first_boundary.add(v)
                    
        second_boundary = set()
        for v in first_boundary:
            for w in adj_list[v]:
                if not visited_mask[w] and w not in first_boundary and w not in R_t:
                    second_boundary.add(w)
        B_t = list(first_boundary | second_boundary)
        return R_t, B_t

    def _degree_of_adjacency(self, adj_mat: csr_matrix, R_t: List[int], B_t: List[int], existing_clusters: dict) -> Tuple[float, int]:
        if not existing_clusters: return 0.0, -1
        C_star = len(R_t) + len(B_t)
        if C_star == 0: return 0.0, -1
        adj_list = adj_mat.tolil().rows
        max_nap, best_s = 0, -1
        for s, (_, B_s) in existing_clusters.items():
            nap = sum(1 for u in R_t for v in adj_list[u] if v in B_s)
            da = nap / C_star
            if da > max_nap: max_nap, best_s = da, s
        return max_nap, best_s

    def _run_bombing_phase(self, X: pd.DataFrame, D_scaled: np.ndarray, geo_dist: np.ndarray, rho_tilde: np.ndarray, jnng_adj: csr_matrix):
        n = len(X)
        labels = np.full(n, -1, dtype=int)
        visited = np.zeros(n, dtype=bool)
        existing_clusters = {}
        t = 0
        self._reachable_indices = {}
        
        # Copy rho_tilde agar bisa dimanipulasi per iterasi
        current_rho_tilde = rho_tilde.copy()
        
        while not np.all(visited):
            cand = ~visited
            if not np.any(cand): break
            
            # 1. Tentukan Core Point dari titik yang BELUM dikunjungi
            core_idx = np.argmax(current_rho_tilde * cand)
            if current_rho_tilde[core_idx] < 1e-6: break

            # 2. Hitung Ulang Local Neighbor Density (rho_r) berdasarkan sisa titik
            # Paper: r adalah jarak ke k-th nearest neighbor dari core point
            unvisited_indices = np.where(cand)[0]
            if len(unvisited_indices) <= self.k2:
                r = np.max(D_scaled[core_idx, unvisited_indices])
            else:
                r = np.partition(D_scaled[core_idx, unvisited_indices], self.k2)[self.k2]
                
            rho_r = self._local_neighbor_density(D_scaled, r, cand)

            # 3. Mixture Density (Eq. 17)
            P = current_rho_tilde + rho_r
            P_min, P_max = P.min(), P.max()
            if P_max > P_min: 
                P = (P - P_min) / (P_max - P_min)

            # 4. BFS dengan TAU TETAP (0.6) - Ini adalah klaim utama paper!
            R_t, B_t = self._bfs_generalized_cluster(jnng_adj, core_idx, P, visited.copy())
            
            # Tandai titik yang di-cluster dan boundary sebagai visited (dihapus dari X)
            idxs = R_t + B_t
            for idx in idxs: 
                visited[idx] = True

            # 5. Hitung Degree of Adjacency (DA)
            da, best_s = self._degree_of_adjacency(jnng_adj, R_t, B_t, existing_clusters)

            # 6. Aturan Penggabungan (Generalized Cluster)
            if da >= self.T2 and best_s != -1:
                labels[idxs] = best_s
            else:
                labels[idxs] = t
                existing_clusters[t] = (R_t, B_t)
                self._reachable_indices[t] = R_t
                t += 1

            # 7. UPDATE DINAMIS: Nol-kan densitas titik yang sudah di-cluster
            # Agar iterasi berikutnya hanya mencari core point dari sisa dataset
            current_rho_tilde[visited] = 0.0
            
            # Opsional: Recalculate Sigmoid Round 1 & 2 untuk sisa titik jika diperlukan
            # (Paper melakukan recalculate rho_max dari sisa titik)
            rho_max_rem = current_rho_tilde.max()
            if rho_max_rem > 1e-6:
                slope, delta = 10 / rho_max_rem, rho_max_rem / 2
                # Terapkan ulang sigmoid ke titik yang belum visited
                current_rho_tilde[~visited] = 1 / (1 + np.exp(-slope * (current_rho_tilde[~visited] - delta)))

        # Final Clean-up (Assign unvisited/boundary ke nearest visited point)
        unvisited = np.where(labels == -1)[0]
        if len(unvisited) > 0:
            visited_pts = np.where(labels != -1)[0]
            if len(visited_pts) > 0:
                for u in unvisited:
                    labels[u] = labels[visited_pts[np.argmin(D_scaled[u, visited_pts])]]
            else:
                labels[:n//2] = 0
                labels[n//2:] = 1

        self.K_auto_ = len(np.unique(labels[labels != -1]))
        return self.K_auto_, labels

    def _compute_initial_centroids(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        """
        [ABLATION VAR-2] Tanpa Density-Excluded Init.
        Inisialisasi sentroid menggunakan SEMUA titik dalam klaster Bombing 
        (termasuk boundary/noise), bukan hanya reachable (core) points.
        """
        centers = []
        unique_labels = np.unique(labels)
        
        for lbl in unique_labels:
            # PERUBAHAN UTAMA: Mengabaikan self._reachable_indices.
            # Langsung ambil semua titik yang memiliki label lbl (termasuk boundary & noise).
            cl = X[labels == lbl]
            
            c = {}
            for col in X.columns:
                if self.attr_types[col] in ('Nominal', 'Ordinal'):
                    c[col] = _mode(cl[col])
                else:
                    c[col] = cl[col].mean()
            centers.append(c)
            
        return pd.DataFrame(centers, columns=X.columns)

    # --- EW-LMD Refinement Helpers ---
    def _update_nominal_ordinal_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] not in ('Nominal', 'Ordinal'): continue
            vals = list(pd.unique(X[col]))
            if len(vals) < 2: continue
            P = {}
            for a in vals:
                for b in vals:
                    if a == b: continue
                    val_sum = sum(((X.iloc[np.where(labels==c)[0]][col]==a).sum()/(len(X[labels==c])+eps)) * 
                                  ((X.iloc[np.where(labels==c)[0]][col]==b).sum()/(len(X[labels==c])+eps)) 
                                  for c in range(k) if len(X[labels==c]) > 0)
                    P[(a,b)] = val_sum
            if not P: continue
            P_vals = list(P.values())
            P_max, P_min = max(P_vals), min(P_vals)
            if abs(P_max-P_min) < eps: continue
            
            phi_raw = {key: (-P_max/self.eta_cat if v==P_max else (1-P_min)/self.eta_cat if v==P_min else 0.0) for key, v in P.items()}
            phi_old = self.nominal_phi[col] if self.attr_types[col]=='Nominal' else self.ordinal_phi[col]
            phi_new = {key: np.clip(self.alpha_reg*phi_old.get(key,0.0) + (1-self.alpha_reg)*phi_raw[key], -1, 1) for key in phi_raw}
            
            if self.attr_types[col]=='Nominal':
                self.nominal_phi[col] = phi_new
                self.nominal_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}
            else:
                self.ordinal_phi[col] = phi_new
                self.ordinal_adj_dist[col] = {k: self.dis_const + v for k, v in phi_new.items()}

    def _update_numeric_phi(self, X: pd.DataFrame, labels: np.ndarray):
        eps = 1e-12
        B = self.n_bins_numeric
        k = self.K_auto_
        for col in X.columns:
            if self.attr_types[col] != 'Numeric': continue
            if self.num_min[col] >= self.num_max[col]: continue
            z_full = np.clip((pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean()).to_numpy() - self.num_min[col]) / (self.num_max[col] - self.num_min[col]), 0.0, 1.0)
            P_sum = np.zeros(B)
            
            for c in range(k):
                idx = np.where(labels==c)[0]
                m = len(idx)
                if m < 2: continue
                if m > self.max_pairs_per_cluster:
                    idx_sample = self._rng.choice(idx, size=self.max_pairs_per_cluster, replace=False)
                else:
                    idx_sample = idx
                m_s = len(idx_sample)
                if m_s < 2: continue
                
                bin_counts = np.zeros(B, dtype=float)
                for i in range(m_s):
                    for j in range(i + 1, m_s):
                        delta = abs(z_full[idx_sample[i]] - z_full[idx_sample[j]])
                        bidx = int(np.clip(delta * B, 0, B - 1))
                        bin_counts[bidx] += 1.0
                total_pairs = bin_counts.sum()
                if total_pairs <= 0: continue
                probs = bin_counts / (total_pairs + eps)
                P_sum += probs * probs

            if np.all(P_sum <= eps): continue
            P_max, P_min = P_sum.max(), P_sum.min()
            if abs(P_max - P_min) < eps: continue

            phi_raw_vec = np.zeros(B, dtype=float)
            for bidx in range(B):
                val = P_sum[bidx]
                if val == P_max: phi = -P_max / self.eta_num
                elif val == P_min: phi = (1.0 - P_min) / self.eta_num
                else: phi = 0.0
                phi_raw_vec[bidx] = phi

            phi_old_vec = self.numeric_phi[col]
            if phi_old_vec.shape[0] != B: phi_old_vec = np.zeros(B, dtype=float)
            phi_new_vec = np.clip(self.alpha_reg * phi_old_vec + (1.0 - self.alpha_reg) * phi_raw_vec, -1.0, 1.0)
            self.numeric_phi[col] = phi_new_vec
            self.num_bin_psi[col] = phi_new_vec.copy()

    def _assign_refinement(self, X: pd.DataFrame, jnng_adj: csr_matrix) -> np.ndarray:
        """
        REFINEMENT DENGAN KENDALA KONEKTIVITAS jNNG (FIX CELAH #1)
        Titik hanya boleh pindah klaster jika memiliki edge jNNG dengan anggota klaster target.
        """
        n = len(X)
        labels = np.empty(n, dtype=int)
        
        # 1. Pre-compute anggota klaster saat ini untuk lookup O(1)
        cluster_members = {c: set(np.where(self.labels_ == c)[0]) for c in range(self.K_auto_)}
        
        # 2. Konversi adjacency matrix ke list of sets untuk akses tetangga yang cepat
        adj_list = jnng_adj.tolil().rows

        for i in range(n):
            # Hitung jarak EW-LMD ke semua sentroid
            dists = [
                sum(self._dist_attr(col, X.iloc[i][col], self.centers_.iloc[c][col]) 
                    for col in X.columns) 
                for c in range(self.K_auto_)
            ]
            best_c = int(np.argmin(dists)) # Klaster terdekat secara geometris
            
            # Cek konektivitas topologi jNNG
            neighbors_i = set(adj_list[i])
            
            # Cari klaster mana saja yang memiliki anggota terhubung dengan titik i
            connected_clusters = [
                c for c, members in cluster_members.items() 
                if neighbors_i.intersection(members)
            ]
            
            # 3. Keputusan Alokasi
            if connected_clusters:
                # Jika ada koneksi, pilih klaster yang TERHUBUNG yang jaraknya paling minimum
                labels[i] = min(connected_clusters, key=lambda c: dists[c])
            else:
                # Fallback: Titik terisolasi (tidak ada edge ke klaster manapun)
                # Gunakan jarak geometris murni agar tidak error
                labels[i] = best_c

        return labels

    def _update_centers_refinement(self, X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
        centers = []
        for c in range(self.K_auto_):
            cl = X[labels==c]
            if cl.empty: 
                centers.append(self.centers_.iloc[c].to_dict())
                continue
            c_dict = {col: _mode(cl[col]) if self.attr_types[col] in ('Nominal','Ordinal') else cl[col].mean() for col in X.columns}
            centers.append(c_dict)
        return pd.DataFrame(centers)

    def fit(self, df: pd.DataFrame):
        X = df.drop(columns=["Label"], errors="ignore").copy()
        for col in X.columns:
            if self.attr_types[col] == "Numeric":
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(X[col].mean())

        # 1. HYBRID PARAMETER TUNING
        structure = self._compute_structure(X)
        params = hybrid_selection_conservative(X, self.attr_types, structure)
        self.dis_const = params['dist_cons']
        self.eta_cat = self.eta_num = params['eta']
        self.n_bins_numeric = params['n_bins_numeric']  # ← PERBAIKAN
        self.auto_params_log = params.copy()
        if self.verbose: print(f"[AUTONOMOUS] Hybrid-tuned: {self.auto_params_log}")

        print("[Phase 0] Computing initializing distances...")
        # self._compute_entropy_weights(X)
        self._init_distances(X)
        
        print("[Phase 1] Building jNNG with LMD distance...")
        D = self._compute_distance_matrix(X)
        n = len(X)
        median_dist = np.median(D[np.triu_indices(n, k=1)])
        D_scaled = D / (median_dist + 1e-12)
        
        jnng_sparse, geo_dist, jnng_adj = self._build_jnng(D_scaled)
        
        # Menggunakan Local Scaling di sini
        rho_gjnn = self._gaussian_kernel_density(geo_dist)
        rho_tilde = self._two_round_sigmoid_correction(rho_gjnn)
        
        print("[Phase 2] Running Bombing process (Automatic K detection)...")
        K_before, labels_bombing = self._run_bombing_phase(X, D_scaled, geo_dist, rho_tilde, jnng_adj)
        print(f"   -> Detected K = {K_before}")
        
        self.labels_ = labels_bombing
        self.centers_ = self._compute_initial_centroids(X, labels_bombing)
        self.K_auto_ = K_before
        
        # print("[Phase 3] Computing density-excluded core centroids...")
        # self.centers_ = self._compute_initial_centroids(X, labels_refined)
        
        print("[Phase 4-6] EW-LMD Iterative Refinement Loop...")
        prev_labels = self.labels_.copy()
        best_silhouette = -1.0
        best_labels = self.labels_.copy()
        restart_count = 0
        
        for step in range(self.max_iter_refine):
            current_alpha = 0.5 if step < 2 else self.alpha_reg
            orig_alpha = self.alpha_reg
            self.alpha_reg = current_alpha
            
            self._update_nominal_ordinal_phi(X, self.labels_)
            self._update_numeric_phi(X, self.labels_)
            self.alpha_reg = orig_alpha
            
            self.labels_ = self._assign_refinement(X, jnng_adj)
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
            if step >= 2:
                ari_internal = adjusted_rand_score(prev_labels, self.labels_)
                
                # === PERBAIKAN: TRAP DETECTION via Silhouette ===
                sample_idx = np.random.choice(len(X), min(1000, len(X)), replace=False) if len(X) > 1000 else np.arange(len(X))
                D_sample = np.zeros((len(sample_idx), len(sample_idx)))
                for i, idx_i in enumerate(sample_idx):
                    for j, idx_j in enumerate(sample_idx):
                        # Menggunakan fungsi distance yang sudah ada
                        D_sample[i,j] = self.distance(X.iloc[idx_i], X.iloc[idx_j])
                
                try:
                    current_sil = silhouette_score(D_sample, self.labels_[sample_idx], metric='precomputed')
                except ValueError:
                    current_sil = 0.0
                    
                if current_sil > best_silhouette:
                    best_silhouette = current_sil
                    best_labels = self.labels_.copy()
                    
                print(f"   Iter {step+1}: Internal ARI = {ari_internal:.4f} | Silhouette = {current_sil:.4f}")
                
                if ari_internal > self.conv_threshold:
                    if current_sil < 0.1 and restart_count < 2:
                        print(f"   -> [Trap Detected] Silhouette < 0.1. Perturbing labels to escape local optimum...")
                        noise_mask = np.random.rand(len(X)) < 0.2
                        self.labels_[noise_mask] = self._rng.integers(0, self.K_auto_, size=np.sum(noise_mask))
                        self.centers_ = self._update_centers_refinement(X, self.labels_)
                        restart_count += 1
                        prev_labels = self.labels_.copy()
                        continue
                    else:
                        print("   -> Convergence reached.")
                        break
                        
            prev_labels = self.labels_.copy()

        if best_silhouette > current_sil:
            if self.verbose: print(f"   -> [Recovery] Restoring best labels from Iter with Silhouette = {best_silhouette:.4f}")
            self.labels_ = best_labels
            self.centers_ = self._update_centers_refinement(X, self.labels_)
            
        return self

## One-Step Missing Value Handling

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from copy import deepcopy

# ---------------------------------------------------
# Fungsi bantu
# ---------------------------------------------------
def mixed_distance(x, c, numeric_cols, cat_cols):
    """
    Menghitung jarak mixed-type seperti k-prototypes.
    Missing values (None/np.nan) diabaikan.
    """
    dist = 0.0
    
    # Bagian numerik: Euclidean
    for col in numeric_cols:
        val_x = x[col]
        val_c = c[col]
        if pd.notna(val_x) and pd.notna(val_c):
            # Pastikan keduanya numeric — aman karena sudah dikonversi di awal
            dist += (float(val_x) - float(val_c)) ** 2

    # Bagian kategorikal: simple matching
    for col in cat_cols:
        val_x = x[col]
        val_c = c[col]
        if pd.notna(val_x) and pd.notna(val_c):
            dist += 1 if str(val_x) != str(val_c) else 0
    
    return dist


def compute_centroid(df, cluster_members, numeric_cols, cat_cols):
    """
    Menghitung pusat cluster:
    - numerik: mean dari nilai yang tidak missing
    - kategorikal: modus (fallback ke nilai pertama jika semua NaN)
    """
    centroid = {}
    cluster_df = df.loc[cluster_members]

    for col in numeric_cols:
        # Ambil nilai non-null dan hitung mean
        values = pd.to_numeric(cluster_df[col], errors='coerce').dropna()
        centroid[col] = values.mean() if len(values) > 0 else np.nan

    for col in cat_cols:
        # Mode dari nilai non-null
        values = cluster_df[col].dropna()
        if len(values) > 0:
            mode_series = values.mode()
            centroid[col] = mode_series.iloc[0] if len(mode_series) > 0 else np.nan
        else:
            centroid[col] = np.nan

    return centroid


# ---------------------------------------------------
# Implementasi One-Step Imputation (imp.onestep)
# ---------------------------------------------------

def one_step_kprototypes(df, k, numeric_cols, cat_cols, max_iter=20):
    """
    One-step imputation using k-prototypes.
    
    Parameters:
        df (pd.DataFrame): Input data (may contain missing values)
        k (int): Number of clusters
        numeric_cols (list): List of numeric column names
        cat_cols (list): List of categorical column names
        max_iter (int): Max iterations
    
    Returns:
        df_imputed (pd.DataFrame): Imputed DataFrame
        clusters (dict): {cluster_id: [indices]}
        centroids (list of dict): Final centroids
    """
    # ✅ Ensure inputs are lists (not dicts!)
    if not isinstance(numeric_cols, list):
        raise TypeError("`numeric_cols` must be a list.")
    if not isinstance(cat_cols, list):
        raise TypeError("`cat_cols` must be a list.")
    
    df = df.copy()
    n = len(df)

    # ✅ 🔑 PERBAIKAN UTAMA: Konversi kolom numerik ke numeric (float), aman terhadap string/missing
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        else:
            raise KeyError(f"Column '{col}' in numeric_cols not found in DataFrame.")

    # Pastikan centroid juga akan berisi numeric (kami konversi saat inisialisasi & update)

    # 1. Inisialisasi centroid secara acak
    centroids = []
    # Hindari error jika n < k
    if k > n:
        raise ValueError(f"k={k} > number of samples ({n}). Reduce k.")
        
    chosen = np.random.choice(n, k, replace=False)
    for idx in chosen:
        row = df.loc[idx].to_dict()
        # Konversi numeric cols di centroid juga
        for col in numeric_cols:
            if col in row:
                val = row[col]
                row[col] = float(val) if pd.notna(val) else np.nan
        centroids.append(row)

    clusters = {}
    for iteration in range(max_iter):
        # 2. Assignment: tentukan cluster setiap objek
        clusters = {i: [] for i in range(k)}

        for i in range(n):
            x = df.loc[i].to_dict()  # aman: Series → dict
            distances = [
                mixed_distance(x, centroids[c], numeric_cols, cat_cols)
                for c in range(k)
            ]
            cluster_id = int(np.argmin(distances))  # pastikan int
            clusters[cluster_id].append(i)

        # 3. Update centroid
        new_centroids = []
        for c in range(k):
            members = clusters[c]
            if len(members) > 0:
                cent = compute_centroid(df, members, numeric_cols, cat_cols)
                # Pastikan numeric cols di centroid adalah float
                for col in numeric_cols:
                    if col in cent and pd.notna(cent[col]):
                        cent[col] = float(cent[col])
                new_centroids.append(cent)
            else:
                # Jika cluster kosong, pilih random baru & konversi numeric
                new_row = df.loc[np.random.choice(n)].to_dict()
                for col in numeric_cols:
                    if col in new_row:
                        val = new_row[col]
                        new_row[col] = float(val) if pd.notna(val) else np.nan
                new_centroids.append(new_row)

        # 4. Cek konvergensi (hanya numeric cols)
        converged = True
        for j in range(k):
            old_num = np.array([centroids[j].get(col, np.nan) for col in numeric_cols], dtype=float)
            new_num = np.array([new_centroids[j].get(col, np.nan) for col in numeric_cols], dtype=float)
            if not np.allclose(old_num, new_num, equal_nan=True):
                converged = False
                break

        if converged:
            print(f"✅ Converged at iteration {iteration + 1}")
            centroids = new_centroids
            break

        centroids = deepcopy(new_centroids)
    else:
        print(f"⚠️ Max iterations ({max_iter}) reached without full convergence.")

    # ---------------------------------------------------
    # ONE-STEP IMPUTATION
    # ---------------------------------------------------
    df_imputed = df.copy()

    for cluster_id, members in clusters.items():
        center = centroids[cluster_id]
        for i in members:
            for col in numeric_cols + cat_cols:
                if pd.isna(df_imputed.loc[i, col]) and col in center:
                    impute_val = center[col]
                    # Pastikan tipe data sesuai (hindari float untuk kolom kategorikal integer)
                    if col in cat_cols and not isinstance(impute_val, str):
                        # Konversi ke str kecuali sudah NaN
                        if pd.notna(impute_val):
                            impute_val = str(impute_val)
                    df_imputed.loc[i, col] = impute_val

    return df_imputed, clusters, centroids

## METRIC EVALUATOR (Internal & External)

In [ ]:
# ==============================================================================
# 1. METRIC EVALUATOR (Internal & External)
# ==============================================================================
class AblationEvaluator:
    """Penghitung metrik clustering yang konsisten untuk semua varian."""
    
    # =============================================================================
    # 2. IMPLEMENTASI METRIK EVALUASI BERBASIS MATRIKS JARAK EW-LMD
    # =============================================================================
    
    def silhouette_score_ewlmd(D, labels):
        n = len(labels)
        unique_labels = np.unique(labels)
        if len(unique_labels) < 2: return 0.0
        s_i = np.zeros(n)
        for i in range(n):
            cluster_i = labels[i]
            mask_intra = (labels == cluster_i)
            mask_intra[i] = False
            n_intra = np.sum(mask_intra)
            a_i = np.mean(D[i, mask_intra]) if n_intra > 0 else 0.0
            b_i = np.inf
            for c in unique_labels:
                if c == cluster_i: continue
                mask_inter = (labels == c)
                if np.sum(mask_inter) == 0: continue
                avg_dist = np.mean(D[i, mask_inter])
                if avg_dist < b_i: b_i = avg_dist
            denom = max(a_i, b_i)
            s_i[i] = (b_i - a_i) / denom if denom > 1e-12 else 0.0
        return float(np.mean(s_i))
    
    def dunn_index_ewlmd(D, labels):
        unique_labels = np.unique(labels)
        if len(unique_labels) < 2: return 0.0
        diameters = []
        for c in unique_labels:
            mask = (labels == c)
            idx = np.where(mask)[0]
            if len(idx) < 2: diameters.append(0.0)
            else:
                sub_D = D[np.ix_(idx, idx)]
                diameters.append(np.max(sub_D[np.triu_indices(len(idx), k=1)]))
        max_diam = max(diameters) if diameters else 1e-12
        if max_diam < 1e-12: max_diam = 1e-12
        min_inter_dist = np.inf
        for i_idx, c1 in enumerate(unique_labels):
            for c2 in unique_labels[i_idx+1:]:
                mask1, mask2 = (labels == c1), (labels == c2)
                inter_D = D[mask1][:, mask2]
                min_dist = np.min(inter_D)
                if min_dist < min_inter_dist: min_inter_dist = min_dist
        return float(min_inter_dist / max_diam)
    
    def davies_bouldin_index_ewlmd(D, labels):
        unique_labels = np.unique(labels)
        k = len(unique_labels)
        if k < 2: return np.inf
        scatter = {}
        centroids = {}
        for c in unique_labels:
            mask = (labels == c)
            idx = np.where(mask)[0]
            if len(idx) == 1: scatter[c] = 0.0
            else:
                sub_D = D[np.ix_(idx, idx)]
                avg_dist = np.mean(sub_D, axis=1)
                scatter[c] = np.min(avg_dist)
                centroids[c] = idx[np.argmin(avg_dist)]
        R = []
        for i in unique_labels:
            R_i = 0
            for j in unique_labels:
                if i == j: continue
                d_ij = D[centroids[i], centroids[j]]
                if d_ij < 1e-12: continue
                R_ij = (scatter[i] + scatter[j]) / d_ij
                if R_ij > R_i: R_i = R_ij
            R.append(R_i)
        return np.mean(R)
    
    def calinski_harabasz_ewlmd(D, labels):
        n = len(labels)
        unique_labels = np.unique(labels)
        k = len(unique_labels)
        if k < 2 or k == n: return 0.0
        SS_W = 0
        centroids = {}
        n_c = {}
        for c in unique_labels:
            mask = (labels == c)
            idx = np.where(mask)[0]
            n_c[c] = len(idx)
            if len(idx) < 2: continue
            sub_D = D[np.ix_(idx, idx)]
            SS_W += np.sum(np.triu(sub_D, k=1))
            avg_dist = np.mean(sub_D, axis=1)
            centroids[c] = idx[np.argmin(avg_dist)]
        SS_B = 0
        for i in unique_labels:
            for j in unique_labels:
                if i >= j: continue
                d_ij = D[centroids[i], centroids[j]]
                SS_B += n_c[i] * n_c[j] / (n_c[i] + n_c[j]) * (d_ij ** 2)
        return (SS_B / (k - 1)) / (SS_W / (n - k)) if (n - k) > 0 else 0.0

    def calculate_accuracy_from_confusion_matrix(y_true, y_pred):
        """
        Menghitung akurasi eksternal setelah optimal label mapping menggunakan Hungarian Algorithm.
        
        Parameters:
        - y_true: array-like, true labels
        - y_pred: array-like, predicted cluster labels
        
        Returns:
        - accuracy: float, akurasi setelah optimal mapping
        - mapping: dict, mapping dari cluster label ke true label
        """
        # Pastikan y_true dan y_pred berupa array numpy
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
    
        # Buat confusion matrix (true x pred)
        cm = confusion_matrix(y_true, y_pred)
        
        # Gunakan scipy.linear_sum_assignment untuk mendapatkan mapping optimal
        # Balikkan nilai confusion matrix menjadi cost matrix (negatif)
        cost_matrix = -cm
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
        # Jumlah total kesesuaian setelah optimal assignment
        total_correct = cm[row_ind, col_ind].sum()
        total_samples = len(y_true)
    
        accuracy = total_correct / total_samples
    
        # Buat mapping dari cluster label ke true label
        mapping = {col: row for row, col in zip(row_ind, col_ind)}
    
        return accuracy, mapping
    
    def purity_score(y_true, y_pred):
        df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
        return np.sum([grp["y_true"].value_counts().max() for _, grp in df.groupby("y_pred")]) / len(df)
    
    def category_utility(df, labels):
        cu = 0
        k = len(np.unique(labels))
        for cl in np.unique(labels):
            subset = df[labels==cl]
            p = len(subset)/len(df)
            for c in df.columns:
                freq_global = df[c].value_counts(normalize=True)
                freq_local = subset[c].value_counts(normalize=True)
                cu += p * (freq_local.pow(2).sum() - freq_global.pow(2).sum())
        return cu / k
    
    def entropy_reduction(df, labels):
        H_before = sum(df[c].value_counts(normalize=True).pow(2).sum() for c in df.columns)
        H_after = 0
        for cl in np.unique(labels):
            subset = df[labels==cl]
            p = len(subset)/len(df)
            H_after += p * sum(subset[c].value_counts(normalize=True).pow(2).sum() for c in df.columns)
        return H_after - H_before
        
    @staticmethod
    def compute_all_metrics(X: np.ndarray, D_ewlmd: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray, 
                            dist_func: Optional[Callable] = None) -> Dict[str, float]:
        """Menghitung semua metrik dalam satu call."""
        mask_valid = y_true != -1
        y_pred_valid = y_pred[mask_valid]
        D_valid = D_ewlmd[np.ix_(mask_valid, mask_valid)]
        acc, map_dict = AblationEvaluator.calculate_accuracy_from_confusion_matrix(y_true, y_pred)
        
        metrics = {
            'ACC': acc,
            'ARI': adjusted_rand_score(y_true[mask_valid], y_pred_valid),
            'NMI': normalized_mutual_info_score(y_true[mask_valid], y_pred_valid),
            'PUR': AblationEvaluator.purity_score(y_true[mask_valid], y_pred_valid),
            'SIL': AblationEvaluator.silhouette_score_ewlmd(D_valid, y_pred_valid),
            'DUN': AblationEvaluator.dunn_index_ewlmd(D_valid, y_pred_valid),
            'CLU': AblationEvaluator.category_utility(X, y_pred),
            'ENR': AblationEvaluator.entropy_reduction(X, y_pred),
            # 'DBI': AblationEvaluator.davies_bouldin_index_ewlmd(D_valid, y_pred_valid),
            # 'CHI': AblationEvaluator.calinski_harabasz_ewlmd(D_valid, y_pred_valid)
        }
        return metrics


## STATISTICAL ANALYZER & CD DIAGRAM

In [ ]:
# ==============================================================================
# 2. STATISTICAL ANALYZER & CD DIAGRAM
# ==============================================================================
class AblationStatistics:
    """Analisis signifikansi non-parametrik standar ML clustering."""
    
    @staticmethod
    def get_ranks(results_df: pd.DataFrame, metric: str) -> pd.DataFrame:
        """Hitung rank per dataset untuk SATU metrik (Higher is Better)."""
        df = results_df.copy()
        
        # 1. Agregasi mean per (dataset, config) untuk menghilangkan duplikat dari multiple seeds
        agg_df = df.groupby(['dataset', 'config'])[metric].mean().reset_index()
        
        # 2. Hitung rank untuk setiap config di dalam dataset yang sama (nilai lebih besar = rank lebih kecil)
        agg_df['rank'] = agg_df.groupby('dataset')[metric].rank(ascending=False, method='min')
        
        # 3. Pivot ke format matriks: Baris=Dataset, Kolom=Config (format standar Friedman)
        return agg_df.pivot(index='dataset', columns='config', values='rank')

    @staticmethod
    def friedman_test(ranks_df: pd.DataFrame) -> Tuple[float, float]:
        """Uji global Friedman pada matriks rank."""
        df_clean = ranks_df.dropna(how='any')
        if df_clean.shape[0] < 2 or df_clean.shape[1] < 2:
            return np.nan, 1.0
            
        stat, pval = friedmanchisquare(*[df_clean[col].values for col in df_clean.columns])
        return stat, pval

    @staticmethod
    def nemenyi_posthoc_1(ranks_df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
        """Uji post-hoc Nemenyi dengan koreksi Holm."""
        df_clean = ranks_df.dropna(how='any')
        if df_clean.shape[0] < 2 or df_clean.shape[1] < 2:
            return pd.DataFrame()
        return sp.posthoc_nemenyi(df_clean)

    @staticmethod
    def nemenyi_posthoc(ranks_df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
        """Uji post-hoc Nemenyi."""
        df_clean = ranks_df.dropna(how='any')
        if df_clean.shape[0] < 2 or df_clean.shape[1] < 2:
            return pd.DataFrame()
            
        # ✅ FIX: Pass DataFrame langsung agar scikit_posthocs mempertahankan nama grup (config)
        # Library otomatis mendeteksi kolom sebagai grup dan baris sebagai blok/dataset
        result = sp.posthoc_nemenyi(df_clean)
        
        return result

    @staticmethod
    def plot_critical_difference(ranks_mean: np.ndarray, config_names: List[str], 
                                 alpha: float = 0.05, save_path: Optional[str] = None):
        """Plot Critical Difference (CD) Diagram."""
        n_methods = len(config_names)
        n_datasets = ranks_mean.shape[0] if len(ranks_mean.shape) > 0 else 1
        if n_datasets == 0: return
        
        CD = 2.77 * np.sqrt((n_methods * (n_methods + 1)) / (6 * n_datasets))
        
        sorted_idx = np.argsort(ranks_mean)
        sorted_names = [config_names[i] for i in sorted_idx]
        sorted_ranks = ranks_mean[sorted_idx]
        
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.set_xlim(0.5, n_methods + 0.5)
        ax.set_ylim(0.5, 1.5)
        ax.set_yticks([])
        ax.set_xlabel('Average Rank (Lower is Better)')
        ax.set_title(f'Critical Difference Diagram (α={alpha})')
        
        ax.plot([CD/2, n_methods + 0.5 - CD/2], [1, 1], 'k-', lw=3)
        ax.text(n_methods/2 + 0.5, 1.05, f'CD = {CD:.2f}', ha='center', va='bottom', fontweight='bold')
        
        for i, (name, rank) in enumerate(zip(sorted_names, sorted_ranks)):
            ax.plot([i+1, i+1], [0.8, 1.2], 'k-')
            ax.text(i+1, 1.25, f'{name}\n{rank:.2f}', ha='center', va='bottom', fontsize=9)
            
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

## ABLATION CONFIGURATIONS & RUNNER

In [ ]:
# ==============================================================================
# 3. ABLATION CONFIGURATIONS & RUNNER
# ==============================================================================
ABLA_CONFIGS = [
    # {'name': 'V1_BombingEWLMD_wo_EW',
    #  'autonomous': True, 'entropy_weighting': False, 'density-excluded_init': True, 'posthoc': True, 'refinement': True},
    # {'name': 'V2_BombingEWLMD_wo_DE',
    #  'autonomous': True, 'entropy_weighting': True, 'density-excluded_init': False, 'posthoc': True, 'refinement': True},
    # {'name': 'V3_BombingEWLMD_wo_PM',
    #  'autonomous': True, 'entropy_weighting': True, 'density-excluded_init': True, 'posthoc': False, 'refinement': True},
    # {'name': 'V4_BombingEWLMD_wo_RF',
    #  'autonomous': True, 'entropy_weighting': True, 'density-excluded_init': True, 'posthoc': True, 'refinement': False},
    # {'name': 'V5_BombingEWLMD_full',
    #  'autonomous': True, 'entropy_weighting': True, 'density-excluded_init': True, 'posthoc': True, 'refinement': True},
    {'name': 'V1_BombingLMD',
     'autonomous': True, 'density-excluded_init': True, 'posthoc': True, 'refinement': True},
    {'name': 'V2_BombingLMD_woPM',
     'autonomous': True, 'density-excluded_init': True, 'posthoc': False, 'refinement': True},
    {'name': 'V3_BombingLMD_woDE',
     'autonomous': True, 'density-excluded_init': False, 'posthoc': True, 'refinement': True},
    {'name': 'V4_BombingLMD_woDE+PM',
     'autonomous': True, 'density-excluded_init': False, 'posthoc': False, 'refinement': True}
]

import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. FUNGSI DASAR: MATRIKS JARAK & MEDOID
# =============================================================================
def compute_ewlmd_distance_matrix(X, model):
    """Menghitung matriks jarak pairwise EW-LMD (Precomputed Distance Matrix)."""
    n_samples = X.shape[0]
    D = np.zeros((n_samples, n_samples))
    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            dist = model.distance(X.iloc[i], X.iloc[j])
            D[i, j] = dist
            D[j, i] = dist
    return D

def find_cluster_medoids(D, labels):
    """
    Mencari medoid (pusat representatif) untuk setiap klaster berdasarkan matriks jarak D.
    Medoid adalah titik di dalam klaster yang memiliki total jarak ke titik lain paling minimum.
    Ini menggantikan konsep 'mean centroid' yang tidak valid untuk data campuran.
    """
    unique_labels = np.unique(labels)
    medoids = {}
    
    for k in unique_labels:
        idx_k = np.where(labels == k)[0]
        if len(idx_k) == 1:
            medoids[k] = idx_k[0]
            continue
            
        # Sub-matriks jarak untuk klaster k
        D_k = D[np.ix_(idx_k, idx_k)]
        # Jumlahkan jarak setiap titik ke semua titik lain di klaster yang sama
        sum_distances = np.sum(D_k, axis=1)
        # Medoid adalah indeks lokal dengan jarak total minimum
        local_medoid_idx = np.argmin(sum_distances)
        # Konversi ke indeks global
        medoids[k] = idx_k[local_medoid_idx]
        
    return medoids

class AblationRunner:
    """Orchestrator utama untuk menjalankan ablasi secara reproducible."""
    
    def __init__(self, datasets: List[Tuple[str, np.ndarray, np.ndarray, np.ndarray, np.ndarray]], 
                 seeds: List[int] = list(range(10)),
                 # metric_names: List[str] = ['SIL', 'DUN', 'DBI', 'CHI','CLU', 'ENR']):
                 metric_names: List[str] = ['SIL', 'CLU', 'ENR', 'PUR','ARI', 'NMI']):
        self.datasets = datasets  
        self.seeds    = seeds
        self.metric_names = metric_names
        self.results  = []
        
    def _run_single_variant(self, config: Dict, X: np.ndarray, y_true: np.ndarray, attr_types: np.ndarray, ord_orders: np.ndarray, seed: int) -> Dict:
        """Placeholder: Integrasikan dengan class EWLMDBombingHybrid Anda."""

        k = len(np.unique(y_true))
        current_params = {}  # Akan diisi setelah fit
        
        # if config['name'] == 'V1_BombingEWLMD_wo_EW': 
        #     model = LMDBombingHybrid_V1(attr_types=attr_types, ordinal_orders=ord_orders, random_state=42, verbose=True)
        # elif config['name'] == 'V2_BombingEWLMD_wo_DE': 
        #     model = EWLMDBombingHybrid_V2(attr_types=attr_types, ordinal_orders=ord_orders, random_state=42, verbose=True)
        # elif config['name'] == 'V3_BombingEWLMD_wo_PM': 
        #     model = EWLMDBombingHybrid_V3(attr_types=attr_types, ordinal_orders=ord_orders, random_state=42, verbose=True)
        # elif config['name'] == 'V4_BombingEWLMD_wo_RF': 
        #     model = EWLMDBombingHybrid_V4(attr_types=attr_types, ordinal_orders=ord_orders, random_state=42, verbose=True)
        # else:
        #     model = EWLMDBombingHybrid(attr_types=attr_types, ordinal_orders=ord_orders, random_state=42, verbose=True)

        if config['name'] == 'V1_BombingLMD': 
            model = LMDBombingHybrid_V1(attr_types=attr_types, ordinal_orders=ord_orders, random_state=42, verbose=True)
        elif config['name'] == 'V2_BombingLMD_woPM': 
            model = LMDBombingHybrid_V2(attr_types=attr_types, ordinal_orders=ord_orders, random_state=42, verbose=True)
        elif config['name'] == 'V3_BombingLMD_woDE': 
            model = LMDBombingHybrid_V3(attr_types=attr_types, ordinal_orders=ord_orders, random_state=42, verbose=True)
        else: 
            model = LMDBombingHybrid_V4(attr_types=attr_types, ordinal_orders=ord_orders, random_state=42, verbose=True)

        print('Menjalankan ', config['name'])
        model.fit(X)
        y_pred = model.labels_
        k = model.K_auto_
        current_params = {
            "dist_cons": float(model.dis_const),
            "eta": float(model.eta_cat),
            "n_bins_numeric": int(model.n_bins_numeric)
        }
        print(f"Params: {current_params} | K = {k}")

        print('Evaluasi ...')
        y_pred = model.labels_
        D_ewlmd = compute_ewlmd_distance_matrix(X, model)
        metrics = AblationEvaluator.compute_all_metrics(X, D_ewlmd, y_true, y_pred)
        metrics.update({'dataset': '', 'config': config['name'], 'seed': seed, 'k': k})
        
        print('Selesai')
        
        return metrics

    def execute(self) -> pd.DataFrame:
        """Menjalankan seluruh pipeline ablasi."""
        print(f"🚀 Memulai Ablasi Study: {len(self.datasets)} datasets × {len(ABLA_CONFIGS)} configs × {len(self.seeds)} seeds")
        
        all_results = []
        for ds_name, X, y_true, attr_types, ord_orders in self.datasets:
            print(f"Dataset: {ds_name} ...")
            for cfg in ABLA_CONFIGS:
                for seed in self.seeds:
                    res = self._run_single_variant(cfg, X, y_true, attr_types, ord_orders, seed)
                    res['dataset'] = ds_name
                    all_results.append(res)
            print(f"✅ Dataset: {ds_name} selesai.")
        self.results = pd.DataFrame(all_results)
        print("✅ Ablasi selesai. Mengagregasi hasil...")
        return self.results

    def aggregate_results(self) -> pd.DataFrame:
        """Agregasi mean ± std per dataset & config."""
        agg = self.results.groupby(['dataset', 'config'])[self.metric_names].agg(['mean', 'std'])
        return agg.reset_index()

## VISUALIZATION MODULE

In [ ]:
class AblationVisualizer:
    """Plotting siap publikasi."""
    
    @staticmethod
    def plot_metric_boxplot(results_df: pd.DataFrame, metric: str, save_path: str = None):
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=results_df, x='config', y=metric, palette='viridis', fliersize=4)
        sns.stripplot(data=results_df, x='config', y=metric, color='black', size=4, jitter=True, alpha=0.6)
        plt.xticks(rotation=45, ha='right')
        plt.title(f'{metric} Distribution Across Configurations')
        plt.ylabel(metric)
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=300)
        plt.show()
        
    @staticmethod
    def plot_summary_heatmap(agg_df: pd.DataFrame, save_path: str = None):
        # ✅ FIX: groupby(level=0).mean() menggantikan mean(level=0) untuk pandas ≥ 2.0
        pivot = agg_df.set_index(['config', 'dataset']).groupby(level=0).mean()
        
        plt.figure(figsize=(10, 6))
        sns.heatmap(pivot, annot=True, cmap='YlGnBu', fmt='.3f', linewidths=.5)
        plt.title('Mean Performance Across Datasets')
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=300)
        plt.show()

## MAIN EXECUTION PIPELINE

In [ ]:
# ==============================================================================
# 5. MAIN EXECUTION PIPELINE
# ==============================================================================
import os
import json
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, fowlkes_mallows_score

# 1. Generate/Load Synthetic Datasets (sesuaikan dengan generator Anda)
json_path = "new_53_dataset_schemas.json"
dataset_folder = "53_mix_datasets"
CANDIDATE_LABELS = ["class", "Class", "label", "Label", "target", "Target"]

with open(json_path, "r") as f:
    schema_all = {entry["filename"]: entry for entry in json.load(f)}

DATASET_LIST = [
    # *[f'data-sintesis-{i:02d}.csv' for i in range(26, 31)],
    'new_01_automobile.csv', 'new_02_german_credit.csv', 'new_03_heart_disease.csv',
    'new_04_australian_credit.csv', 'new_05_student_performace_mat.csv', 
    # 'new_06_telco_customer_churn.csv',
    'new_08_exasens.csv', 'new_10_npha-doctor-visits.csv', 'new_11_cirrhosis.csv',
    'new_12_caesar.csv', 'new_13_obesity.csv', 'new_14_heart_failure.csv', 'new_15_stroke-data.csv'
]

datasets = []

for filename, meta in schema_all.items():
    if filename not in DATASET_LIST:
        continue
        
    path = os.path.join(dataset_folder, filename)
    if not os.path.exists(path): 
        print(f"️ {filename} not found."); continue
        
    print(f"📂 Processing: {filename}")
    df = pd.read_csv(path).replace([" ?", "?"], np.nan)
    
    # Label Detection & Encoding
    label_col = meta.get("label_column") or next((c for c in CANDIDATE_LABELS if c in df.columns), None)
    if not label_col: 
        print("️ No label column."); continue
    
    le = LabelEncoder()
    y_true = le.fit_transform(df[label_col].astype(str))
    k = len(np.unique(y_true))
    X = df.drop(columns=[label_col] + [c for c in df.columns if c.lower() == 'id'])
    
    # Attribute Mapping
    attr_types, attr_orders = {}, {}
    for col in meta.get("nominal_attributes", []) + meta.get("binary_attributes", []):
        if col in X.columns: attr_types[col] = "Nominal"
    for col in meta.get("numeric_attributes", []):
        if col in X.columns: attr_types[col] = "Numeric"
    for col in meta.get("ordinal_attributes", []):
        if col in X.columns:
            attr_types[col] = "Ordinal"
            attr_orders[col] = meta["ordinal_orders"].get(col, sorted(X[col].dropna().unique()))
            
    # Imputation
    numeric_cols = [c for c, t in attr_types.items() if t == "Numeric"]
    cat_cols = [c for c, t in attr_types.items() if t != "Numeric"]
    # Cek apakah masih ada missing value
    if X.isnull().sum().any():
        print("\nMasih ada missing value:")
        print(X.isnull().sum())  # Menampilkan jumlah missing value per kolom
        X_imputed, _, _ = one_step_kprototypes(
            df=X.copy(), 
            k=k, 
            numeric_cols=numeric_cols, 
            cat_cols=cat_cols
        )
        # Cek apakah masih ada missing value
        if X_imputed.isnull().sum().any():
            print("Masih ada missing value:")
            print(df.isnull().sum())  # Menampilkan jumlah missing value per kolom
        else:
            print("Tidak ada missing value.")
    else:
        X_imputed = X.copy()
    
    dataset_entry = {"Dataset": filename, "k": k, "n_rows": len(X_imputed)}
    datasets.append([filename, X_imputed, y_true, attr_types, attr_orders])

# 2. Inisialisasi & Jalankan Runner
runner = AblationRunner(datasets=datasets, seeds=list(range(1)))
raw_results = runner.execute()

# 3. Agregasi
agg_df = runner.aggregate_results()
AblationVisualizer.plot_summary_heatmap(agg_df, save_path='ablation_heatmap.png')

# 5. Simpan hasil lengkap
raw_results.to_csv('14_ablation2_raw_results.csv', index=False)
agg_df.to_csv('14_ablation2_aggregated_results.csv', index=False)
print("\n💾 Semua hasil & plot tersimpan. Pipeline selesai.")